# Backlog localization

Per-frame poses for the buggy-roll backlog against the `rtk_base` COLMAP map, on Kaggle, resumably.
One roll at a time: fetch, sample at 10 Hz, extract ALIKED on 2xT4, bootstrap an arc position, fit
the roll's intrinsics, match, PnP, bank a per-frame record plus its correspondences.

Preserved from `tmp/backlog/scripts/b3_kernel.py` (same source; `b3_gen.py` assembles it into the
pushed kernels, `b8_nbgen.py` into this notebook), one cell per `# === cell: loc-* ===` block. See
that file's header for the full rationale and environment surface, and `tmp/backlog/PLAN.md`.

## Needs

- Dataset `yu5uf5/buggy-rtk-base-cache-8218` attached, and internet on -- `loc-install` installs
  onnxruntime, TensorRT, faiss and pycolmap at runtime.
- Two rclone remotes: `srs:` for the `[[gdrive]]` rolls, `yusufishabazz:` for the rest and the work
  directory. **Credentials are not in this file** -- see `loc-config`.
- Staged inputs under `<remote>/inputs` (`plan.json`, intrinsics seeds, intro reference, map-clock
  offsets), written by `b1_plan.py` / `b2_stage_inputs.py`.

## The GPU/CPU split

Bootstrap and the roll's tail are CPU work in a session whose scarce resource is a pair of T4s, so
both move to CPU-only sessions, which run concurrently and consume no GPU quota.

| channel | request | answer |
| --- | --- | --- |
| bootstrap | `out/anchors` | `out/boot` |
| tail | `out/tail` | `out/tailres` |

If no answer arrives inside the wait the GPU does the stage itself; both sides run the same seeded
code, so `boot_source`/`tail_source` say only where the time went, never what the numbers are.

## Running it

The role is **discovered** from the GPU inventory, not passed -- Kaggle metadata carries no
environment. Push the same source as one GPU kernel (`machine_shape: NvidiaTeslaT4`) and N CPU
kernels differing only in metadata; production runs four workers.

⚠ All kernels must be **byte-identical**. `loc_run_id` is a sha of the source and the handoff is
namespaced under it, so a kernel that hashes differently cannot see the others' work -- and deletes
their requests as foreign. A notebook has no `__file__`, so `loc_run_id` falls back to `'unknown'`
and does not share a resume namespace with a pushed kernel; use `SRS_LOC_TAG` to separate runs.

`SRS_LOC_DRY=1` runs the whole control flow with no network.

## `loc-config`

Every tunable, each with an `SRS_LOC_*` override.

In [ ]:
# === cell: loc-config ===
import calendar
import hashlib
import json
import os
import queue
import re
import shutil
import subprocess
import sys
import threading
import time
from pathlib import Path

import numpy as np

# `srs:` holds the [[gdrive]] rolls, `yusufishabazz:` the rest and the work directory.  Inlined
# by b3_gen.py and redacted from the run id -- rclone rewrites its config on every OAuth refresh.
# Not checked in; set via Kaggle secret.  Each must stay ONE line or loc_run_id's redaction breaks.
RCLONE_CONF_SRS = os.environ.get('SRS_RCLONE_CONF_SRS', '<paste [srs] section>')
RCLONE_CONF_BACKUP = os.environ.get('SRS_RCLONE_CONF_BACKUP', '<paste [yusufishabazz] section>')

LOC_REMOTE = os.environ.get('SRS_LOC_REMOTE', 'yusufishabazz:buggy_hloc/backlog')
LOC_IN_REMOTE = LOC_REMOTE + '/inputs'
LOC_OUT_REMOTE = LOC_REMOTE + '/out'
LOC_LOG_REMOTE = LOC_REMOTE + '/log'

LOC_ROOT = Path(os.environ.get('SRS_LOC_ROOT', '/tmp/backlog'))
LOC_IN = LOC_ROOT / 'inputs'
LOC_VID = LOC_ROOT / 'vid'
LOC_OUT = LOC_ROOT / 'out'
LOC_MODELS = LOC_ROOT / 'models'
LOC_TRTCACHE = LOC_ROOT / 'trtcache'
LOC_LOGF = LOC_ROOT / 'kernel.log'
LOC_STATEF = LOC_ROOT / 'state.json'
LOC_KWORK = Path('/kaggle/working')
# the attached Kaggle dataset yu5uf5/buggy-rtk-base-cache-8218: p4/, p6a/, p7/
LOC_INPUT = Path(os.environ.get('SRS_LOC_INPUT', '/kaggle/input'))

LOC_DRY = os.environ.get('SRS_LOC_DRY') == '1'
LOC_TAG = os.environ.get('SRS_LOC_TAG', '')
LOC_STAGES = os.environ.get('SRS_LOC_STAGES', 'setup,gate,map,run,final').split(',')

# sampling
LOC_HZ = float(os.environ.get('SRS_LOC_HZ', 10.0))
LOC_W, LOC_H = 1280, 720              # every frame is downscaled to the map's image size
LOC_DIVISOR = 32                      # -> padded 1280x736, what the static graph is built for
# Bounds are consumed VERBATIM as raw video-file offsets.  Asserted, not assumed: a plan on a
# superseded time base would sample every frame from the wrong place and merely look bad.
LOC_TIME_BASE = tuple(os.environ.get('SRS_LOC_TIME_BASE', 'video_ms,raw_video_ms').split(','))
# VIRB Edit title cards (485 rolls) match nothing and would pollute the bootstrap.  Detected per
# frame against intro_ref.npy, not from the bounds.  Calibrated: card MAD <= 1.0, others >= 73.
LOC_INTRO_MAD = float(os.environ.get('SRS_LOC_INTRO_MAD', 20.0))
LOC_INTRO_REF = [None]
# per-run map naming-clock offsets, if they have been derived from the FIT recording start;
# absent, the in-map control fits a shift instead and says so
LOC_MAP_OFFSETS = [None]
# One frame at 29.97 fps is 33.4 ms; 70 ms is that plus slack for 2 m map frame spacing.  A
# larger residual is an upstream regression, not a bad offset, and it fails the pilot.
LOC_CTL_MAX_RESIDUAL_MS = float(os.environ.get('SRS_LOC_CTL_MAX_RESIDUAL_MS', 70.0))
LOC_REQUIRE_OFFSETS = os.environ.get('SRS_LOC_REQUIRE_OFFSETS', '1') == '1'

# ALIKED, exactly as colmap/feature/aliked.cc
LOC_MAX_KEYPOINTS = 4096
LOC_MIN_SCORE = 0.2
# TensorRT's ITopKLayer cap is EXCLUSIVE at 3840; 3839 costs 5 keypoints in 52,828 (p6d)
LOC_TOPK_K = int(os.environ.get('SRS_LOC_TOPK_K', 3839))
LOC_EP = os.environ.get('SRS_LOC_EP', 'trt')             # trt | cuda | cpu
LOC_DEVICES = [int(x) for x in os.environ.get('SRS_LOC_DEVICES', '0,1').split(',') if x != '']
LOC_BANK_DEVICE = os.environ.get('SRS_LOC_BANK_DEVICE', 'cuda:0')
# ONE replica: two was measured and reverted -- match 19-34 s -> 45.59 s with a 2.4x device
# asymmetry, and the second replica took the GPU the look-ahead extracted on (72 s -> 210 s).
LOC_BANK_DEVICE_LIST = ([x for x in os.environ.get('SRS_LOC_BANK_DEVICES', '').split(',') if x]
                        or [LOC_BANK_DEVICE])
LOC_MATCH_DEVICES = int(os.environ.get('SRS_LOC_MATCH_DEVICES', 1))

# bootstrap (PLAN 3.2)
LOC_BOOT_STRIDE = int(os.environ.get('SRS_LOC_BOOT_STRIDE', 10))
LOC_BOOT_GATE = int(os.environ.get('SRS_LOC_BOOT_GATE', 40))
LOC_NPROBE = int(os.environ.get('SRS_LOC_NPROBE', 8))
# 'gpu' (exhaustive on the resident bank) was measured and reverted: 1434-1663 ms/anchor against
# faiss's ~1460 on the CPU pool.  The search is memory-bound on ~47 GB/anchor of tile traffic,
# not compute-bound as the 6.03 TFLOP/anchor estimate assumed.  Keep 'faiss'.
LOC_BOOT_KNN = os.environ.get('SRS_LOC_BOOT_KNN', 'faiss')
# 0 = loc_workers() (physical cores) = 2 on Kaggle.  ⚠ 4 IS A MEASURED NEGATIVE: the IVFPQ scan
# runs 1.007 / 0.517 / 0.671 s per anchor at 1 / 2 / 4 threads.  The box is 4 vCPU but 2 physical,
# so threads 3-4 are hyperthreads on cores already saturated by a SIMD-bound scan.
LOC_BOOT_WORKERS = int(os.environ.get('SRS_LOC_BOOT_WORKERS', 0))
# Tile budget: the full score matrix is 43.9 GiB per anchor.  Width derives from this and nq.
LOC_KNN_TILE_B = int(float(os.environ.get('SRS_LOC_KNN_TILE_GB', 1.0)) * 2 ** 30)
# TensorRT fp32 is SLOWER than plain CUDA (7.27 vs 12.72 f/s).  Must match on the probe and the
# runners: they share an engine cache, so a probe could hand an fp32 engine straight over.
LOC_TRT_FP16 = os.environ.get('SRS_LOC_TRT_FP16', '1') == '1'
# The guard measures free VRAM per device and declines one rather than risk a mid-session OOM.
LOC_BANK_DEVICES = [x for x in os.environ.get('SRS_LOC_BANK_DEVICES', '').split(',') if x]
LOC_MATCH_CHUNK = int(os.environ.get('SRS_LOC_MATCH_CHUNK', 32))   # frames dispatched at once
# Score-block size in ELEMENTS.  2**25 ran the GEMM/reduce loop twice per frame; 2**26 is one
# pass at 268 MB, which fits beside the 3.17 GB bank.
LOC_MATCH_BLOCK = int(os.environ.get('SRS_LOC_MATCH_BLOCK', 1 << 26))
LOC_VRAM_HEADROOM_B = int(float(os.environ.get('SRS_LOC_VRAM_HEADROOM_GB', 5.0)) * 2 ** 30)
LOC_KNN = int(os.environ.get('SRS_LOC_KNN', 32))
LOC_SMOOTH_WIN = 9

# windowed match (PLAN 3.3 / 5)
LOC_RATIO = 0.95
LOC_WINDOW_M = 30.0
LOC_SHORT_K = 8                       # r0 nearest-arc map frames
LOC_CAP = 4                           # observations per point
LOC_TOPK = LOC_CAP + 1                # provably distinct 2nd point under the cap
LOC_MIN_CORRS = 6
LOC_ARC_MAX = 1387.05                 # centreline length, metres

# intrinsics (PLAN 3.4)
LOC_PP = (631.72, 338.74)
LOC_INTR_STRIDE = 4
LOC_INTR_MIN_INL = 30
LOC_INTR_MIN_FRAMES = 5               # fewer -> retry every frame
LOC_INTR_MIN_FALLBACK = 3             # fewer still -> map default
# (lens_correction, stabilizer) -> (focal, k1).  A SEED for the rolls it covers, never a final
# answer: 8/1/1 map runs plus 20 fitted previews, and 66% of rolls have no udta at all.
LOC_SEED_CLUSTERS = {('ON', 'ON'): (783.0, -0.036), ('OFF', 'ON'): (772.0, -0.196),
                     ('OFF', 'OFF'): (584.0, -0.185), ('ON', 'OFF'): (608.0, -0.013)}
LOC_SEED_DEFAULT = LOC_SEED_CLUSTERS[('ON', 'ON')]
LOC_PNP_MAX_ERROR_PX = 12.0

# operations
LOC_SESSION_MAX_MIN = float(os.environ.get('SRS_LOC_SESSION_MAX_MIN', 690.0))   # 11h30 of 12h
# Inlined by b3_gen.py --budget-h: Kaggle metadata carries no environment, so it must be baked
# in.  Redacted from the run id, so revising ONLY the budget does not discard the cursor.  Any
# other source edit does change the run id, which resets LOC_BASE_S -- push a smaller budget.
LOC_BUDGET_H_DEFAULT = 20.0
LOC_BUDGET_H = float(os.environ.get('SRS_LOC_BUDGET_H', LOC_BUDGET_H_DEFAULT))  # user's cap
LOC_HEARTBEAT_S = float(os.environ.get('SRS_LOC_HEARTBEAT_S', 25.0))
# Kaggle passes `machine_shape` through UNVALIDATED, so the push cannot tell what accelerator it
# got.  One T4 instead of two doubles the run, which would otherwise surface half way through.
LOC_MIN_GPUS = int(os.environ.get('SRS_LOC_MIN_GPUS', 2))
# The 2xT4 rate the budget is priced on.  25.60 is the best figure backed by an artifact, and
# production measures 25.8; the 32.95 that stood here was never traceable to one.
LOC_REF_FPS = float(os.environ.get('SRS_LOC_REF_FPS', 25.60))
LOC_PILOT = os.environ.get('SRS_LOC_PILOT', '1') == '1'
LOC_PILOT_N = int(os.environ.get('SRS_LOC_PILOT_N', 10))
LOC_IN_MAP_ROLLS = (37, 38, 45, 391, 764, 916, 982, 1005, 1388, 1401)
LOC_MAX_ROLLS = int(os.environ.get('SRS_LOC_MAX_ROLLS', 0))       # 0 = the whole plan
# A failed roll is recorded and skipped; a run of them is a defect, not bad luck, and aborts.
LOC_MAX_CONSEC_FAIL = int(os.environ.get('SRS_LOC_MAX_CONSEC_FAIL', 5))
LOC_MAX_FRAMES = int(os.environ.get('SRS_LOC_MAX_FRAMES', 0))     # 0 = every sampled frame
LOC_QUEUE = int(os.environ.get('SRS_LOC_QUEUE', 12))              # decoded frames in flight
LOC_SEC_PER_FRAME0 = 0.036            # PLAN 4: 30.35 ms extract + 5.6 ms match, seeds the ETA

# b6_stop.py writes <remote>/stop.json; the kernel polls it between rolls and exits 0 -- a
# requested stop is not a failure and must not read as the pilot (5), failures (6) or gates
# (7/9).  Renamed on the way out: a stale flag would kill the next session before its first roll.
LOC_STOP_NAME = os.environ.get('SRS_LOC_STOP_NAME', 'stop.json')
LOC_STOP_POLL_S = float(os.environ.get('SRS_LOC_STOP_POLL_S', 30.0))
LOC_STOP = {'seen': None, 't': 0.0, 'where': None}

# runtime installs (loc-install).  The base image ships onnx but NOT onnxruntime.
LOC_PIP = os.environ.get('SRS_LOC_PIP', '1') == '1'
LOC_TRT_INSTALL = os.environ.get('SRS_LOC_TRT', '1') == '1'
# A TensorRT engine build is minutes, and a wedged one would otherwise burn the session: every
# probe that can trigger a build is bounded by this and a timeout is recorded, not raised.
LOC_TRT_BUILD_MIN = float(os.environ.get('SRS_LOC_TRT_BUILD_MIN', 25.0))
LOC_ORT_RECIPES = os.environ.get('SRS_LOC_ORT_RECIPES', '')       # comma-separated subset
# numpy's identity BEFORE any pip runs; every recipe is pinned back to it.  Swapping it leaves
# numpy.ma disagreeing with its own umath, and every later cv2/pandas import dies with it.
LOC_NUMPY0 = {'version': np.__version__, 'file': np.__file__}
# the GPU pycolmap the map was built with (backend/src/notebooks/remote/mapping.ipynb installs
# the same wheel); cp312 matches the Kaggle image's python
LOC_PYCOLMAP_WHEEL = os.environ.get('SRS_LOC_PYCOLMAP_WHEEL', (
    'https://github.com/lyehe/build_gpu_colmap/releases/download/v4.1.0/'
    'pycolmap-4.1.0+cu128.bundled.cudss-cp312-cp312-manylinux_2_35_x86_64.whl'))
# Modules the base image may not ship.  b4_dryrun.py audits every import against this list --
# including imports inside function bodies, which is where onnxruntime hid.
LOC_INSTALLS = ('onnxruntime', 'faiss', 'pycolmap')

# Per-stage wall clock, in the production path on purpose: a per-roll total once got read as one
# stage's cost and put the run 11x off its model when it was 5.3x, in a different stage.
LOC_PROF = {}
LOC_PROF_TAG = ['']
LOC_ROLLS_DONE = [0]
# CUDA is async, so without a sync every span's cost lands on whichever later call forces one --
# d2h reads as the expensive stage.  Syncing costs time, so it runs on the first roll only.
LOC_PROF_SYNC = [False]
LOC_PROF_SYNC_ROLLS = int(os.environ.get('SRS_LOC_PROF_SYNC_ROLLS', 1))
LOC_TRIM_LAST = [None]                # the last loc_malloc_trim, reported on the next roll

# 0 lets load_index decide from whether the anchor pool is running; non-zero forces the main
# thread's OpenMP width.  It was hard-coded to one thread while the bootstrap was ~45% of a roll.
LOC_FAISS_THREADS = int(os.environ.get('SRS_LOC_FAISS_THREADS', 0))
# Across FRAMES, never inside one solve: RANSAC keeps num_threads=1 and a fixed seed, so every
# answer stays bit-identical to the serial one and the in-map controls cannot move.  Optimum is
# PHYSICAL cores (3.34x at 4, 2.51x at 8).  ransac.num_threads=4 measured SLOWER -- rejected.
LOC_PNP_WORKERS = int(os.environ.get('SRS_LOC_PNP_WORKERS', 0))
LOC_PNP_CHUNK = int(os.environ.get('SRS_LOC_PNP_CHUNK', 64))    # frames between heartbeats
LOC_SHORTLIST_QUEUE = int(os.environ.get('SRS_LOC_SHORTLIST_QUEUE', 2))   # windows in flight
# Extract the next rolls on the GPU under this roll's CPU stages.  Holding their features is the
# cost, so it is guarded on measured MemAvailable; the guard declines and runs serially.
LOC_AHEAD = os.environ.get('SRS_LOC_AHEAD', '1') == '1'
# QUEUE DEPTH, not concurrency -- one extract thread regardless, since two racing for the same
# T4s finish no sooner.  Depth posts bootstrap requests further ahead: at 1 there were only 3
# questions outstanding for 4 workers, so one was idle by construction (13% billed idle).
LOC_AHEAD_DEPTH = max(1, int(os.environ.get('SRS_LOC_AHEAD_DEPTH', 3)))
LOC_AHEAD_SAFETY = float(os.environ.get('SRS_LOC_AHEAD_SAFETY', 1.3))
# Covers the transient beyond a roll's own feature bytes (~1 GiB measured).  3 GiB is the FLOOR:
# an OOM kills the session outright, with no recovery path but a manual re-push.
LOC_AHEAD_HEADROOM_B = int(float(os.environ.get('SRS_LOC_AHEAD_HEADROOM_GB', 4.0)) * 2 ** 30)
# ⚠ NOT the depth -- LOC_AHEAD_DEPTH is.  This pins how many rolls the guard sizes for, and
# raising it only makes the guard decline MORE.  0 (derive) is correct; it is a debug hatch.
#
# ⚠ The failure mode is not an OOM, it is the guard declining silently for ever: that costs ~1.3x
# and looks exactly like a slow session.  Hence the counters -- `declined_consec` counts only
# fills that left the queue EMPTY, `declined_by_slot` says which slot is refused.  Size any check
# against free RAM WHILE ROLLING (10.3-12.7 GiB), never at startup (28-30 GiB).
LOC_AHEAD_ROLLS = int(os.environ.get('SRS_LOC_AHEAD_ROLLS', 0))     # 0 = derive from the queue
LOC_AHEAD_DECLINE_WARN = int(os.environ.get('SRS_LOC_AHEAD_DECLINE_WARN', 3))
LOC_AHEAD_STATS = {'declined_consec': 0, 'declined_total': 0, 'started_total': 0,
                   'last_reason': None, 'last_free_gib': None, 'declined_by_slot': {},
                   'depth_target': LOC_AHEAD_DEPTH, 'depth_last': None, 'depth_hist': {},
                   'ready_last': None, 'miss_total': 0, 'miss_consec': 0,
                   'reordered': 0, 'mismatched': 0, 'failed': 0}

# ---- the split: the bootstrap runs in a separate CPU-only Kaggle session ------------------
# Kaggle metadata cannot pass an environment variable, so the role is discovered from the GPU
# probe.  Only a probe that completed AND agreed with itself on zero devices demotes a session;
# a disagreeing probe is a broken instrument and still exits 8/7.
LOC_ROLE_ENV = os.environ.get('SRS_LOC_ROLE', 'auto')    # gpu | cpu | auto
LOC_ROLE = ['gpu']                                       # resolved from LOC_ROLE_ENV in main
LOC_SPLIT = os.environ.get('SRS_LOC_SPLIT', '1') == '1'  # 0 pins the role to gpu
LOC_SESSION_ID = [None]
# Per-role names, so a worker cannot overwrite the run's state, heartbeat or log.  The gpu role
# keeps the un-split names, so the monitor and every hand `rclone cat` read what they always did.
LOC_STATE_NAME = ['state.json']
LOC_HB_NAME = ['heartbeat.json']
LOC_LOG_NAME = ['kernel.log']
# Under out/ on purpose: loc_resume_cursor's '*.npz' filter already drops rclone's 'anchors/'.
LOC_ANCHOR_REMOTE = LOC_OUT_REMOTE + '/anchors'      # gpu -> worker, one roll's anchor features
LOC_BOOT_REMOTE = LOC_OUT_REMOTE + '/boot'           # worker -> gpu, one roll's bootstrap result
LOC_CLAIM_REMOTE = LOC_OUT_REMOTE + '/claims'        # worker -> worker, who is doing what
# Second channel, same protocol.  Separate directories so one `lsf` per channel answers "what is
# outstanding"; claims are shared, keyed by request name, which the two channels cannot collide.
LOC_TAIL_REMOTE = LOC_OUT_REMOTE + '/tail'           # gpu -> worker, one roll's correspondences
LOC_TAILRES_REMOTE = LOC_OUT_REMOTE + '/tailres'     # worker -> gpu, that roll's poses

# A timeout costs `wait + local stage`, so it must stay under the local stage: 379 s measured
# contended, which is what the fallback actually competes with -- not the 131-153 s quiet figure.
LOC_BOOT_WAIT_S = float(os.environ.get('SRS_LOC_BOOT_WAIT_S', 135.0))
# Ceiling from the first poll regardless of GPU work.  The clock below arms only once the
# look-ahead is done -- correct, since waiting during extract is free -- but a hung look-ahead
# would then wait for ever.
LOC_BOOT_WAIT_MAX_S = float(os.environ.get('SRS_LOC_BOOT_WAIT_MAX_S', 600.0))
LOC_BOOT_POLL_S = float(os.environ.get('SRS_LOC_BOOT_POLL_S', 10.0))
# After this many CONSECUTIVE fallbacks the split is presumed broken and the next N rolls go
# local without paying the upload or the wait; then it re-probes with one request.
# ⚠ THREE, NOT TWO: the first two rolls of a session get ~0 s and ~78 s of head start no matter
# the queue depth, so at 2 that structural ramp opened the breaker on every single restart.
LOC_BOOT_BREAK_AFTER = int(os.environ.get('SRS_LOC_BOOT_BREAK_AFTER', 3))
LOC_BOOT_BREAK_ROLLS = int(os.environ.get('SRS_LOC_BOOT_BREAK_ROLLS', 5))
# A claim is void when its OWNER is gone, which the heartbeat says and the claim's age does not:
# 81 of 135 requests were once worked by two to four workers at once with zero steals among them,
# a check-then-act race that loc_worker_claim_confirmed fixes.  The age is only a backstop for an
# owner that never wrote a heartbeat, so it is deliberately long.
LOC_CLAIM_TTL_S = float(os.environ.get('SRS_LOC_CLAIM_TTL_S', 1800.0))
LOC_CLAIM_REFRESH_S = float(os.environ.get('SRS_LOC_CLAIM_REFRESH_S', 60.0))
# How stale the OWNER's heartbeat must be before its claim is fair game.  A worker heartbeats
# every LOC_HEARTBEAT_S (25 s) and forces one on every claim refresh (60 s), so three minutes of
# silence is a dead session and not a slow roll.
LOC_CLAIM_BEAT_STALE_S = float(os.environ.get('SRS_LOC_CLAIM_BEAT_STALE_S', 180.0))
# Must exceed the write's own visibility lag (copyto+moveto, ~1-3 s here): two workers that both
# list before either move is visible are exactly the observed race.
LOC_CLAIM_CONFIRM_S = float(os.environ.get('SRS_LOC_CLAIM_CONFIRM_S', 4.0))
# Worker liveness.  A result that arrived recently IS proof of a live worker and costs nothing
# to observe, so the heartbeat files are only read when that evidence has gone stale.
LOC_WORKER_TRUST_S = float(os.environ.get('SRS_LOC_WORKER_TRUST_S', 300.0))
LOC_WORKER_STALE_S = float(os.environ.get('SRS_LOC_WORKER_STALE_S', 180.0))
LOC_WORKER_POLL_S = float(os.environ.get('SRS_LOC_WORKER_POLL_S', 15.0))
# HOURS, NOT ONE: ordinary operation measured a 26-minute idle streak, and a CPU session burns no
# GPU quota, so retiring a live worker costs more than keeping an idle one.  The session cap
# retires workers; this is only the backstop for an orphan whose GPU session died.
LOC_WORKER_IDLE_MAX_S = float(os.environ.get('SRS_LOC_WORKER_IDLE_MAX_S', 6 * 3600.0))
LOC_BOOT_SOURCE = {}                  # boot_source -> count, reported in state and heartbeat
LOC_BREAKER = {'consec': 0, 'skip_left': 0, 'opened': 0}
LOC_WORKERS_SEEN = {'t': 0.0, 'alive': None, 'detail': [], 'last_result_t': 0.0,
                    'beats': {}, 'beats_t': 0.0}
LOC_BOOT_STALL_S = [0.0]              # BILLED idle: seconds the GPU sat waiting with no own work

# ---- the second channel: the roll's TAIL (two-pass intrinsics + the final PnP) --------------
# Measured on the live run, 108 complete rolls: pnp_s p50 43.35 s and intrinsics_s p50 5.35 s,
# so the tail is ~49 s of a ~200 s roll -- a quarter of it, on two physical cores, needing the
# map and the correspondences and nothing on the accelerator.  Exactly the bootstrap's shape.
#
# ⚠ IT IS COLLECTED A ROLL LATE, AND THAT IS THE WHOLE DESIGN.  A worker box is the same 4
# vCPU / 2 physical as the GPU box, so it computes the tail in the same ~49 s the GPU would;
# every second of transfer and polling on top of that is pure loss.  Posted after match and
# waited for inside the same roll, the round trip is ~60 s against ~49 s local and the offload
# CANNOT pay -- it would time out on every roll and bill the wait as well.  So the question goes
# out the instant the correspondences exist and the answer is consumed at the END OF THE NEXT
# ROLL, which gives the worker the same full-roll head start the bootstrap already gets.  The
# roll's record, its bank and its cursor advance all move with it; nothing about WHAT is
# computed changes, only when.
LOC_TAIL_SPLIT = os.environ.get('SRS_LOC_TAIL', '1') == '1'
# A TIMEOUT, not a delay -- a generous one costs nothing when the answer arrives in time, and
# only a dead pool pays it.  Tail cost scales with correspondences: 90-160 s on dense rolls.
LOC_TAIL_WAIT_S = float(os.environ.get('SRS_LOC_TAIL_WAIT_S', 150.0))
LOC_TAIL_WAIT_MAX_S = float(os.environ.get('SRS_LOC_TAIL_WAIT_MAX_S', 600.0))
# Shorter than the bootstrap's 10 s: the answer is ~2 MB and the poll is the last thing between
# a finished roll and the next one, so a slow poll is billed idle in its purest form.
LOC_TAIL_POLL_S = float(os.environ.get('SRS_LOC_TAIL_POLL_S', 4.0))
LOC_TAIL_SOURCE = {}                  # tail_source -> count, beside boot_source
LOC_TAIL_BREAKER = {'consec': 0, 'skip_left': 0, 'opened': 0}
LOC_TAIL_STALL_S = [0.0]

# ---- keeping the network off the thread the accelerator waits behind -----------------------
# Measured over 108 rolls of the live run: 43.5 s p50 between the `stages` line and the `solved`
# line -- 22% of a 200 s roll -- spent compressing and pushing the two output objects with
# everything else stopped, and a further 24.2 s p50 uploading 194 MB of anchors on the
# look-ahead thread.  The anchor upload does not block today (the look-ahead's wait was 0.0 s on
# 107 of 107 rolls) but its margin is gone the moment the tail leaves: the look-ahead chain is
# fetch + extract + post ~= 110 s against a solve span that drops to ~110 s with the tail
# offloaded.  Both go to background threads and are joined where correctness needs them.
LOC_POST_ASYNC = os.environ.get('SRS_LOC_POST_ASYNC', '1') == '1'
# How many anchor uploads may be in flight before the next post waits for the oldest.  2 is one
# per resident roll; the cap exists so a slow Drive queues rather than piling ~200 MB temp files
# on /tmp without bound.
LOC_POST_MAX_INFLIGHT = int(os.environ.get('SRS_LOC_POST_MAX_INFLIGHT', 2))
LOC_BANK_ASYNC = os.environ.get('SRS_LOC_BANK_ASYNC', '1') == '1'
# Per-roll I/O accounting, so the next reader can SEE whether the overlap worked instead of
# inferring it from the wall clock.  `_bg` is what the background thread spent, `_block` is what
# the main thread waited for it; overlapped = bg - block.
LOC_IO = {'post_bg_s': 0.0, 'post_block_s': 0.0, 'bank_bg_s': 0.0, 'bank_block_s': 0.0,
          'fetch_bg_s': 0.0, 'fetch_block_s': 0.0, 'rm_bg_s': 0.0, 'n_rolls': 0}
LOC_POOLS = {}
LOC_POST_FUT = {}                     # request name -> the future uploading it
# The two channels a worker serves, TAIL FIRST: a bootstrap question is posted a roll before its
# answer is needed and a tail question is the last thing between a finished roll and the next
# one, so latency is worth more on the second.
LOC_CHANNELS = (
    # (key, request remote, answer remote, heartbeat stage)
    ('tail', 'LOC_TAIL_REMOTE', 'LOC_TAILRES_REMOTE', 'worker_tail'),
    ('boot', 'LOC_ANCHOR_REMOTE', 'LOC_BOOT_REMOTE', 'worker_bootstrap'),
)
# The scalar per-frame columns of a PnP result, and the type each is carried in.  float64
# throughout: `loc_record` is the one place that narrows them, and it must narrow the worker's
# answer and this session's identically.
LOC_TAIL_COLS = (('ok', np.uint8), ('n_corr', np.int32), ('n_inl', np.int32),
                 ('n_sup_img', np.int32), ('run_mask', np.int64),
                 ('arc', np.float64), ('lat', np.float64),
                 ('inl_arc_span', np.float64), ('inl_arc_med', np.float64),
                 ('rep_p50', np.float64), ('rep_p90', np.float64))

# ---- the runtime A/B: does match get faster on two devices now the bootstrap has left? -----
# LOC_MATCH_DEVICES=1 is a settled v5 measurement (match 19-34 s -> 45.59 s, cuda:1 50.56 vs
# cuda:0 21.22 ms/frame) -- but that was taken while the bootstrap was ALSO using both devices,
# so it never separated "the second device is slow" from "the second device was busy".  With the
# bootstrap off the box entirely the question is open again and has to be re-measured.
#
# It is measured as a CONTROLLED SAME-INPUT comparison, both arms on the same roll, because
# kp/frame varies 2.5x across rolls (1,244 -> 3,097) and a between-rolls A/B would be reading
# that variation rather than the device count.  Two arms x ~40 s x LOC_MATCH_AB_ROLLS rolls is
# the whole cost, and the decision is written into state.json so it is paid ONCE PER run_id and
# not again on every resume.
LOC_MATCH_AB = os.environ.get('SRS_LOC_MATCH_AB', '1') == '1'
LOC_MATCH_AB_ROLLS = int(os.environ.get('SRS_LOC_MATCH_AB_ROLLS', 2))
# How much faster two devices must be before the incumbent is displaced.  A 3% win is noise on a
# 40 s stage; ties go to the configuration that is already known to work.
LOC_MATCH_AB_MARGIN = float(os.environ.get('SRS_LOC_MATCH_AB_MARGIN', 0.03))
LOC_MATCH_AB_STATE = [None]           # the decision, loaded from state.json
LOC_MATCH_DEVICES_EFF = [LOC_MATCH_DEVICES]

# How stale Drive's copy of kernel.log may get.  60 s is well under the shortest stage that
# logs (extract, ~80 s) so no stage passes unseen, and at ~1 s per rclone call it costs under
# 2% of a stage even if every line tried to push.
LOC_LOG_PUSH_S = float(os.environ.get('SRS_LOC_LOG_PUSH_S', 60.0))
LOC_LOG_PUSHED = [0.0]

LOC_T0 = time.time()
LOC_RUN_ID = None
LOC_HB = {'t': 0.0}
LOC_HB_EXTRA = {}                     # merged into every heartbeat (the GPU inventory)
# Filled by the loc-split block with its own reporter.  An indirection rather than a call,
# because loc-io comes before loc-split and this file has to stay liftable into localize.ipynb
# one cell at a time -- a cell that called forward would not run on its own.
LOC_HB_HOOK = [None]
# cumulative session wall-clock banked by EARLIER sessions of this run.  Kaggle bills session
# wall-clock, so that -- not GPU-busy time -- is what the 20 h budget counts; the name
# `gpu_seconds` is the contract's.  Every save re-derives the total, so a session that dies
# without reaching its final save still leaves the last roll's total on the remote.
LOC_BASE_S = [0.0]

## `loc-io`

Session identity and role, logging, rclone, resume state and heartbeats.

In [ ]:
# === cell: loc-io ===
def loc_session_id():
    """A stable, unique id for THIS session, cached on /tmp so a re-exec keeps it.

    It names the session's own state/heartbeat/log on the remote, so it has to survive
    `os.execve` (which changes the pid) and it has to differ between two CPU sessions started
    from two kernels with the same source.  Host + pid + start time hashed gives both.
    """
    if LOC_SESSION_ID[0] is None:
        env = os.environ.get('SRS_LOC_SESSION_ID')
        p = LOC_ROOT / 'session_id'
        if env:
            LOC_SESSION_ID[0] = env
        elif p.exists():
            LOC_SESSION_ID[0] = p.read_text().strip()
        else:
            import socket
            seed = f'{socket.gethostname()}|{os.getpid()}|{LOC_T0}'
            LOC_SESSION_ID[0] = hashlib.sha256(seed.encode()).hexdigest()[:8]
        p.parent.mkdir(parents=True, exist_ok=True)
        p.write_text(LOC_SESSION_ID[0])
    return LOC_SESSION_ID[0]


def loc_role():
    return LOC_ROLE[0]


def loc_set_role(role):
    """Fix this session's role and, with it, the three objects it owns on the remote.

    The GPU session keeps the un-split names, so every existing tool -- b5_monitor.py, b6_stop.py
    and any hand `rclone cat` -- goes on reading the same paths it always has.  A CPU session
    takes suffixed names instead: it is a helper, and a helper that overwrote the run's state
    would be worse than no helper at all.
    """
    assert role in ('gpu', 'cpu'), role
    LOC_ROLE[0] = role
    sid = loc_session_id()
    sfx = '' if role == 'gpu' else f'_cpu_{sid}'
    LOC_STATE_NAME[0] = f'state{sfx}.json'
    LOC_HB_NAME[0] = f'heartbeat{sfx}.json'
    LOC_LOG_NAME[0] = f'kernel{sfx}.log'
    LOC_HB_EXTRA.update(role=role, session_id=loc_session_id(), split=LOC_SPLIT)
    return role


def loc_resolve_role(inv):
    """gpu | cpu, from SRS_LOC_ROLE or -- when it says `auto` -- from the GPU inventory.

    Only a COMPLETED probe that AGREES WITH ITSELF and found zero devices demotes a session.  An
    incomplete probe is a broken instrument, not evidence of a CPU box, and loc_gpu_gate still
    exits 8 on it: silently turning a GPU session into a CPU one would spend a GPU-quota session
    doing CPU work.  The same goes for a probe whose two counts disagree -- `n_gpus` is their
    MINIMUM, so a transient nvidia-smi failure on a real 2xT4 box reads as zero, and demoting on
    that would quietly turn the run's only GPU session into a second helper.  Disagreement keeps
    the gpu role, and the gate then exits 7 loudly, which is the outcome that can be seen.
    """
    want = LOC_ROLE_ENV
    if want in ('gpu', 'cpu'):
        return loc_set_role(want)
    if not LOC_SPLIT or LOC_DRY:
        # a dry run's inventory is a laptop's, not a session's: the local harness has zero CUDA
        # devices and is not thereby a CPU worker.  It asks for a role explicitly when it wants
        # one, and the auto path is exercised against stand-in inventories instead.
        return loc_set_role('gpu')
    if (inv.get('probe_complete') and not inv.get('counts_disagree')
            and (inv.get('n_gpus') or 0) == 0):
        return loc_set_role('cpu')
    return loc_set_role('gpu')


def loc_log(msg, push_=False):
    """One line to the console, the on-disk log, /kaggle/working, and -- on a cadence -- Drive.

    The cadence is the point.  v3 only pushed the lines that passed push_=True, which is the
    startup and gate lines and nothing else, so Drive's log/kernel.log stalled at the re-exec
    while the console carried every per-roll line: b5_monitor.py and every remote checkpoint
    read Drive, so the run was invisible for its whole length.  Pushing every line instead
    would be an rclone process per line, ~1 s each, inside stages that log per frame.  So:
    push when asked, and otherwise at most every LOC_LOG_PUSH_S, which bounds Drive's staleness
    without bounding how often the kernel may log.
    """
    line = f'[{time.strftime("%H:%M:%S")} {(time.time() - LOC_T0) / 60:6.1f}m] {msg}'
    print(line, flush=True)
    LOC_LOGF.parent.mkdir(parents=True, exist_ok=True)
    with open(LOC_LOGF, 'a') as f:
        f.write(line + '\n')
    if LOC_KWORK.exists():
        shutil.copy(LOC_LOGF, LOC_KWORK / LOC_LOGF.name)
    if push_ or (time.time() - LOC_LOG_PUSHED[0]) >= LOC_LOG_PUSH_S:
        LOC_LOG_PUSHED[0] = time.time()
        try:
            loc_push(LOC_LOGF, LOC_LOG_NAME[0], LOC_LOG_REMOTE)
        except Exception as e:                                  # noqa: BLE001
            # a log push must never be able to kill the run it is only reporting on
            print(f'log push failed (non-fatal): {type(e).__name__}: {e}', flush=True)


def loc_physical_cores():
    """Physical cores, which is what the CPU pools should be sized to.

    Measured: a 4-physical/8-logical machine runs 40 PnP solves in 0.215 s on 4 workers and
    0.286 s on 8 -- hyperthreads make a RANSAC pool slower, not faster.  /proc/cpuinfo's
    (physical id, core id) pairs are the answer where it exists; os.cpu_count() is the
    fallback, and it can only over-count, which costs ~33% rather than being wrong.
    """
    try:
        cur, seen = {}, set()
        for line in Path('/proc/cpuinfo').read_text().split('\n'):
            if ':' in line:
                k, v = (x.strip() for x in line.split(':', 1))
                if k in ('physical id', 'core id'):
                    cur[k] = v
            elif cur:
                seen.add((cur.get('physical id'), cur.get('core id')))
                cur = {}
        if cur:
            seen.add((cur.get('physical id'), cur.get('core id')))
        seen.discard((None, None))
        if seen:
            return len(seen)
    except Exception:                                           # noqa: BLE001
        pass
    return os.cpu_count() or 1


def loc_workers(n=None):
    """How many CPU workers to run, honouring the affinity mask this process actually has."""
    if LOC_PNP_WORKERS:
        return max(1, LOC_PNP_WORKERS)
    try:
        avail = len(os.sched_getaffinity(0))
    except AttributeError:                                      # non-Linux
        avail = os.cpu_count() or 1
    return max(1, min(loc_physical_cores(), avail, n or 10 ** 6))


def loc_pmap(fn, items, workers=None, tag='', initializer=None):
    """`[fn(x) for x in items]`, over a thread pool, in the SAME ORDER.

    Only ever used for work whose per-item result is independent and deterministic -- PnP with
    ransac.num_threads=1 and a fixed seed, faiss search with one OpenMP thread -- so the output
    list is identical, element for element, to the serial comprehension.  Both were verified
    bit-identical against the serial path before being adopted; that is the whole reason a pool
    is allowed anywhere near the in-map controls.
    """
    items = list(items)
    w = workers or loc_workers(len(items))
    if w <= 1 or len(items) <= 1:
        return [fn(x) for x in items]
    import concurrent.futures as cf
    prev = LOC_PROF_TAG[0]
    if tag:
        LOC_PROF_TAG[0] = tag
    try:
        with cf.ThreadPoolExecutor(max_workers=w, initializer=initializer) as ex:
            return list(ex.map(fn, items))
    finally:
        LOC_PROF_TAG[0] = prev


def loc_prof(name, t0):
    """Close one span and return `now`, so consecutive spans cost one perf_counter each.

    Spans raised from inside loc_pmap sum to CORE-seconds, not wall-clock, and the accumulate
    is deliberately unlocked: a lock per span in a per-frame loop would distort the thing it
    measures more than the occasional lost update does.
    """
    now = time.perf_counter()
    e = LOC_PROF.get(LOC_PROF_TAG[0] + name)
    if e is None:
        LOC_PROF[LOC_PROF_TAG[0] + name] = [now - t0, 1]
    else:
        e[0] += now - t0
        e[1] += 1
    return now


def loc_prof_sync():
    """Make the NEXT span boundary mean something on CUDA.  No-op unless profiling deeply."""
    if LOC_PROF_SYNC[0]:
        try:
            import torch
            if torch.cuda.is_available():
                torch.cuda.synchronize()
        except Exception:                                       # noqa: BLE001
            pass


def loc_prof_report(n_frames=0, reset=True):
    # whether the GPU spans mean anything: unsynced, CUDA's asynchrony parks the whole GPU
    # tail on whichever call forces the sync, and a reader sees a huge 'd2h' that is really
    # the GEMM and the reduce.  That misread cost a whole round of optimising the wrong thing.
    out = {'_gpu_spans_synced': {'s': 0.0, 'n': int(LOC_PROF_SYNC[0]), 'ms_per_frame': None}}
    out.update({k: {'s': round(v[0], 3), 'n': v[1],
               'ms_per_frame': (round(1000.0 * v[0] / n_frames, 3) if n_frames else None)}
                for k, v in sorted(LOC_PROF.items(), key=lambda kv: -kv[1][0])})
    if reset:
        LOC_PROF.clear()
    return out


def loc_sh(cmd, check=True, timeout=None, env=None):
    r = subprocess.run(cmd, shell=True, capture_output=True, text=True, timeout=timeout,
                       env=env)
    if r.returncode and check:
        raise RuntimeError(f'{cmd}\n{r.stdout[-2000:]}\n{r.stderr[-2000:]}')
    return r


def loc_push(local, name, remote=None):
    """Atomic put: copy to a PER-SESSION .tmp, then move, so no reader sees a half-written object.

    Per-session, not `{name}.tmp`, and that was not cosmetic.  Two workers answering the same
    request wrote the same `{name}.npz.tmp` and then both renamed it; the loser's `rclone moveto`
    failed with "Source doesn't exist", the exception was counted as a bootstrap failure, and
    130-230 s of finished work plus a ~200 MB download went in the bin.  That was every one of
    the 22 `boot_failed` in the live run.  The claim confirmation stops the duplication; this
    stops the duplication from being able to destroy work even when it does happen.
    """
    remote = remote or LOC_OUT_REMOTE
    if LOC_DRY:
        dst = Path(os.environ['SRS_LOC_FAKE_REMOTE']) / remote.split(':', 1)[-1] / name
        dst.parent.mkdir(parents=True, exist_ok=True)
        shutil.copy(local, dst)
        return
    tmp = f'{name}.{loc_session_id()}.tmp'
    loc_sh(f'rclone copyto "{local}" "{remote}/{tmp}"')
    loc_sh(f'rclone moveto "{remote}/{tmp}" "{remote}/{name}"')


def loc_pull(name, local, remote=None):
    remote = remote or LOC_OUT_REMOTE
    if LOC_DRY:
        src = Path(os.environ['SRS_LOC_FAKE_REMOTE']) / remote.split(':', 1)[-1] / name
        if not src.exists():
            return False
        Path(local).parent.mkdir(parents=True, exist_ok=True)
        shutil.copy(src, local)
        return True
    return loc_sh(f'rclone copyto "{remote}/{name}" "{local}"', check=False).returncode == 0


def loc_rm(name, remote=None):
    """Delete one object.  Never raises and never reports a missing object as a failure.

    Only ever used on the split's scratch objects (a roll's anchor features, a claim), and the
    handoff has to survive both sides deleting the same one: absent is the desired state, so
    getting there twice is success, not an error.
    """
    remote = remote or LOC_OUT_REMOTE
    try:
        if LOC_DRY:
            (Path(os.environ['SRS_LOC_FAKE_REMOTE']) / remote.split(':', 1)[-1]
             / name).unlink(missing_ok=True)
            return True
        return loc_sh(f'rclone deletefile "{remote}/{name}"', check=False).returncode == 0
    except Exception as e:                                      # noqa: BLE001
        print(f'delete {remote}/{name} failed (non-fatal): {type(e).__name__}: {e}', flush=True)
        return False


def loc_bg(name, workers=1):
    """A named, lazily-created background thread pool, one per kind of transfer.

    Named rather than shared so a slow anchor upload cannot delay a bank, and one worker each by
    default so the transfers of a kind stay ordered and self-throttling.  ThreadPoolExecutor's
    threads are NOT daemons and the interpreter joins them at exit, which is the property that
    matters here: a session that halts on a gate still finishes the pushes it started.
    """
    ex = LOC_POOLS.get(name)
    if ex is None:
        import concurrent.futures as cf
        ex = LOC_POOLS[name] = cf.ThreadPoolExecutor(max_workers=workers,
                                                     thread_name_prefix=f'loc-{name}')
    return ex


def loc_pools_flush():
    """Wait for every background transfer, and close the pools.  Called once, at the end.

    The interpreter would join these threads at exit anyway, so this is not what makes the
    pushes land -- it is what makes the moment they land OBSERVABLE.  A session that returns
    from loc_main with deletes still queued leaves a remote that disagrees with its own state
    file for a few seconds, which is exactly the window a checker or the next session reads in.
    """
    out = {}
    for name, ex in list(LOC_POOLS.items()):
        t0 = time.time()
        ex.shutdown(wait=True)
        out[name] = round(time.time() - t0, 2)
        LOC_POOLS.pop(name, None)
    return out


def loc_rm_async(names_remotes):
    """Delete objects on a background thread.  A delete is housekeeping, never a dependency.

    Retiring one request is two `rclone deletefile` calls and they sat on the main thread
    between the bootstrap answer and the match, which is the thread the accelerator waits
    behind.  Nothing downstream reads the deleted object -- the answer is already in hand and
    the name can never be re-minted -- so the only thing lost by not waiting is the log line,
    and `loc_rm` never raises anyway.
    """
    def _rm():
        t0 = time.time()
        for name, remote in names_remotes:
            loc_rm(name, remote)
        LOC_IO['rm_bg_s'] += time.time() - t0
    return loc_bg('rm').submit(_rm)


def loc_stop_requested(force=False, where=None):
    """The graceful-stop flag, or None.  UNREADABLE ALWAYS MEANS NO.

    A transient rclone failure, a half-written object and malformed JSON all read as "no stop
    requested": this is an operator convenience and it must never be able to abort a roll on
    its own.  One `rclone cat` is ~1 s against a ~150 s roll, so the between-rolls poll is
    free; the mid-roll boundary polls are rate-limited to LOC_STOP_POLL_S so that a short
    stage cannot turn into an rclone loop.  Once seen, the answer is cached -- the flag is
    renamed at exit and re-reading it would race that.
    """
    if LOC_STOP['seen']:
        return LOC_STOP['seen']
    now = time.time()
    if not force and (now - LOC_STOP['t']) < LOC_STOP_POLL_S:
        return None
    LOC_STOP['t'] = now
    try:
        p = LOC_ROOT / '_stop.json'
        p.unlink(missing_ok=True)
        if not loc_pull(LOC_STOP_NAME, p, LOC_REMOTE) or not p.exists():
            return None
        d = json.loads(p.read_text())
        if not (isinstance(d, dict) and d.get('stop') is True):
            return None
        LOC_STOP['seen'], LOC_STOP['where'] = d, where
        loc_log(f'*** GRACEFUL STOP REQUESTED (seen at: {where}); requested_utc='
                f'{d.get("requested_utc")} reason={d.get("reason")!r}. The roll in flight is '
                f'finished and BANKED first -- work already paid for on the GPU is never '
                f'abandoned; only the next roll is not started. ***', push_=True)
        return d
    except Exception as e:                                      # noqa: BLE001
        print(f'stop poll failed, treated as NO stop: {type(e).__name__}: {e}', flush=True)
        return None


def loc_stop_clear():
    """Rename the flag once honoured, so it cannot kill the next session too.

    Renamed rather than deleted: the request is the evidence for why the session ended, and it
    costs nothing to keep it beside the log it explains.
    """
    name = f'stop_honoured_{time.strftime("%Y%m%dT%H%M%SZ", time.gmtime())}.json'
    try:
        if LOC_DRY:
            d = Path(os.environ['SRS_LOC_FAKE_REMOTE']) / LOC_REMOTE.split(':', 1)[-1]
            src = d / LOC_STOP_NAME
            if src.exists():
                src.rename(d / name)
            return name
        loc_sh(f'rclone moveto "{LOC_REMOTE}/{LOC_STOP_NAME}" "{LOC_REMOTE}/{name}"',
               check=False)
        return name
    except Exception as e:                                      # noqa: BLE001
        loc_log(f'could not withdraw the stop flag ({type(e).__name__}: {e}) - CLEAR IT BY '
                f'HAND with b6_stop.py --clear, or the next session stops before roll 1',
                push_=True)
        return None


def loc_lsf(remote):
    """Names present under a remote directory; [] when it does not exist."""
    if LOC_DRY:
        d = Path(os.environ['SRS_LOC_FAKE_REMOTE']) / remote.split(':', 1)[-1]
        return sorted(p.name for p in d.iterdir()) if d.exists() else []
    r = loc_sh(f'rclone lsf "{remote}"', check=False)
    return sorted(x.strip('/') for x in (r.stdout or '').split('\n') if x.strip())


def loc_lsf_t(remote):
    """[(epoch, name), ...] from ONE listing, no reads.

    `lsjson`, not `lsf --format t`: that one prints in the machine's LOCAL zone, so an age
    computed from it is silently hours out on a box in a different zone.
    """
    if LOC_DRY:
        d = Path(os.environ['SRS_LOC_FAKE_REMOTE']) / remote.split(':', 1)[-1]
        return sorted((p.stat().st_mtime, p.name) for p in d.iterdir()) if d.exists() else []
    r = loc_sh(f'rclone lsjson "{remote}"', check=False)
    try:
        listing = json.loads(r.stdout or '[]')
    except Exception:                                           # noqa: BLE001
        return []
    out = []
    for d in listing:
        n, t = d.get('Name'), d.get('ModTime')
        if not n or not t or d.get('IsDir'):
            continue
        try:
            out.append((calendar.timegm(time.strptime(t[:19], '%Y-%m-%dT%H:%M:%S')), n))
        except ValueError:
            continue
    return out


def loc_run_id():
    """Identity of THIS kernel, so resume cannot cross runs.

    state.json on the remote outlives the run that wrote it, and a stale copy would make the
    kernel skip rolls it never processed.  Keying resume on the kernel's own sha means a
    re-exec or a VM reclaim resumes and a new push starts clean, with no flag to remember.

    BOTH credential lines are redacted.  rclone rewrites ~/.config/rclone/rclone.conf on every
    OAuth token refresh, so a raw hash changes with no code change at all -- and this kernel
    inlines TWO configs (srs: for the [[gdrive]] videos, yusufishabazz: for the rest and for
    the work directory), so redacting one of them would still let a token rotation on the other
    invalidate the run id and silently discard the state it was pushed to resume.

    LOC_BUDGET_H_DEFAULT is redacted for the same reason and one more.  It is inlined at push
    time because Kaggle metadata carries no environment, and the operator will re-push with a
    smaller number as hours are spent; that is an OPERATING decision about how much more to
    spend, not a change to what is computed, and it must not throw away the cursor.  It is also
    what lets the ONE source produce three kernels -- gpu, cpu, cpu -- that share a run identity.
    """
    try:
        txt = re.sub(r'(?m)^RCLONE_CONF_\w+ = .*$', 'RCLONE_CONF = <redacted>',
                     Path(__file__).read_text())
        n = len(re.findall(r'(?m)^RCLONE_CONF = <redacted>$', txt))
        assert n == 2, f'expected 2 redacted credential lines, redacted {n}'
        txt, nb = re.subn(r'(?m)^LOC_BUDGET_H_DEFAULT = .*$',
                          'LOC_BUDGET_H_DEFAULT = <redacted>', txt)
        assert nb == 1, f'expected 1 budget line to redact, redacted {nb}'
        return hashlib.sha256(txt.encode()).hexdigest()[:16] + (('-' + LOC_TAG) if LOC_TAG else '')
    except Exception:
        return 'unknown'


def loc_jsonable(o):
    if isinstance(o, (np.integer,)):
        return int(o)
    if isinstance(o, (np.floating,)):
        return float(o)
    if isinstance(o, np.ndarray):
        return o.tolist()
    return str(o)


def loc_load_state():
    """Resume state, with FOREIGN state discarded (different kernel source = different run)."""
    global LOC_RUN_ID
    LOC_RUN_ID = loc_run_id()
    fresh = {'run_id': LOC_RUN_ID, 'started_utc': time.strftime('%Y-%m-%dT%H:%M:%SZ',
                                                                time.gmtime()),
             'cursor': 0, 'done': [], 'gpu_seconds': 0.0, 'chunk': 1}
    LOC_OUT.mkdir(parents=True, exist_ok=True)
    if os.environ.get('SRS_LOC_RESUME') == '0':
        return fresh
    if not (loc_pull(LOC_STATE_NAME[0], LOC_STATEF, LOC_REMOTE) and LOC_STATEF.exists()):
        return fresh
    try:
        st = json.loads(LOC_STATEF.read_text())
    except json.JSONDecodeError:
        return fresh
    if st.get('run_id') == LOC_RUN_ID:
        st['chunk'] = int(st.get('chunk', 0)) + 1
        LOC_BASE_S[0] = float(st.get('gpu_seconds', 0.0))
        return st
    loc_log(f'state.json on the remote belongs to a DIFFERENT kernel '
            f'(run_id {st.get("run_id")} != {LOC_RUN_ID}); starting clean rather than '
            f'inheriting its cursor, done list and budget')
    return dict(fresh, previous_run_id=st.get('run_id'),
                previous_done=len(st.get('done') or []),
                previous_gpu_seconds=float(st.get('gpu_seconds') or 0.0))


def loc_save_state(st):
    st['run_id'] = LOC_RUN_ID or loc_run_id()
    st['role'] = loc_role()
    st['session_id'] = loc_session_id()
    st['gpu_seconds'] = round(LOC_BASE_S[0] + (time.time() - LOC_T0), 1)
    LOC_STATEF.write_text(json.dumps(st, indent=1, default=loc_jsonable))
    loc_push(LOC_STATEF, LOC_STATE_NAME[0], LOC_REMOTE)


def loc_heartbeat(rec, force=False):
    """<= every LOC_HEARTBEAT_S seconds, so a stall is visible from the laptop."""
    now = time.time()
    if not force and now - LOC_HB['t'] < LOC_HEARTBEAT_S:
        return
    LOC_HB['t'] = now
    rec = dict(rec, **LOC_HB_EXTRA)
    rec['utc'] = time.strftime('%Y-%m-%dT%H:%M:%SZ', time.gmtime())
    rec['session_started_utc'] = time.strftime('%Y-%m-%dT%H:%M:%SZ', time.gmtime(LOC_T0))
    rec['min_to_cap'] = rec.get('min_to_cap', round(LOC_SESSION_MAX_MIN - loc_session_min(), 1))
    rec['budget_h'] = LOC_BUDGET_H
    rec['budget_used_h'] = round((LOC_BASE_S[0] + (now - LOC_T0)) / 3600.0, 3)
    # the split's numbers ride on every heartbeat, on BOTH roles, so one monitor read covers the
    # whole run: billed idle, where each bootstrap came from, and which match config is in force
    if LOC_HB_HOOK[0] is not None:
        try:
            rec['split'] = LOC_HB_HOOK[0]()
        except Exception:                                       # noqa: BLE001
            pass
    p = LOC_ROOT / 'heartbeat.json'
    p.write_text(json.dumps(rec, indent=1, default=loc_jsonable))
    try:
        loc_push(p, LOC_HB_NAME[0], LOC_REMOTE)
    except Exception as e:
        print(f'heartbeat push failed (non-fatal): {type(e).__name__}: {e}', flush=True)


def loc_session_min():
    return (time.time() - LOC_T0) / 60.0


def loc_freespace(p='/tmp'):
    s = os.statvfs(p)
    return s.f_bavail * s.f_frsize / 2 ** 30


def loc_rss_b():
    """This process's RESIDENT bytes, or None -- /proc/self/statm field 2, in pages.  NOT
    `info['feats_bytes']`, which is an accounting sum over the arrays one roll holds."""
    try:
        return int(Path('/proc/self/statm').read_text().split()[1]) * os.sysconf('SC_PAGE_SIZE')
    except Exception:                                           # noqa: BLE001
        return None


def loc_vm_hwm_b():
    """Peak RSS this process has ever held, or None.  Separates a high-water mark left by one
    bad roll from a floor that is still rising."""
    try:
        for line in Path('/proc/self/status').read_text().split('\n'):
            if line.startswith('VmHWM:'):
                return int(line.split()[1]) * 1024
    except Exception:                                           # noqa: BLE001
        pass
    return None


def loc_malloc_trim(tag=None):
    """Return freed-but-retained heap to the OS, and report what that was worth.

    A roll's features are thousands of ~1 MB arrays, below glibc's 32 MiB mmap threshold, so they
    come from brk and none of it returns on `del` -- measured, 1.68 GiB freed 0 at `del` and
    1,714 MiB at the trim.  Anything larger is mmap'd and comes back either way.

    gc and trim are timed separately: they scale with the live object graph and the free lists
    respectively, so `gc_ms` says which half to drop if this ever costs a roll.  Never raises;
    returns None when it could not run.
    """
    try:
        import ctypes
        import gc
        t0 = time.perf_counter()
        before = loc_rss_b()
        gc.collect()
        t1 = time.perf_counter()
        ctypes.CDLL('libc.so.6').malloc_trim(0)
        t2 = time.perf_counter()
        after = loc_rss_b()
        rec = {'tag': tag, 'ms': round((t2 - t0) * 1000, 1),
               'gc_ms': round((t1 - t0) * 1000, 1), 'trim_ms': round((t2 - t1) * 1000, 1),
               'rss_before_b': before, 'rss_after_b': after,
               'freed_mib': (None if before is None or after is None
                             else round((before - after) / 2 ** 20, 1))}
    except Exception as e:                                      # noqa: BLE001
        LOC_TRIM_LAST[0] = {'tag': tag, 'error': f'{type(e).__name__}: {str(e)[:120]}'}
        return None
    LOC_TRIM_LAST[0] = rec
    return rec


def loc_trim_str():
    """The last trim as one log field, or a placeholder if it never ran or could not."""
    t = LOC_TRIM_LAST[0]
    if not t:
        return 'trim none yet'
    at = '' if t.get('tag') is None else f'@{t["tag"]}'
    if t.get('error'):
        return f'trim{at} unavailable ({t["error"]})'
    f = '?' if t.get('freed_mib') is None else f'{t["freed_mib"]:.0f}'
    return (f'trim{at} {t["ms"]:.1f}ms (gc {t.get("gc_ms")} + trim {t.get("trim_ms")}) '
            f'freed {f} MiB')

## `loc-install`

Runtime installs (onnxruntime, TensorRT, faiss, pycolmap); re-execs once if the CUDA loader environment has to change.

In [ ]:
# === cell: loc-install ===
# Facts about the loader and about ORT's option parser, not tunables, so they live next to the
# code that uses them.  libnvinfer dlopens its BUILDER RESOURCE by name at engine-build time
# from the default loader path: it is not a DT_NEEDED of anything, so ldd never mentions it and
# preloading libnvinfer alone does not cover it.
LOC_CUDA_LIB_ORDER = ('libcudart', 'libnvJitLink', 'libcublasLt', 'libcublas', 'libcudnn',
                      'libcufft', 'libcurand', 'libcusolver', 'libcusparse',
                      'libnvinfer', 'libnvinfer_plugin', 'libnvonnxparser',
                      'libnvinfer_builder_resource')


def loc_pip(args, check=False):
    """pip, always through THIS interpreter -- `pip` on PATH need not be the kernel's."""
    return loc_sh(f'{sys.executable} -m pip {args}', check=check)


def loc_trt_cache_dir(device_id, model):
    """One engine/timing cache per (device, graph), shared by the probes and the runners.

    Shared on purpose: an ALIKED engine build is minutes, and the gate's probe and the runner
    on the same device would otherwise build the same engine twice.
    """
    p = LOC_TRTCACHE / f'trt{device_id}_{Path(model).stem}'
    p.mkdir(parents=True, exist_ok=True)
    return p


def loc_lib_roots():
    """Directories where pip-installed CUDA / cuDNN / TensorRT shared objects live."""
    import glob
    import site
    sps = []
    try:
        sps += list(site.getsitepackages())
    except Exception:                                           # noqa: BLE001
        pass
    sps.append(str(Path(np.__file__).parent.parent))
    roots = []
    for sp in dict.fromkeys(sps):
        roots += sorted(glob.glob(os.path.join(sp, 'nvidia', '*', 'lib')))
        roots += sorted(glob.glob(os.path.join(sp, 'tensorrt_libs')))
        roots += sorted(glob.glob(os.path.join(sp, 'torch', 'lib')))
    return [r for r in dict.fromkeys(roots) if os.path.isdir(r)]


def loc_lib_env(extra=None):
    """os.environ plus an LD_LIBRARY_PATH covering the pip-installed CUDA/TensorRT libs.

    An RTLD_GLOBAL ctypes load makes a soname resolvable in THIS process and does not survive
    fork+exec, so every subprocess probe needs the loader path in its environment as well.
    """
    e = dict(os.environ)
    roots = loc_lib_roots() + ['/usr/local/cuda/lib64', '/usr/lib/x86_64-linux-gnu']
    e['LD_LIBRARY_PATH'] = ':'.join(dict.fromkeys(
        roots + [p for p in e.get('LD_LIBRARY_PATH', '').split(':') if p]))
    e.update(extra or {})
    return e


def loc_cuda_env():
    """What CUDA the IMAGE has, read BEFORE anything is installed.

    onnxruntime-gpu on PyPI tracks the newest CUDA (13.x as of 1.28); on a CUDA 12 image its
    provider library cannot resolve libcublasLt.so.13, ORT drops the provider and every frame
    is extracted on the CPU while two T4s are billed.  Never assume the major -- read it off
    the libraries actually on disk.
    """
    import glob
    out = {'python': sys.version.split()[0], 'libs': {},
           'nvcc': (loc_sh('nvcc --version 2>/dev/null | tail -1',
                           check=False).stdout or '').strip()}
    dirs = ['/usr/local/cuda/lib64', '/usr/lib/x86_64-linux-gnu'] + loc_lib_roots()
    dirs += sorted(glob.glob('/usr/local/cuda-*/lib64'))
    for stem in ('libcublas', 'libcublasLt', 'libcudart', 'libcudnn', 'libnvinfer'):
        found = {}
        for d in dict.fromkeys(dirs):
            for so in glob.glob(os.path.join(d, stem + '.so.*')):
                m = re.search(r'\.so\.(\d+)', os.path.basename(so))
                if m:
                    found.setdefault(int(m.group(1)), os.path.dirname(so))
        out['libs'][stem] = {str(k): v for k, v in sorted(found.items())}
    majors = sorted(int(k) for k in out['libs'].get('libcublas', {}))
    out['cublas_majors'] = majors
    out['image_cuda_major'] = majors[-1] if majors else None
    return out


def loc_preload_gpu_libs(major=None):
    """ctypes-preload the CUDA / cuDNN / TensorRT objects that ship inside pip wheels.

    onnxruntime dlopens its provider libraries by soname and LD_LIBRARY_PATH is frozen at
    process start, so exporting it from python is too late.  An RTLD_GLOBAL load is not: once
    a soname is resident, ORT's dlopen resolves against it.  `major` keeps a stray second CUDA
    toolkit out of the process.
    """
    import ctypes
    import glob
    roots = loc_lib_roots()
    loaded, failed = [], []
    for stem in LOC_CUDA_LIB_ORDER:
        for d in roots:
            for so in sorted(glob.glob(os.path.join(d, stem + '.so*'))):
                base = os.path.basename(so)
                m = re.search(r'\.so\.(\d+)', base)
                if major and m and stem.startswith('libcu') and int(m.group(1)) != int(major):
                    continue
                try:
                    ctypes.CDLL(so, mode=ctypes.RTLD_GLOBAL)
                    loaded.append(base)
                except OSError as e:
                    failed.append(f'{base}: {str(e)[:80]}')
    return {'roots': roots, 'loaded': loaded, 'failed': failed[:10], 'major': major}


def loc_restore_numpy():
    """Pin numpy back to the version the image's compiled extensions were built against.

    `pip install --force-reinstall <pkg>` reinstalls the whole resolved set, numpy included,
    and the replacement disagrees with the image's prebuilt umath.  --no-deps on every recipe
    prevents it; this is the belt to that pair of braces, and it is cheap.
    """
    r = loc_pip(f'install -q --no-deps --force-reinstall "numpy=={LOC_NUMPY0["version"]}"')
    return {'pinned_to': LOC_NUMPY0['version'], 'rc': r.returncode,
            'tail': ((r.stderr or '') + (r.stdout or ''))[-200:] if r.returncode else ''}


LOC_ENV_PROBE_SRC = r'''
"""Is the interpreter still usable after the pip installs?  A FRESH process on purpose -- the
parent holds an already-imported numpy that would hide a mismatch on disk."""
import json, sys
out = {}
try:
    import numpy
    out['numpy'] = numpy.__version__
    out['numpy_file'] = numpy.__file__
    import numpy.ma                 # the canary: a swapped numpy breaks this, not plain numpy
    m = numpy.ma.masked_array([1.0, 2.0, 3.0], mask=[0, 1, 0])
    out['masked_mean'] = float(m.mean())
    assert out['masked_mean'] == 2.0, out['masked_mean']
    import sqlite3
    out['sqlite3'] = sqlite3.sqlite_version
    import cv2
    out['cv2'] = cv2.__version__
    import onnx
    out['onnx'] = onnx.__version__
    out['ok'] = True
except Exception as e:
    out['ok'] = False
    out['error'] = f'{type(e).__name__}: {str(e)[:400]}'
print('LOC_ENV ' + json.dumps(out))
'''


def loc_check_env():
    """numpy / numpy.ma / cv2 / sqlite3 / onnx all importable in a fresh process.  Never raises."""
    src = LOC_ROOT / 'loc_env_probe.py'
    src.parent.mkdir(parents=True, exist_ok=True)
    src.write_text(LOC_ENV_PROBE_SRC)
    r = loc_sh(f'{sys.executable} "{src}"', check=False)
    for line in (r.stdout or '').splitlines():
        if line.startswith('LOC_ENV '):
            return json.loads(line[len('LOC_ENV '):])
    return {'ok': False, 'error': f'env probe produced no verdict (rc={r.returncode}): '
                                  f'{((r.stderr or "") + (r.stdout or ""))[-500:]}'}


LOC_PKG_PROBE_SRC = r'''
"""Is <module> importable AND does it carry the symbols this pipeline calls?  A subprocess,
so a half-installed extension module cannot take the kernel down with it."""
import importlib, json, sys
mod, attrs = sys.argv[1], [a for a in sys.argv[2].split(',') if a]
out = {'module': mod}
try:
    import numpy.ma                      # the same canary: an install that broke numpy is not ok
    m = importlib.import_module(mod)
    out['version'] = str(getattr(m, '__version__', '?'))
    out['file'] = getattr(m, '__file__', '?')
    out['missing_attrs'] = [a for a in attrs if not hasattr(m, a)]
    out['ok'] = not out['missing_attrs']
except Exception as e:
    out['ok'] = False
    out['error'] = f'{type(e).__name__}: {str(e)[:300]}'
print('LOC_PKG ' + json.dumps(out))
'''


def loc_probe_module(mod, attrs=()):
    """The module's own verdict about itself, from a fresh interpreter."""
    src = LOC_ROOT / 'loc_pkg_probe.py'
    src.parent.mkdir(parents=True, exist_ok=True)
    src.write_text(LOC_PKG_PROBE_SRC)
    r = loc_sh(f'{sys.executable} "{src}" "{mod}" "{",".join(attrs)}"', check=False,
               env=loc_lib_env())
    for line in (r.stdout or '').splitlines():
        if line.startswith('LOC_PKG '):
            return json.loads(line[len('LOC_PKG '):])
    return {'module': mod, 'ok': False,
            'error': f'no verdict (rc={r.returncode}): '
                     f'{((r.stderr or "") + (r.stdout or ""))[-400:]}'}


def loc_pkg_recipes():
    """Ladders for the non-onnxruntime imports, `preinstalled` first.

    faiss is used CPU-only here (read_index / IVF nprobe / search), so the cpu wheel is the
    first thing to try; pycolmap has to be the GPU build the map was made with, because the
    PnP options and `estimate_and_refine_absolute_pose` are that build's API.
    """
    return {
        'faiss': [('preinstalled', None),
                  ('faiss-cpu', 'install -q --no-deps faiss-cpu'),
                  ('faiss-cpu-deps', 'install -q faiss-cpu'),
                  ('faiss-gpu-cu12', 'install -q --no-deps faiss-gpu-cu12')],
        'pycolmap': [('preinstalled', None),
                     ('pycolmap-gpu-wheel', f'install -q --no-deps "{LOC_PYCOLMAP_WHEEL}"'),
                     ('pycolmap-pypi', 'install -q --no-deps pycolmap')],
    }


def loc_pkg_attrs():
    """The symbols each module must actually carry; an import alone proves too little."""
    return {'faiss': ('read_index', 'extract_index_ivf', 'omp_set_num_threads'),
            'pycolmap': ('Camera', 'AbsolutePoseEstimationOptions',
                         'AbsolutePoseRefinementOptions',
                         'estimate_and_refine_absolute_pose')}


def loc_install_pkgs():
    """faiss and pycolmap, each proven by import + the symbols this pipeline calls.

    Every attempt is recorded including the failures, and numpy is pinned back before a recipe
    is judged: a package that works but leaves the interpreter broken is not a working recipe.
    """
    attrs = loc_pkg_attrs()
    out = {}
    for mod, ladder in loc_pkg_recipes().items():
        rec = {'attempts': []}
        for name, args in ladder:
            att = {'recipe': name, 'cmd': args}
            if args:
                r = loc_pip(args)
                att['rc'] = r.returncode
                att['tail'] = ((r.stderr or '') + (r.stdout or ''))[-300:] if r.returncode else ''
                att['numpy_restore'] = loc_restore_numpy()
            att['probe'] = loc_probe_module(mod, attrs.get(mod, ()))
            att['ok'] = bool(att['probe'].get('ok'))
            rec['attempts'].append(att)
            loc_log(f'  {mod} recipe {name}: rc={att.get("rc")} ok={att["ok"]} '
                    f'version={att["probe"].get("version")} '
                    f'{att["probe"].get("error") or att["probe"].get("missing_attrs") or ""}')
            if att['ok']:
                rec.update(chosen=name, ok=True, version=att['probe'].get('version'),
                           file=att['probe'].get('file'))
                break
        rec.setdefault('ok', False)
        out[mod] = rec
    return out


LOC_PROBE_SRC = r'''
"""Does a GPU execution provider ACTUALLY execute this graph?  Run as a SUBPROCESS.

A subprocess for two reasons: an onnxruntime whose CUDA major does not match the image can
abort the interpreter, and after a pip reinstall the parent's already-imported onnxruntime
would not pick up the new shared objects.  The verdict is the NODE-PROVIDER HISTOGRAM out of
ORT's own profile -- never get_available_providers(), which lists REGISTERED providers rather
than loadable ones and says nothing about what ran.
"""
import ctypes, glob, json, os, re, site, subprocess, sys

CUDA_LIB_ORDER = ('libcudart', 'libnvJitLink', 'libcublasLt', 'libcublas', 'libcudnn',
                  'libcufft', 'libcurand', 'libcusolver', 'libcusparse',
                  'libnvinfer', 'libnvinfer_plugin', 'libnvonnxparser',
                  'libnvinfer_builder_resource')
model, ep, dev, prefix, major, ph, pw, maxkp, minscore, cache, fp16 = sys.argv[1:12]
out = {'ep': ep, 'ld_library_path': os.environ.get('LD_LIBRARY_PATH', '')}
try:
    sps = list(site.getsitepackages())
    try:
        import numpy as _np0
        sps.append(os.path.dirname(os.path.dirname(_np0.__file__)))
    except Exception:
        pass
    roots = []
    for sp in dict.fromkeys(sps):
        roots += sorted(glob.glob(os.path.join(sp, 'nvidia', '*', 'lib')))
        roots += sorted(glob.glob(os.path.join(sp, 'tensorrt_libs')))
        roots += sorted(glob.glob(os.path.join(sp, 'torch', 'lib')))
    roots = list(dict.fromkeys(roots))
    got, failed = [], []
    for stem in CUDA_LIB_ORDER:
        for d in roots:
            for so in sorted(glob.glob(os.path.join(d, stem + '.so*'))):
                m = re.search(r'\.so\.(\d+)', os.path.basename(so))
                if major != '0' and m and stem.startswith('libcu') and m.group(1) != major:
                    continue
                try:
                    ctypes.CDLL(so, mode=ctypes.RTLD_GLOBAL); got.append(os.path.basename(so))
                except OSError as e:
                    failed.append(os.path.basename(so) + ': ' + str(e)[:120])
    out['preloaded'] = len(got)
    out['preload_failed'] = failed[:6]
    # what is actually ON DISK, listed rather than inferred from a package name
    out['trt_files'] = sorted({os.path.basename(p) for d in roots
                               for pat in ('libnvinfer*', 'libnvonnxparser*')
                               for p in glob.glob(os.path.join(d, pat))})

    import numpy as np
    import onnxruntime as ort
    out['ort'] = ort.__version__
    if ep == 'trt':
        # THE decisive test for the TensorRT EP: dlopen its provider library ourselves, with
        # the preload already done, and report the loader's own message.  ORT catches this
        # failure, prints '*** EP Error ***' and silently rebuilds the session on CUDA, so
        # `realised` tells you the EP is gone but never why.  ALL OF IT IS ADVISORY -- a
        # diagnostic must never be able to fail the thing it diagnoses.
        try:
            mode = getattr(os, 'RTLD_NOW', 0) | ctypes.RTLD_GLOBAL
            prov_so = os.path.join(os.path.dirname(ort.__file__), 'capi',
                                   'libonnxruntime_providers_tensorrt.so')
            out['provider_so_exists'] = os.path.exists(prov_so)
            if out['provider_so_exists']:
                try:
                    ctypes.CDLL(prov_so, mode=mode)
                    out['provider_dlopen'] = 'ok'
                except Exception as e:
                    out['provider_dlopen'] = f'{type(e).__name__}: {str(e)[:400]}'
                try:
                    ldd = subprocess.run(['ldd', prov_so], capture_output=True, text=True).stdout
                    out['provider_ldd_missing'] = sorted(
                        {m for m in re.findall(r'(\S+) => not found', ldd)})
                except Exception as e:
                    out['provider_ldd_missing'] = f'{type(e).__name__}: {str(e)[:200]}'
            try:
                import tensorrt as _trt
                out['tensorrt'] = _trt.__version__
                # the tensorrt wheel's __init__ loads its own bundled libs; retry the dlopen
                # after it, so "importing tensorrt is what fixes the EP" is attributable
                if out.get('provider_dlopen', 'ok') != 'ok':
                    try:
                        ctypes.CDLL(prov_so, mode=mode)
                        out['provider_dlopen_after_trt_import'] = 'ok'
                    except Exception as e:
                        out['provider_dlopen_after_trt_import'] = \
                            f'{type(e).__name__}: {str(e)[:400]}'
            except Exception as e:
                out['tensorrt'] = f'{type(e).__name__}: {str(e)[:200]}'
        except Exception as e:
            out['diag_error'] = f'{type(e).__name__}: {str(e)[:300]}'
    out['available'] = ort.get_available_providers()      # REGISTERED, not loadable
    so_ = ort.SessionOptions()
    so_.enable_profiling = True
    so_.profile_file_prefix = prefix
    if ep == 'trt':
        want = 'TensorrtExecutionProvider'
        # 'True'/'False', NEVER '0'/'1'.  The EP's option parser REJECTS '0' and the session
        # with it, and ORT then falls back to CUDA silently -- four runs of chasing a missing
        # libnvinfer.  The cache is the runners' own, so the engine is built once per device.
        # THE SAME OPTIONS THE RUNNERS USE, fp16 included.  The probe and the runners share an
        # engine cache directory, so a probe that built an fp32 engine here could hand it to
        # the runners; and a histogram measured on a different configuration from the one the
        # run uses is not a measurement of the run.
        opts = {'device_id': dev, 'trt_engine_cache_enable': 'False',
                'trt_fp16_enable': fp16}
        if cache:
            os.makedirs(cache, exist_ok=True)
            opts = {'device_id': dev, 'trt_engine_cache_enable': 'True',
                    'trt_engine_cache_path': cache, 'trt_timing_cache_enable': 'True',
                    'trt_timing_cache_path': cache, 'trt_fp16_enable': fp16}
        provs = [(want, opts), ('CUDAExecutionProvider', {'device_id': dev}),
                 'CPUExecutionProvider']
    elif ep == 'cuda':
        want = 'CUDAExecutionProvider'
        provs = [(want, {'device_id': dev}), 'CPUExecutionProvider']
    else:
        want = 'CPUExecutionProvider'
        provs = ['CPUExecutionProvider']
    out['requested'] = want
    s = ort.InferenceSession(model, so_, providers=provs)
    out['realised'] = s.get_providers()
    out['provider_present'] = want in out['realised']
    x = np.zeros((1, 3, int(ph), int(pw)), np.float32)
    # feeds are read off the SESSION, not assumed: the static graph has max_keypoints and
    # min_score folded into initializers and feeding them by name fails the whole probe with
    # `Invalid input name: min_score`, which would be recorded as "no GPU provider".
    want_in = {i.name for i in s.get_inputs()}
    out['model_inputs'] = sorted(want_in)
    feeds = {'image': x}
    if 'max_keypoints' in want_in:
        feeds['max_keypoints'] = np.array(int(maxkp), np.int64)
    if 'min_score' in want_in:
        feeds['min_score'] = np.array(float(minscore), np.float32)
    s.run(None, feeds)
    with open(s.end_profiling()) as f:
        ev = json.load(f)
    hist, conv = {}, {}
    for e in ev:
        if e.get('cat') != 'Node':
            continue
        a = e.get('args') or {}
        p = a.get('provider', 'unknown')
        hist[p] = hist.get(p, 0) + 1
        if a.get('op_name') == 'Conv':
            conv[p] = conv.get(p, 0) + 1
    out['hist'] = hist
    out['conv'] = conv
    # a TRT subgraph fuses into ONE node, so hist={'Tensorrt': 1} is ambiguous: it could be the
    # whole graph or one fused node beside 400 that fell back.  The share settles it.
    out['n_nodes_total'] = sum(hist.values())
    out['trt_node_share'] = (hist.get('TensorrtExecutionProvider', 0)
                             / max(sum(hist.values()), 1))
    out['fallback_nodes'] = {p: c for p, c in hist.items()
                             if p != 'TensorrtExecutionProvider'}
    out['n_cpu_nodes'] = hist.get('CPUExecutionProvider', 0)
    # reached the node histogram, so whatever `ok` says next is a VERDICT about the provider
    # and not the probe falling over on the way.  A False without this flag is a broken probe,
    # which is a different thing and must not be read as "this image has no GPU".
    out['verdict'] = True
    if ep == 'trt':
        # inside a TRT subgraph the individual Convs vanish into one fused node, so the test
        # is that TRT claimed work at all, not that it claimed the Convs by name
        out['ok'] = bool(out['provider_present'] and hist.get(want, 0) > 0)
    elif ep == 'cuda':
        # ~all of this graph's FLOPs are Conv; a Conv on the CPU means the GPU is idle
        out['ok'] = bool(out['provider_present'] and sum(conv.values()) > 0
                         and conv.get('CPUExecutionProvider', 0) == 0)
    else:
        out['ok'] = bool(sum(hist.values()) > 0)
except Exception as e:
    out['ok'] = False
    out['error'] = f'{type(e).__name__}: {str(e)[:800]}'
print('LOC_PROBE ' + json.dumps(out))
'''


def loc_run_probe(model, ep, device_id, major):
    """The probe's verdict, or a recorded failure.  Never raises.

    The subprocess output is kept EVEN WHEN the verdict parses: ORT reports a provider it
    could not construct by printing `*** EP Error ***` to stdout, or a `Failed to create
    TensorrtExecutionProvider` warning to stderr, and then carries on with whatever is left.
    Neither reaches the session object, so without the tails the reason is thrown away.
    """
    src = LOC_ROOT / 'loc_probe.py'
    src.parent.mkdir(parents=True, exist_ok=True)
    src.write_text(LOC_PROBE_SRC)
    cache = str(loc_trt_cache_dir(device_id, model)) if ep == 'trt' else ''
    cmd = (f'{sys.executable} "{src}" "{model}" "{ep}" "{device_id}" '
           f'"{LOC_ROOT}/probe_{ep}" "{major or 0}" "{LOC_H + 16}" "{LOC_W}" '
           f'"{LOC_MAX_KEYPOINTS}" "{LOC_MIN_SCORE}" "{cache}" '
           f'"{"True" if LOC_TRT_FP16 else "False"}"')
    try:
        r = loc_sh(cmd, check=False, env=loc_lib_env(),
                   timeout=(LOC_TRT_BUILD_MIN * 60 if ep == 'trt' else None))
    except subprocess.TimeoutExpired:
        return {'ok': False, 'ep': ep, 'model': str(model),
                'error': f'the probe did not return within {LOC_TRT_BUILD_MIN} min; a wedged '
                         f'engine build must not burn the session'}
    tails = {'rc': r.returncode, 'model': str(model),
             'stdout_tail': '\n'.join(x for x in (r.stdout or '').splitlines()
                                      if not x.startswith('LOC_PROBE '))[-1200:],
             'stderr_tail': (r.stderr or '')[-1200:]}
    for line in (r.stdout or '').splitlines():
        if line.startswith('LOC_PROBE '):
            d = json.loads(line[len('LOC_PROBE '):])
            d['proc'] = tails
            d['model'] = str(model)
            return d
    return {'ok': False, 'ep': ep, 'proc': tails, 'model': str(model),
            'error': f'probe produced no verdict (rc={r.returncode}): '
                     f'{((r.stderr or "") + (r.stdout or ""))[-500:]}'}


def loc_ort_recipes(major):
    """Ordered onnxruntime-gpu install recipes, most-likely-correct first.

    PyPI's `onnxruntime-gpu` tracks the newest CUDA and its provider library hard-fails on a
    CUDA 12 image, so the CUDA-12 build has to come from the index Microsoft publishes for
    exactly this split.  Each recipe is tried and then PROVEN with the probe; none is trusted
    on its name.
    """
    # --no-deps everywhere: pip resolving onnxruntime-gpu's dependency set is what replaced the
    # image's numpy and broke numpy.ma on a previous run.  Every runtime dependency ORT needs
    # (numpy, packaging, protobuf, flatbuffers, sympy, coloredlogs) is already on the image.
    nd = 'install -q --force-reinstall --no-deps'
    cu12ix = ('--extra-index-url https://aiinfra.pkgs.visualstudio.com/PublicPackages/'
              '_packaging/onnxruntime-cuda-12/pypi/simple/')
    r = [('ort-gpu-cuda12-index', [f'{nd} {cu12ix} onnxruntime-gpu']),
         ('ort-gpu-1.20.1-pypi-cuda12', [f'{nd} "onnxruntime-gpu==1.20.1"']),
         ('ort-gpu-1.19.2-pypi-cuda12', [f'{nd} "onnxruntime-gpu==1.19.2"']),
         ('ort-gpu-latest-plus-cuda13-wheels',
          [f'{nd} onnxruntime-gpu',
           'install -q nvidia-cuda-runtime-cu13 nvidia-cublas-cu13 nvidia-cudnn-cu13 '
           'nvidia-cufft-cu13 nvidia-curand-cu13'])]
    if major == 13:
        r = r[-1:] + r[:-1]
    return r


def loc_install_ort(model, major):
    """Install an onnxruntime-gpu whose CUDA major the image can actually satisfy.

    Returns (report, ok).  EVERY attempt is recorded, including the failures, because which
    recipe worked is the reusable part of this result.  numpy is fingerprinted before and
    after and pinned back BEFORE a recipe is judged: an ORT that works but leaves the
    interpreter broken is not a working recipe.
    """
    recipes = loc_ort_recipes(major)
    if LOC_ORT_RECIPES:
        want = LOC_ORT_RECIPES.split(',')
        recipes = [x for x in recipes if x[0] in want] or recipes
    rep = {'image_cuda_major': major, 'numpy_before': dict(LOC_NUMPY0), 'attempts': []}
    for name, cmds in recipes:
        att = {'recipe': name, 'cmds': cmds, 'pip': []}
        for c in cmds:
            r = loc_pip(c)
            att['pip'].append({'cmd': c, 'rc': r.returncode,
                               'tail': ((r.stderr or '') + (r.stdout or ''))[-300:]
                               if r.returncode else ''})
        att['numpy_restore'] = loc_restore_numpy()
        att['env'] = loc_check_env()
        att['numpy_after'] = {'version': att['env'].get('numpy'),
                              'file': att['env'].get('numpy_file')}
        if not att['env'].get('ok'):
            att['ok'] = False
            rep['attempts'].append(att)
            loc_log(f'  ort recipe {name}: REJECTED, it broke the interpreter: '
                    f'{att["env"].get("error")} (numpy {rep["numpy_before"]["version"]} -> '
                    f'{att["numpy_after"]["version"]})')
            continue
        att['probe'] = loc_run_probe(model, 'cuda', LOC_DEVICES[0] if LOC_DEVICES else 0, major)
        att['ok'] = bool(att['probe'].get('ok'))
        rep['attempts'].append(att)
        loc_log(f'  ort recipe {name}: ok={att["ok"]} ort={att["probe"].get("ort")} '
                f'realised={att["probe"].get("realised")} hist={att["probe"].get("hist")} '
                f'numpy {rep["numpy_before"]["version"]} -> {att["numpy_after"]["version"]} '
                f'({att["numpy_after"]["file"]}) {att["probe"].get("error", "")}')
        if att['ok']:
            rep['chosen'] = name
            rep['ort_version'] = att['probe'].get('ort')
            rep['env'] = att['env']
            rep['numpy_after'] = att['numpy_after']
            return rep, True
    return rep, False


def loc_trt_requirement():
    """Which libnvinfer soname ORT's TensorRT provider links against, and what resolves.

    Read out of the provider library with ldd instead of guessed from versions, with and
    without the candidate LD_LIBRARY_PATH, so "the library is on disk but the loader cannot
    see it" and "the library is not there at all" stop looking the same.
    """
    import glob
    out = {}
    try:
        import onnxruntime as ort
        d = Path(ort.__file__).parent / 'capi'
        cands = sorted(glob.glob(str(d / 'libonnxruntime_providers_tensorrt.so')))
        out['provider_so'] = cands
        if not cands:
            out['error'] = 'no libonnxruntime_providers_tensorrt.so in this onnxruntime'
            return out
        for tag, e in (('bare', None), ('ldpath', loc_lib_env())):
            txt = (loc_sh(f'ldd "{cands[0]}"', check=False, env=e).stdout or '')
            out[f'missing_{tag}'] = sorted({m for m in re.findall(r'(\S+) => not found', txt)})
            if tag == 'bare':
                out['ldd_bare'] = txt[-1500:]
        needs = sorted(set(re.findall(r'(libnvinfer[\w.]*\.so\.\d+)', out['ldd_bare'])))
        out['needs'] = needs
        maj = sorted({int(n.rsplit('.', 1)[1]) for n in needs}) if needs else []
        out['trt_major'] = maj[-1] if maj else None
    except Exception as e:                                      # noqa: BLE001
        out['error'] = f'{type(e).__name__}: {str(e)[:300]}'
    return out


def loc_trt_files_on_disk():
    """Every libnvinfer* / libnvonnxparser* actually present, plus the tensorrt dists.

    `pip install tensorrt-cu12` returns 0 whether or not it put the file ORT needs anywhere --
    the wheel is split across tensorrt-cu12 / -libs / -bindings and the meta `tensorrt`.  List
    what landed; do not infer it from the package name.
    """
    import glob
    files = {}
    for d in loc_lib_roots() + ['/usr/local/cuda/lib64', '/usr/lib/x86_64-linux-gnu']:
        for pat in ('libnvinfer*', 'libnvonnxparser*'):
            for p in glob.glob(os.path.join(d, pat)):
                files.setdefault(os.path.dirname(p), []).append(os.path.basename(p))
    r = loc_sh(f'{sys.executable} -m pip list 2>/dev/null | grep -i tensorrt', check=False)
    return {'files': {k: sorted(v) for k, v in files.items()},
            'pip': [x.strip() for x in (r.stdout or '').splitlines() if x.strip()]}


def loc_install_trt(model, major, prev=None):
    """Try to make a loadable TensorRT, targeting the major ORT actually asks for.

    Runs on the TRT-READY graph -- the same file the runners will load, so the engine the
    probe builds is the engine they get out of loc_trt_cache_dir().  Probing TensorRT with a
    different model is how a gate ends up complaining about Range outputs in a graph that has
    none, and the staged K=4096 graph cannot be built by TensorRT at all (its ITopK cap is
    3839), so this cannot be hoisted above loc_build_model to dodge the ordering problem --
    loc_maybe_reexec is what solves that.  TRT is not fatal here: a failure is recorded and
    the gate decides.
    """
    rec = {'enabled': LOC_TRT_INSTALL}
    if LOC_DRY or not LOC_PIP or not LOC_TRT_INSTALL:
        rec['skipped'] = 'dry run' if LOC_DRY else ('SRS_LOC_PIP=0' if not LOC_PIP
                                                    else 'SRS_LOC_TRT=0')
        rec['ok'] = False
        return rec
    reused = loc_reused(prev, 'tensorrt')
    if reused:
        return reused
    req = loc_trt_requirement()
    want = req.get('trt_major')
    rec.update(requirement=req, files_before=loc_trt_files_on_disk(), attempts=[])
    loc_log(f'  ORT TensorRT provider links against {req.get("needs")} (major {want}); '
            f'not-found bare={req.get("missing_bare")} '
            f'with-LD_LIBRARY_PATH={req.get("missing_ldpath")}')
    ladder = []
    if want:
        ladder.append((f'tensorrt-cu{major or 12}=={want}.*',
                       f'install -q "tensorrt-cu{major or 12}>={want},<{want + 1}"'))
        ladder.append((f'tensorrt=={want}.*-full',
                       f'install -q "tensorrt>={want},<{want + 1}" '
                       f'"tensorrt-cu{major or 12}-libs>={want},<{want + 1}" '
                       f'"tensorrt-cu{major or 12}-bindings>={want},<{want + 1}"'))
    ladder += [('tensorrt-cu12-unpinned', f'install -q "tensorrt-cu{major or 12}"'),
               ('tensorrt-nvidia-index',
                f'install -q --extra-index-url https://pypi.nvidia.com '
                f'"tensorrt-cu{major or 12}"'),
               ('tensorrt-meta', 'install -q tensorrt')]
    seen = set()
    for name, args in ladder:
        if args in seen:
            continue
        seen.add(args)
        r = loc_pip(args)
        att = {'recipe': name, 'cmd': args, 'rc': r.returncode,
               'tail': ((r.stderr or '') + (r.stdout or ''))[-300:] if r.returncode else ''}
        att['files_after'] = loc_trt_files_on_disk()
        att['ldd_after'] = loc_trt_requirement()
        att['probe'] = loc_run_probe(model, 'trt', LOC_DEVICES[0] if LOC_DEVICES else 0, major)
        att['ok'] = bool(att['probe'].get('ok'))
        rec['attempts'].append(att)
        p = att['probe']
        loc_log(f'  trt recipe {name}: rc={r.returncode} ok={att["ok"]} '
                f'realised={p.get("realised")} hist={p.get("hist")} '
                f'tensorrt={p.get("tensorrt")} dlopen={str(p.get("provider_dlopen"))[:120]} '
                f'missing_now={att["ldd_after"].get("missing_ldpath")}')
        if att['ok']:
            rec.update(chosen=name, ok=True)
            return rec
    rec['ok'] = False
    return rec


def loc_reused(prev, what, ok=None):
    """The record of an install that this VM has demonstrably already done, or None.

    Reuse is allowed ONLY on the second half of our own re-exec: same run, same VM, so the
    packages the first half installed are still there.  State alone proves nothing -- it is
    pulled from the remote and outlives the machine that wrote it, and a reclaimed VM has the
    state and none of the packages.
    """
    prev = prev or {}
    if os.environ.get('SRS_LOC_REEXECED') != '1':
        return None
    if not (prev.get('ok') if ok is None else ok):
        return None
    loc_log(f'{what}: reusing the install this run already made on this VM '
            f'({prev.get("chosen") or prev.get("recipe")}); a state file would not be enough '
            f'evidence, being on the far side of our own re-exec is')
    return dict(prev, reused_after_reexec=True)


def loc_reexec_verdict(roots, env, dry=None, pip=None):
    """(lib roots the loader cannot see, reason).  Empty roots -> do not re-exec.

    Separated from the exec, the way loc_gpu_verdict is separated from the exit, so the
    decision is testable without replacing the process.  Two INDEPENDENT stops, either of
    which alone prevents a loop: the SRS_LOC_REEXECED flag, and the fact that the process the
    exec creates has every root on its LD_LIBRARY_PATH already.
    """
    dry = LOC_DRY if dry is None else dry
    pip = LOC_PIP if pip is None else pip
    if dry or not pip:
        return [], ('dry run' if dry else 'SRS_LOC_PIP=0')
    if env.get('SRS_LOC_REEXEC') == '0':
        return [], 'SRS_LOC_REEXEC=0'
    if env.get('SRS_LOC_REEXECED') == '1':
        return [], "this IS the re-exec'd process"
    have = [p for p in env.get('LD_LIBRARY_PATH', '').split(':') if p]
    missing = [r for r in roots if r not in have]
    return missing, ('' if missing else 'every lib root is already on LD_LIBRARY_PATH')


def loc_maybe_reexec(st):
    """Re-exec this kernel ONCE, with an LD_LIBRARY_PATH that covers the installed libraries.

    The loader reads LD_LIBRARY_PATH at process start, so a library installed after this
    process began is invisible to it.  That is exactly what killed v2: TensorRT installed at
    7.1 min, its own SUBPROCESS probe realised TensorrtExecutionProvider with a clean node
    histogram, and then the parent -- same machine, same libraries, older process -- built a
    session that fell back to CUDA, and the gate refused it.  The only process in that run
    that got TensorRT was one started AFTER the libraries existed, with loc_lib_env(); this
    makes the process that creates the real sessions that same kind of process.

    Cheap: everything the second pass repeats is cached on /tmp or skipped outright, and no
    roll has been processed yet.  Cannot loop: SRS_LOC_REEXECED gates it, and independently
    the second process finds every lib root already on its LD_LIBRARY_PATH.  Resume is
    unaffected -- state.json is pushed before the exec, and the run id is a hash of this file,
    which the exec does not change.
    """
    roots = loc_lib_roots()
    missing, reason = loc_reexec_verdict(roots, os.environ)
    if not missing:
        return {'reexeced': False, 'reason': reason, 'roots': roots,
                'ld_library_path': os.environ.get('LD_LIBRARY_PATH', '')}
    try:
        me = os.path.abspath(__file__)
    except NameError:                          # lifted into a notebook: nothing to exec
        loc_log('re-exec skipped: no __file__, so the ctypes preload is the only loader-path '
                'fix in effect and a TensorRT session may fall back to CUDA')
        return {'reexeced': False, 'reason': 'no __file__'}
    st['reexec'] = {'missing_from_ld_library_path': missing, 'roots': roots,
                    'utc': time.strftime('%Y-%m-%dT%H:%M:%SZ', time.gmtime())}
    loc_save_state(st)
    loc_log(f'RE-EXEC once with LD_LIBRARY_PATH += {missing}. The libraries were installed '
            f'after this process started, so its loader cannot see them; state is banked and '
            f'the second pass skips every install it already made.', push_=True)
    sys.stdout.flush()
    sys.stderr.flush()
    os.execve(sys.executable, [sys.executable, me] + sys.argv[1:],
              loc_lib_env({'SRS_LOC_REEXECED': '1'}))


def loc_install(model, prev=None, cpu_only=False):
    """Put the third-party runtime this kernel imports on the image, and PROVE each piece.

    The Kaggle base image does not ship onnxruntime: the first push cleared the GPU inventory
    gate, authenticated both remotes, pulled every input and then died at 117.5 s on `import
    onnxruntime` inside loc_preoptimize.  A local dry run cannot catch that -- the laptop has
    onnxruntime -- so b4_dryrun.py audits the kernel's imports against LOC_INSTALLS instead.

    `model` is the STAGED static graph, not the preoptimised one: preoptimising needs the very
    onnxruntime being installed here, so the probe has to run on a file that exists first.

    `cpu_only` is the worker's install.  It reaches nothing that imports onnxruntime -- no graph
    build, no engine, no gate -- so the whole ORT ladder, which is minutes of a session and the
    single most failure-prone thing this kernel does, is skipped rather than run and ignored.
    """
    rec = {'python': sys.version.split()[0], 'installs': list(LOC_INSTALLS),
           'cpu_only': bool(cpu_only)}
    if LOC_DRY or not LOC_PIP:
        rec['skipped'] = 'dry run' if LOC_DRY else 'SRS_LOC_PIP=0'
        return rec
    if cpu_only:
        rec['ort'] = {'skipped': 'cpu worker role: nothing here imports onnxruntime'}
        rec['ort_ok'] = None
        rec['pkgs'] = loc_install_pkgs()
        rec['pkgs_ok'] = all(v.get('ok') for v in rec['pkgs'].values())
        loc_log('cpu worker packages: ' + ', '.join(
            f'{k}={v.get("version")} via {v.get("chosen")}' for k, v in rec['pkgs'].items()))
        if not rec['pkgs_ok']:
            loc_log(f'*** CPU WORKER MISSING PACKAGES: '
                    f'{[k for k, v in rec["pkgs"].items() if not v.get("ok")]} - it cannot '
                    f'bootstrap anything and the GPU will fall back on every roll ***',
                    push_=True)
        return rec
    reused = loc_reused(prev, 'onnxruntime', ok=(prev or {}).get('ort_ok'))
    if reused:
        return reused
    rec['cuda_env'] = loc_cuda_env()
    major = rec['cuda_major'] = rec['cuda_env']['image_cuda_major']
    loc_log(f'image CUDA: cublas majors {rec["cuda_env"]["cublas_majors"]} -> major {major}; '
            f'libnvinfer {list(rec["cuda_env"]["libs"]["libnvinfer"])}; '
            f'cudnn {list(rec["cuda_env"]["libs"]["libcudnn"])}; '
            f'numpy {LOC_NUMPY0["version"]} ({LOC_NUMPY0["file"]})')
    # a CPU onnxruntime on the image would shadow the gpu wheel's provider libraries
    loc_pip('uninstall -y -q onnxruntime')
    rec['ort'], rec['ort_ok'] = loc_install_ort(str(model), major)
    rec['recipe'] = rec['ort'].get('chosen')
    rec['ort_version'] = rec['ort'].get('ort_version')
    rec['ladder'] = [{'recipe': a['recipe'], 'ok': a['ok'],
                      'numpy': a.get('numpy_after', {}).get('version'),
                      'why': (a['env'].get('error') if not a['env'].get('ok')
                              else (a.get('probe') or {}).get('error'))}
                     for a in rec['ort']['attempts']]
    loc_log(f'onnxruntime install: chosen={rec["recipe"]} version={rec["ort_version"]} '
            f'ok={rec["ort_ok"]}; ladder {rec["ladder"]}')
    rec['pkgs'] = loc_install_pkgs()
    rec['pkgs_ok'] = all(v.get('ok') for v in rec['pkgs'].values())
    loc_log('packages: ' + ', '.join(
        f'{k}={v.get("version")} via {v.get("chosen")}' for k, v in rec['pkgs'].items()))
    # NB: the ctypes preload is deliberately NOT done here -- TensorRT is not installed yet,
    # and a preload that runs before an install is what v2 did.  loc_main does it once, after
    # the last ladder, next to loc_maybe_reexec.
    if not rec['ort_ok']:
        loc_log('*** NO onnxruntime RECIPE PRODUCED A GPU PROVIDER THAT EXECUTED THE GRAPH - '
                'the gate below is what decides; the whole ladder is in '
                'state.json["install"] ***')
    if not rec['pkgs_ok']:
        loc_log(f'*** MISSING PACKAGES after every recipe: '
                f'{[k for k, v in rec["pkgs"].items() if not v.get("ok")]} ***')
    return rec

## `loc-sample`

Decode a roll at 10 Hz between the plan's bounds, downscaled to 1280x720.

In [ ]:
# === cell: loc-sample ===
def loc_video_info(path):
    """fps / native size / duration, read off the container."""
    import cv2
    cap = cv2.VideoCapture(str(path))
    fps = cap.get(cv2.CAP_PROP_FPS)
    w = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
    h = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
    n = cap.get(cv2.CAP_PROP_FRAME_COUNT)
    cap.release()
    if not fps or fps <= 0:
        raise IOError(f'{path}: no usable fps ({fps})')
    return dict(fps=round(float(fps), 4), native_wh=[w, h], n_frames=int(n),
                video_ms=round(n * 1000.0 / fps))


def loc_targets(t0_ms, t1_ms, video_ms, hz=None):
    """The 10 Hz target grid inside [t0, t1), clamped to the footage that exists.

    Both clamps are physical, not conventional, and stay whatever plan.json says:
      * a roll_end past EOF is normal (42 rolls, worst 53.1 s over; 385 once +2 s is added) --
        clamp the GRID, never edit the event data;
      * a negative roll_start is real (roll 1387 is -21848: its camera started 21.8 s into the
        roll) and simply means the footage does not cover the head of the window.
    The bounds themselves are used verbatim; nothing here transforms a time base.
    """
    hz = LOC_HZ if hz is None else hz
    half = 1000.0 / hz / 2.0
    lo = max(0.0, float(t0_ms))
    tg = np.arange(lo + half, float(t1_ms), 1000.0 / hz)
    n_past = int((tg > video_ms).sum())
    rec = {'start_clamped_ms': (lo - float(t0_ms)) if lo != float(t0_ms) else 0.0,
           'end_clamped_ms': max(0.0, float(t1_ms) - float(video_ms)),
           'n_targets_past_eof': n_past, 'n_targets_full': int(len(tg))}
    return tg[tg <= video_ms], rec


def loc_intro_ref(path=None):
    """The 160x90 grayscale VIRB-Edit title card reference, or None if it was not staged."""
    if LOC_INTRO_REF[0] is None:
        p = Path(path or (LOC_IN / 'intro_ref.npy'))
        LOC_INTRO_REF[0] = np.load(p).astype(np.float32) if p.exists() else False
    return LOC_INTRO_REF[0] if LOC_INTRO_REF[0] is not False else None


def loc_is_intro(bgr, ref=None, thr=LOC_INTRO_MAD):
    """Is this frame a VIRB Edit title card?  Both polarities, because the card is rendered
    dark-on-light or light-on-dark depending on the template."""
    ref = loc_intro_ref() if ref is None else ref
    if ref is None:
        return False
    import cv2
    a = cv2.resize(cv2.cvtColor(bgr, cv2.COLOR_BGR2GRAY), (ref.shape[1], ref.shape[0]),
                   interpolation=cv2.INTER_AREA).astype(np.float32)
    return float(min(np.abs(a - ref).mean(), np.abs(a - (255.0 - ref)).mean())) <= thr


def loc_sharpness(bgr):
    import cv2
    g = cv2.resize(cv2.cvtColor(bgr, cv2.COLOR_BGR2GRAY), (426, 240))
    return float(cv2.Laplacian(g, cv2.CV_32F).var())


def loc_mask(path, wh=(LOC_W, LOC_H)):
    """Boolean keep-mask at the map's image size.

    a1_extract.py resolves masks only through tmp/era_probe/season/manifest.json and hard-exits
    otherwise, which covers ~34 rolls; here the mask is inputs/masks/<file_id>.png staged from
    data/masks and is at the video's NATIVE size, so it is downscaled with the frame.  No shape
    assert: the native size varies (1920x1080, 1706x960, 1280x720).
    """
    import cv2
    m = cv2.imread(str(path), cv2.IMREAD_GRAYSCALE)
    if m is None:
        raise IOError(f'cannot read mask {path}')
    if (m.shape[1], m.shape[0]) != tuple(wh):
        m = cv2.resize(m, tuple(wh), interpolation=cv2.INTER_AREA)
    return m >= 128


def loc_sample(path, targets, out=None, wh=(LOC_W, LOC_H), max_frames=0):
    """Yield (frame_idx, t_ms, dec_idx, rgb) for the sharpest frame in each target cell.

    a1_extract.py's seek-once-then-sequential pattern (measured 15-19 ms/kept frame): one
    CAP_PROP_POS_MSEC seek before the first target, then a straight read to the end.  At 10 Hz
    the +/-50 ms cells tile the window with no gaps, so every decoded frame lands in a cell and
    `grab()` never skips anything; at a lower rate it skips the retrieve, which is what makes a
    reduced-rate dry run cheap without changing the code path.
    """
    import cv2
    half = 1000.0 / LOC_HZ / 2.0 if len(targets) < 2 else float(targets[1] - targets[0]) / 2.0
    cap = cv2.VideoCapture(str(path))
    fps = cap.get(cv2.CAP_PROP_FPS)
    resize = None
    cap.set(cv2.CAP_PROP_POS_MSEC, max(0.0, float(targets[0]) - 1500))
    ti, kept, best = 0, 0, None
    n_dec, n_intro = 0, 0
    ref = loc_intro_ref()
    while ti < len(targets):
        idx = int(round(cap.get(cv2.CAP_PROP_POS_FRAMES)))
        ts = idx * 1000.0 / fps
        tgt = float(targets[ti])
        if ts < tgt - half:
            if not cap.grab():
                break
            continue
        if ts <= tgt + half:
            if not cap.grab():
                break
            ok, fr = cap.retrieve()
            if not ok:
                break
            n_dec += 1
            s = loc_sharpness(fr)
            if best is None or s > best[0]:
                best = (s, ts, idx, fr)
            continue
        if best is not None:
            s, bts, bidx, bfr = best
            best = None
            if ref is not None and loc_is_intro(bfr, ref):
                n_intro += 1
                ti += 1
                continue
            if resize is None:
                resize = (bfr.shape[1], bfr.shape[0]) != tuple(wh)
            if resize:
                bfr = cv2.resize(bfr, tuple(wh), interpolation=cv2.INTER_AREA)
            yield kept, int(round(bts)), bidx, np.ascontiguousarray(bfr[:, :, ::-1])
            kept += 1
            if max_frames and kept >= max_frames:
                break
        ti += 1
    cap.release()
    if out is not None:
        out.update(n_kept=kept, n_decoded=n_dec, n_targets=len(targets),
                   n_intro_dropped=n_intro, resized=bool(resize))

## `loc-aliked`

ALIKED-n16rot via onnxruntime, reproducing colmap's `aliked.cc` exactly.

In [ ]:
# === cell: loc-aliked ===
def loc_to_input_tensor(rgb, divisor=LOC_DIVISOR):
    """RGB uint8 HWC -> ([1,3,ph,pw] float32, (h,w), (ph,pw)).

    COLMAP's InputPadder::MaybePad REPLICATES the last row/column to a multiple of 32, which
    np.pad(mode='edge') reproduces exactly.  Zero-padding instead costs descriptor dot 0.9998
    rather than 1.0000 and corrupts the descriptors this map was built with.
    """
    h, w = rgb.shape[:2]
    ph = ((h + divisor - 1) // divisor) * divisor
    pw = ((w + divisor - 1) // divisor) * divisor
    chw = rgb.astype(np.float32).transpose(2, 0, 1) / np.float32(255.0)
    if (ph, pw) != (h, w):
        chw = np.pad(chw, ((0, 0), (0, ph - h), (0, pw - w)), mode='edge')
    return np.ascontiguousarray(chw[None]), (h, w), (ph, pw)


def loc_postprocess(kp, ds, sc, hw, phw, min_score=LOC_MIN_SCORE):
    """aliked.cc:249-285, including the filter ORDER, so survivors keep COLMAP's order."""
    h, w = hw
    ph, pw = phw
    K = np.asarray(kp[0] if kp.ndim == 3 else kp, np.float32)
    D = np.asarray(ds[0] if ds.ndim == 3 else ds, np.float32)
    S = np.asarray(sc[0] if sc.ndim == 2 else sc, np.float32)
    px = (K[:, 0] + np.float32(1.0)) * (np.float32(0.5) * np.float32(pw - 1)) + np.float32(0.5)
    py = (K[:, 1] + np.float32(1.0)) * (np.float32(0.5) * np.float32(ph - 1)) + np.float32(0.5)
    keep = (S >= np.float32(min_score)) & (px >= 0) & (px < w) & (py >= 0) & (py < h)
    return np.stack([px[keep], py[keep]], 1).astype(np.float32), D[keep], S[keep]


def loc_topk_k(model):
    """The K the graph's TopK actually asks for, read off the initializer."""
    import onnx
    from onnx import numpy_helper
    g = onnx.load(str(model)).graph
    vals = {i.name: numpy_helper.to_array(i) for i in g.initializer}
    vals.update({n.output[0]: numpy_helper.to_array(
        [a for a in n.attribute if a.name == 'value'][0].t)
        for n in g.node if n.op_type == 'Constant'
        and any(a.name == 'value' for a in n.attribute)})
    return {n.name: (None if vals.get(n.input[1]) is None
                     else int(np.ravel(vals[n.input[1]])[0]))
            for n in g.node if n.op_type == 'TopK'}


def loc_set_topk_k(src, k, dst):
    """Rewrite the graph's TopK K.  `max_keypoints` does NOT control it: the export hard-codes
    TopK's K at 4096 and max_keypoints only drives a downstream slice, so every model with a
    folded max_keypoints still left TensorRT looking at 4096 and rejecting the build."""
    import onnx
    from onnx import numpy_helper
    dst = Path(dst)
    if dst.exists():
        return dst, {'path': str(dst), 'k': int(k), 'cached': True, 'after': loc_topk_k(dst)}
    m = onnx.load(str(src))
    g = m.graph
    named = {i.name for i in g.initializer}
    changed = 0
    for n in g.node:
        if n.op_type != 'TopK':
            continue
        kin = n.input[1]
        arr = np.array([int(k)], np.int64)
        if kin in named:
            for i in g.initializer:
                if i.name == kin:
                    i.CopyFrom(numpy_helper.from_array(arr, kin))
                    changed += 1
        else:
            new = f'{kin}_loc{k}'
            g.initializer.append(numpy_helper.from_array(arr, new))
            n.input[1] = new
            changed += 1
    onnx.save(m, str(dst))
    return dst, {'path': str(dst), 'k': int(k), 'nodes_changed': changed,
                 'after': loc_topk_k(dst)}


def loc_preoptimize(src, dst):
    """ORT's BASIC-level optimiser, whose constant folding is what makes the graph TRT-parseable.

    BASIC only: `extended` injects 5 com.microsoft ops and `all` injects 70 com.microsoft.nchwc
    ops, neither of which TensorRT can parse.  Dangling value_info inherited from the source is
    pruned or TensorRT complains about symbolic Range outputs that no longer exist.
    """
    import onnx
    import onnxruntime as ort
    dst = Path(dst)
    if not dst.exists():
        tmp = dst.with_suffix('.raw.onnx')
        so = ort.SessionOptions()
        so.intra_op_num_threads = 1
        so.graph_optimization_level = ort.GraphOptimizationLevel.ORT_ENABLE_BASIC
        so.optimized_model_filepath = str(tmp)
        ort.InferenceSession(str(src), so, providers=['CPUExecutionProvider'])
        m = onnx.load(str(tmp))
        g = m.graph
        live = ({o for n in g.node for o in n.output} | {i.name for i in g.input}
                | {i.name for i in g.initializer})
        keep = [v for v in g.value_info if v.name in live]
        del g.value_info[:]
        g.value_info.extend(keep)
        onnx.save(m, str(dst))
        tmp.unlink(missing_ok=True)
    g = onnx.load(str(dst)).graph
    import collections
    h = collections.Counter(n.op_type for n in g.node)
    rec = {'path': str(dst), 'n_nodes': len(g.node),
           'residual_dynamic_ops': {k: h[k] for k in ('Range', 'Shape', 'If', 'NonZero') if h[k]},
           'domains': sorted({n.domain for n in g.node}),
           'n_symbolic_value_info': sum(
               1 for v in g.value_info
               if any(not d.HasField('dim_value') for d in v.type.tensor_type.shape.dim))}
    rec['trt_ready'] = bool(not rec['residual_dynamic_ops'] and rec['n_symbolic_value_info'] == 0
                            and rec['domains'] in ([''], []))
    return dst, rec


def loc_build_model():
    """The one graph every arm runs: static 1x3x736x1280, folded scalars, TopK K = 3839,
    BASIC-preoptimised.  Staged variants are reused; anything missing is rebuilt."""
    LOC_MODELS.mkdir(parents=True, exist_ok=True)
    rec = {}
    base = LOC_IN / 'aliked-n16rot.onnx'
    static = LOC_IN / 'aliked-n16rot-static-const.onnx'
    if not static.exists():
        raise RuntimeError(f'no static ALIKED graph at {static}; b2_stage_inputs.py stages it')
    rec['base'] = str(base)
    kmod, rec['topk'] = loc_set_topk_k(
        static, LOC_TOPK_K, LOC_MODELS / f'aliked-n16rot-static-const-k{LOC_TOPK_K}.onnx')
    pre, rec['pre'] = loc_preoptimize(
        kmod, LOC_MODELS / f'aliked-n16rot-static-const-k{LOC_TOPK_K}-pre.onnx')
    got = set(rec['topk']['after'].values())
    assert got == {LOC_TOPK_K}, f'TopK K read back as {got}, wanted {LOC_TOPK_K}'
    return pre, rec


# Every TensorRT EP option whose parser wants the literal 'True'/'False'.  '0'/'1' is REJECTED
# and the whole session with it -- ORT then falls back to CUDA and every symptom looks like a
# missing libnvinfer.
LOC_TRT_BOOL_OPTS = frozenset({
    'trt_fp16_enable', 'trt_bf16_enable', 'trt_int8_enable', 'trt_dla_enable',
    'trt_engine_cache_enable', 'trt_timing_cache_enable', 'trt_force_timing_cache',
    'trt_detailed_build_log', 'trt_build_heuristics_enable', 'trt_sparsity_enable',
    'trt_cuda_graph_enable', 'trt_dump_subgraphs', 'trt_force_sequential_engine_build',
    'trt_context_memory_sharing_enable', 'trt_layer_norm_fp32_fallback',
    'trt_weight_stripped_engine_enable', 'trt_dump_ep_context_model',
    'trt_engine_hw_compatible', 'trt_int8_use_native_calibration_table',
})


def loc_trt_options(**kw):
    out = {}
    for k, v in kw.items():
        if k in LOC_TRT_BOOL_OPTS:
            if isinstance(v, str):
                v = v.strip().lower() in ('1', 'true', 'yes')
            out[k] = 'True' if v else 'False'
        else:
            out[k] = str(v)
    bad = {k: v for k, v in out.items() if k in LOC_TRT_BOOL_OPTS and v not in ('True', 'False')}
    if bad:
        raise RuntimeError(f'TensorRT boolean options must be True/False: {bad}')
    return out


def loc_providers_for(ep, device_id, model=''):
    import onnxruntime as ort
    avail = ort.get_available_providers()
    cuda = ('CUDAExecutionProvider', {'device_id': str(device_id)})
    if ep == 'cpu':
        return ['CPUExecutionProvider']
    if ep == 'cuda':
        if 'CUDAExecutionProvider' not in avail:
            raise RuntimeError(f'CUDAExecutionProvider unavailable; have {avail}')
        return [cuda, 'CPUExecutionProvider']
    if 'TensorrtExecutionProvider' not in avail:
        raise RuntimeError(f'TensorrtExecutionProvider unavailable; have {avail}')
    cache = loc_trt_cache_dir(device_id, model)      # shared with the gate/install probes
    trt = ('TensorrtExecutionProvider', loc_trt_options(
        device_id=device_id, trt_fp16_enable=LOC_TRT_FP16, trt_engine_cache_enable=True,
        trt_engine_cache_path=str(cache), trt_timing_cache_enable=True,
        trt_timing_cache_path=str(cache), trt_max_workspace_size=4 << 30,
        trt_builder_optimization_level=3, trt_detailed_build_log=True))
    return [trt, cuda, 'CPUExecutionProvider']


def loc_make_session(model, ep, device_id):
    """Create the session and REFUSE it if ORT quietly gave us a different provider.

    ORT's python wrapper catches a provider-construction failure, prints `*** EP Error ***` and
    rebuilds on whatever is left -- usually the CPU.  Timing a T4 session that is really running
    on the CPU is the failure this run cannot afford.
    """
    import onnxruntime as ort
    so = ort.SessionOptions()
    so.graph_optimization_level = ort.GraphOptimizationLevel.ORT_ENABLE_ALL
    sess = ort.InferenceSession(str(model), so,
                                providers=loc_providers_for(ep, device_id, str(model)))
    want = {'trt': 'TensorrtExecutionProvider', 'cuda': 'CUDAExecutionProvider'}.get(ep)
    if want and want not in sess.get_providers():
        raise RuntimeError(f'onnxruntime fell back: requested {want}, realised '
                           f'{sess.get_providers()}. Refusing the session.')
    return sess


class LocRunner:
    """One ONNX session on one device, plus the scalar inputs the static graph folded away."""

    def __init__(self, model, ep, device_id=0):
        self.sess = loc_make_session(model, ep, device_id)
        self.device_id = device_id
        self.eps = self.sess.get_providers()
        self.scalars = {n.name for n in self.sess.get_inputs()} - {'image'}
        self.mk = np.array(LOC_MAX_KEYPOINTS, np.int64)
        self.ms = np.array(LOC_MIN_SCORE, np.float32)

    def run(self, x):
        f = {'image': x}
        if 'max_keypoints' in self.scalars:
            f['max_keypoints'] = self.mk
        if 'min_score' in self.scalars:
            f['min_score'] = self.ms
        return self.sess.run(None, f)


def loc_extract_frame(runner, rgb, mask):
    """One frame -> (kp (n,2) float32, desc (n,128) float32, score (n,) float32).

    Masking is a POST-filter: the extractor never sees the mask, so keypoints under the buggy
    nose and the VIRB overlay are still extracted and still consume the budget, and the budget
    is NOT redistributed to the visible region.  They are dropped here, after the fact, exactly
    as COLMAP's feature_extraction.cc MaskFeatures does.
    """
    x, hw, phw = loc_to_input_tensor(rgb)
    kp, ds, sc = runner.run(x)[:3]
    K, D, S = loc_postprocess(kp, ds, sc, hw, phw)
    if mask is not None and len(K):
        yy = np.clip(K[:, 1].astype(np.int32), 0, mask.shape[0] - 1)
        xx = np.clip(K[:, 0].astype(np.int32), 0, mask.shape[1] - 1)
        keep = mask[yy, xx]
        K, D, S = K[keep], D[keep], S[keep]
    return K, D, S


def loc_extract_roll(runners, path, targets, mask, out=None, max_frames=0, hb=None):
    """Decode on one thread, extract on one thread per device.

    Decode is 18.6 ms/kept frame against 30.4 ms of 2xT4 inference, so a single decoder keeps
    both GPUs fed; serialising them instead caps the whole pipeline at ~54 f/s regardless of
    GPU speed.  Only (keypoints, descriptors, scores) survive -- a roll's frames at 1280x720
    would be 4.7 GB of RAM, its fp16 descriptors are ~1 GB.
    """
    q = queue.Queue(maxsize=LOC_QUEUE)
    res = {}
    err = []
    info = {}
    # per DEVICE, because a 2xT4 rate that is really one T4 doing most of the work looks
    # exactly like a slow pair from the outside.  loc_make_session refuses a silent CUDA
    # fallback, so a provider difference would already have raised; a load IMBALANCE would
    # not, and only counting frames per device can tell them apart.
    per_dev = {r.device_id: {'n': 0, 'busy_s': 0.0, 'eps': list(r.eps)} for r in runners}

    def produce():
        try:
            for item in loc_sample(path, targets, info, max_frames=max_frames):
                q.put(item)
        except Exception as e:                                  # noqa: BLE001
            err.append(f'decode: {type(e).__name__}: {e}')
        finally:
            for _ in runners:
                q.put(None)

    def consume(r):
        d = per_dev[r.device_id]
        try:
            while True:
                item = q.get()
                if item is None:
                    return
                i, t_ms, dec, rgb = item
                t0 = time.perf_counter()
                K, D, S = loc_extract_frame(r, rgb, mask)
                d['busy_s'] += time.perf_counter() - t0
                d['n'] += 1
                res[i] = (t_ms, dec, K, D.astype(np.float16), S,
                          D if (i % LOC_BOOT_STRIDE == 0) else None)
                if hb is not None:
                    hb(len(res))
        except Exception as e:                                  # noqa: BLE001
            err.append(f'extract[{r.device_id}]: {type(e).__name__}: {e}')

    th = [threading.Thread(target=produce, daemon=True)]
    th += [threading.Thread(target=consume, args=(r,), daemon=True) for r in runners]
    for t in th:
        t.start()
    for t in th:
        t.join()
    if err:
        raise RuntimeError('; '.join(err))
    idx = sorted(res)
    assert idx == list(range(len(idx))), 'frame indices are not contiguous'
    for d in per_dev.values():
        d['busy_s'] = round(d['busy_s'], 2)
        d['ms_per_frame'] = round(1000.0 * d['busy_s'] / max(d['n'], 1), 2)
    info['per_device'] = per_dev
    tot = sum(d['n'] for d in per_dev.values()) or 1
    info['device_share'] = {str(k): round(d['n'] / tot, 3) for k, d in per_dev.items()}
    if out is not None:
        out.update(info)
    return dict(t_ms=np.array([res[i][0] for i in idx], np.int64),
                dec_idx=np.array([res[i][1] for i in idx], np.int32),
                kp=[res[i][2] for i in idx], desc=[res[i][3] for i in idx],
                score=[res[i][4] for i in idx], desc32={i: res[i][5] for i in idx
                                                        if res[i][5] is not None})

## `loc-map`

Load the map cache: faiss index, fp16 observation bank, row permutation.

In [ ]:
# === cell: loc-map ===
def loc_map_root(root=None):
    """The attached buggy-rtk-base-cache-8218 dataset, found by CONTENT not by name."""
    root = Path(root or LOC_INPUT)
    for p in sorted(root.rglob('p4/meta.json')):
        if (p.parent.parent / 'p7/obs_desc_arc.f16').exists():
            return p.parent.parent
    raise RuntimeError(f'no map cache bundle under {root} (need p4/meta.json + '
                       f'p7/obs_desc_arc.f16); attach yu5uf5/buggy-rtk-base-cache-8218')


def loc_perm_p6a_to_p7(mroot, cache=None):
    """p6a row (image-major, what the faiss index returns) -> p7 row (arc/point-major).

    The faiss bootstrap index is keyed to the image-major observation order of cache/p6a, and
    the only descriptor bank shipped is cache/p7's arc-sorted fp16 one.  This is the map between
    them, rebuilt from p4's two observation indexes rather than shipped, and self-checked
    against p7/row_img before it is used.
    """
    cache = Path(cache or (LOC_ROOT / 'p6a2p7.npy'))
    if cache.exists():
        return np.load(cache, mmap_mode='r')
    m = lambda n: np.load(f'{mroot}/p4/{n}.npy', mmap_mode='r')                 # noqa: E731
    n_obs = int(json.loads(Path(f'{mroot}/p4/meta.json').read_text())['n_obs'])
    img_ptr = np.asarray(m('img_ptr')).astype(np.int64)
    pt_ptr = np.asarray(m('pt_ptr')).astype(np.int64)
    pt_start = np.load(f'{mroot}/p7/pt_start.npy').astype(np.int64)
    pt_order = np.load(f'{mroot}/p7/pt_order.npy').astype(np.int64)
    assert int(pt_start[-1]) == n_obs and int(img_ptr[-1]) == n_obs

    # keypoint indices are < 4096, so (image << 20 | keypoint) is globally nondecreasing over
    # the image-major array and one searchsorted resolves every point-major row at once
    key_img = ((np.repeat(np.arange(len(img_ptr) - 1, dtype=np.int64), np.diff(img_ptr)) << 20)
               | np.asarray(m('img_obs_kp')).astype(np.int64))
    q = ((np.asarray(m('pt_obs_img')).astype(np.int64) << 20)
         | np.asarray(m('pt_obs_kp')).astype(np.int64))
    pm2p6a = np.searchsorted(key_img, q)
    assert (key_img[pm2p6a] == q).all(), 'observation not found in its image (index mismatch)'
    del key_img, q

    k = np.repeat(np.arange(len(pt_start) - 1, dtype=np.int64), np.diff(pt_start))
    p7_to_pm = pt_ptr[pt_order[k]] + (np.arange(n_obs, dtype=np.int64) - pt_start[k])
    del k
    p7_to_p6a = pm2p6a[p7_to_pm]
    row_img7 = np.load(f'{mroot}/p7/row_img.npy', mmap_mode='r')
    s = np.linspace(0, n_obs - 1, 50000).astype(np.int64)
    assert (np.asarray(m('pt_obs_img'))[p7_to_pm[s]] == np.asarray(row_img7[s])).all(), \
        'p7 permutation self-check FAILED: rebuilt row_img disagrees with the shipped one'
    out = np.empty(n_obs, np.int32)
    out[p7_to_p6a] = np.arange(n_obs, dtype=np.int32)
    np.save(cache, out)
    return np.load(cache, mmap_mode='r')


class LocBank:
    """The resident fp16 observation bank: 12,400,933 x 128 x 2 = 3.17 GB, whole-map.

    Loaded once per session and kept on the GPU; every frame slices its own +/-30 m arc range
    out of it.  There is no window swapping and no per-roll window setup.
    """

    def __init__(self, path, n_obs, device, dim=128):
        """`device` may be one device or a list; a copy lands on each one that can hold it.

        Replicated rather than sharded because both consumers are a GEMM of one query block
        against the WHOLE bank -- match against a 30 m slice of it, bootstrap against all of
        it -- so a shard would make every query a cross-device reduction, while a replica makes
        each device independently able to answer any query.  3.17 GB of 14.6 is worth that.
        """
        self.n, self.dim = n_obs, dim
        self.np = np.memmap(path, np.float16, mode='r', shape=(n_obs, dim))
        want = list(device) if isinstance(device, (list, tuple)) else [device]
        self.ts, self.devices, self.vram = {}, [], {}
        self.device, self.t = want[0], None
        if want[0] == 'cpu':
            self.devices = ['cpu']
            return
        import torch
        need = n_obs * dim * 2 + LOC_KNN_TILE_B + LOC_VRAM_HEADROOM_B
        for d in want:
            try:
                free, total = torch.cuda.mem_get_info(torch.device(d))
            except Exception as e:                              # noqa: BLE001
                free, total = -1, -1
                self.vram[d] = f'unreadable: {type(e).__name__}'
            self.vram[d] = {'free_gib': round(free / 2 ** 30, 2),
                            'total_gib': round(total / 2 ** 30, 2),
                            'need_gib': round(need / 2 ** 30, 2)}
            # the TRT engines and their 4 GiB workspace are built AFTER this, so the headroom
            # has to cover work that has not allocated yet -- hence a guard, not a subtraction
            if free < need:
                self.vram[d]['replicated'] = False
                loc_log(f'bank NOT replicated to {d}: {free / 2 ** 30:.1f} GiB free < '
                        f'{need / 2 ** 30:.1f} GiB needed (bank '
                        f'{n_obs * dim * 2 / 2 ** 30:.2f} + tile + headroom). That device will '
                        f'not serve match or bootstrap; this costs speed and nothing else.')
                continue
            t = torch.empty((n_obs, dim), dtype=torch.float16, device=d)
            step = 1 << 21
            for a in range(0, n_obs, step):
                b = min(n_obs, a + step)
                t[a:b] = torch.from_numpy(np.asarray(self.np[a:b])).to(d)
            self.ts[d] = t
            self.devices.append(d)
            self.vram[d]['replicated'] = True
        assert self.devices, f'the bank fits on no requested device: {self.vram}'
        self.device = self.devices[0]
        self.t = self.ts[self.device]

    def rows(self, idx, device=None):
        import torch
        d = device or self.device
        t = self.ts.get(d)
        if t is not None:
            return t[torch.as_tensor(np.asarray(idx), device=d, dtype=torch.long)]
        return torch.from_numpy(np.asarray(self.np[np.asarray(idx)]))


class LocMap:
    """Everything the pipeline reads out of the map bundle, loaded once per session."""

    def __init__(self, mroot=None, bank_device=None):
        self.root = Path(mroot or loc_map_root())
        meta = json.loads((self.root / 'p4/meta.json').read_text())
        self.n_obs = int(meta['n_obs'])                # never hard-coded: 12,400,933 today
        self.n_img = int(meta['n_images'])
        self.names = meta['names']
        self.runs = meta['runs']
        self.cameras = meta['cameras']
        p4 = lambda n: np.load(self.root / f'p4/{n}.npy', mmap_mode='r')        # noqa: E731
        self.img_ptr = np.asarray(p4('img_ptr')).astype(np.int64)
        self.img_obs_pt = p4('img_obs_pt')
        self.img_arc = np.asarray(p4('img_arc'))
        self.img_run = np.asarray(p4('img_run'))
        self.img_centre = np.asarray(p4('img_centre'))
        self.pt_xyz = np.asarray(p4('pt_xyz'))
        self.pt_id = np.asarray(p4('pt_id'))
        self.cl_P = np.asarray(p4('cl_P'))
        self.cl_S = np.asarray(p4('cl_S'))
        self.pt_order = np.load(self.root / 'p7/pt_order.npy').astype(np.int64)
        self.pt_arc_sorted = np.load(self.root / 'p7/pt_arc_sorted.npy')
        self.pt_start = np.load(self.root / 'p7/pt_start.npy').astype(np.int64)
        self.row_img7 = np.load(self.root / 'p7/row_img.npy', mmap_mode='r')
        self.inv_pt_order = np.empty(len(self.pt_order), np.int64)
        self.inv_pt_order[self.pt_order] = np.arange(len(self.pt_order))
        self.row_pt6 = np.load(self.root / 'p6a/row_pt.npy', mmap_mode='r')
        self.p6a2p7 = loc_perm_p6a_to_p7(self.root)
        self.bank = LocBank(self.root / 'p7/obs_desc_arc.f16', self.n_obs,
                            bank_device or LOC_BANK_DEVICE_LIST)
        self.index = None
        self.name_row = {n: i for i, n in enumerate(self.names)}

    def load_index(self, nprobe=LOC_NPROBE, threads=None):
        """The bootstrap's IVFPQ index.

        The MAIN thread's OpenMP width is deliberately LEFT ALONE unless it is forced.  faiss
        shares its OpenMP runtime with torch's CPU kernels, so setting it here does not stay
        inside faiss: pinning the main thread to 1 for the bootstrap's benefit made the CPU
        match GEMM 1.4x SLOWER in the dry run (16.4 s -> 25.6 s on roll 916), which is a stage
        that has nothing to do with faiss.  loc_run_bootstrap pins its own pool workers, which
        is where the pinning belongs and the only place it has any effect.
        """
        import faiss
        if threads or LOC_FAISS_THREADS:
            faiss.omp_set_num_threads(int(threads or LOC_FAISS_THREADS))
        self.threads = int(faiss.omp_get_max_threads())
        ix = faiss.read_index(str(self.root / 'p6a/idx_ivfpq_m32_n16384.faiss'))
        try:
            faiss.extract_index_ivf(ix).nprobe = int(nprobe)
        except Exception:
            pass
        assert ix.ntotal == self.n_obs, (ix.ntotal, self.n_obs)
        self.index = ix
        return ix


def loc_arc_lat(cl_P, cl_S, xy):
    """(arc metres, signed lateral offset metres) of ENU xy on the course centreline."""
    xy = np.atleast_2d(np.asarray(xy, np.float64))
    A, B = cl_P[:-1], cl_P[1:]
    AB = B - A
    L2 = np.where((AB * AB).sum(1) <= 0, 1e-12, (AB * AB).sum(1))
    d = xy[:, None, :] - A[None]
    t = np.clip((d * AB[None]).sum(2) / L2[None], 0.0, 1.0)
    proj = A[None] + t[..., None] * AB[None]
    j = ((xy[:, None, :] - proj) ** 2).sum(2).argmin(1)
    r = np.arange(len(xy))
    arc = cl_S[j] + t[r, j] * (cl_S[j + 1] - cl_S[j])
    v = xy - proj[r, j]
    tan = AB[j] / np.sqrt(L2[j])[:, None]
    lat = v[:, 0] * tan[:, 1] - v[:, 1] * tan[:, 0]
    return arc, lat

## `loc-pnp`

PnP configuration, and the two-pass per-roll fit of focal and k1.

In [ ]:
# === cell: loc-pnp ===
def loc_pnp_options(seed=0, max_error_px=LOC_PNP_MAX_ERROR_PX,
                    refine_focal=False, refine_extra=False):
    """One PnP configuration for the whole pipeline (p5_lib.pnp_options).

    refine_focal/refine_extra are the intrinsics fit's pass 1 only; both default False in
    pycolmap and both are left False for every production solve.
    """
    import pycolmap
    est = pycolmap.AbsolutePoseEstimationOptions()
    est.ransac.max_error = float(max_error_px)
    est.ransac.min_inlier_ratio = 0.01
    est.ransac.confidence = 0.9999
    est.ransac.min_num_trials = 100
    est.ransac.max_num_trials = 10000
    est.ransac.random_seed = int(seed)
    est.ransac.num_threads = 1
    est.estimate_focal_length = False
    ref = pycolmap.AbsolutePoseRefinementOptions()
    ref.refine_focal_length = bool(refine_focal)
    ref.refine_extra_params = bool(refine_extra)
    ref.print_summary = False
    return est, ref


def loc_camera(focal, k1):
    """A FRESH SIMPLE_RADIAL camera.  estimate_and_refine_absolute_pose writes refined
    intrinsics into the passed Camera IN PLACE, so reusing one across frames silently
    accumulates refinement -- every refining call must be given its own."""
    import pycolmap
    return pycolmap.Camera(model='SIMPLE_RADIAL', width=LOC_W, height=LOC_H,
                           params=[float(focal), LOC_PP[0], LOC_PP[1], float(k1)])


def loc_pnp(p2d, p3d, cam, est, ref, M=None, arcs=None, rows=None, cov=False):
    """PnP + everything PLAN 3.6 wants out of one solve."""
    import pycolmap
    out = dict(ok=0, n_corr=int(len(p2d)), n_inl=0, centre=None, quat=None, arc=np.nan,
               lat=np.nan, inl_arc_span=np.nan, inl_arc_med=np.nan, rep_p50=np.nan,
               rep_p90=np.nan, cov=None, inlier_mask=None, n_sup_img=0, run_mask=0)
    if len(p2d) < LOC_MIN_CORRS:
        return out
    _t = time.perf_counter()
    res = pycolmap.estimate_and_refine_absolute_pose(
        np.ascontiguousarray(p2d, np.float64), np.ascontiguousarray(p3d, np.float64),
        cam, est, ref, cov)
    _t = loc_prof('solve', _t)
    if res is None:
        return out
    cfw = res['cam_from_world']
    centre = np.asarray(cfw.inverse().translation, np.float64)
    m = np.asarray(res['inlier_mask']).astype(bool).ravel()
    out.update(ok=1, n_inl=int(res['num_inliers']), centre=centre,
               quat=np.asarray(cfw.rotation.quat, np.float64), inlier_mask=m)
    if cov and res.get('covariance') is not None:
        out['cov'] = np.asarray(res['covariance'], np.float64)
    if M is not None:
        a, lat = loc_arc_lat(M.cl_P, M.cl_S, centre[:2])
        out['arc'], out['lat'] = float(a[0]), float(lat[0])
    if m.any():
        proj = cam.img_from_cam(cfw * np.ascontiguousarray(p3d, np.float64)[m])
        r = np.linalg.norm(proj - np.asarray(p2d, np.float64)[m], axis=1)
        out['rep_p50'], out['rep_p90'] = float(np.median(r)), float(np.percentile(r, 90))
        if arcs is not None:
            aa = np.asarray(arcs)[m]
            out['inl_arc_span'] = float(aa.max() - aa.min())
            out['inl_arc_med'] = float(np.median(aa))
        if rows is not None and M is not None:
            imgs = np.unique(np.asarray(M.row_img7)[np.asarray(rows)[m]])
            out['n_sup_img'] = int(len(imgs))
            out['run_mask'] = int(np.bitwise_or.reduce(
                (1 << np.asarray(M.img_run)[imgs].astype(np.int64))))
    loc_prof('extras', _t)
    return out


def loc_seed_intrinsics(row):
    """(focal, k1, provenance) for a roll, from its Garmin udta settings where they exist.

    Never assume an uncovered roll falls in one of the four cells: 66% of rolls have no udta at
    all (VIRB Edit strips the blob), the cells rest on 8/1/1 map runs plus 20 fitted previews,
    and the on-disk edited-roll fits span focal 600->777.  This is a SEED for the two-pass fit,
    not an answer.
    """
    lc, stab = row.get('lens_correction'), row.get('stabilizer')
    if (lc, stab) in LOC_SEED_CLUSTERS:
        f, k = LOC_SEED_CLUSTERS[(lc, stab)]
        return f, k, f'udta:{lc}/{stab}'
    if row.get('seed_focal') is not None:
        return float(row['seed_focal']), float(row.get('seed_k1', LOC_SEED_DEFAULT[1])), 'plan'
    return LOC_SEED_DEFAULT[0], LOC_SEED_DEFAULT[1], 'default:ON/ON'


def loc_fit_intrinsics(M, corr, seed, stride=LOC_INTR_STRIDE):
    """Two-pass per-roll fit (PLAN 3.4 pass 1): refine focal per frame, k1 only via the median.

    Per-frame refine_extra_params drifts k1 to +0.05/+0.12/+0.44 against a true -0.036 while
    focal stays within 1%, so k1 is taken from the median over frames and never trusted per
    frame.  A FRESH Camera per frame, because refinement is written back in place.
    """
    est, ref = loc_pnp_options(0, refine_focal=True, refine_extra=True)
    order = sorted(corr)

    def fit_one(i):
        p2d, p3d = corr[i][0], corr[i][1]
        if len(p2d) < LOC_MIN_CORRS:
            return None
        cam = loc_camera(*seed[:2])
        r = loc_pnp(p2d, p3d, cam, est, ref)
        if r['ok'] and r['n_inl'] >= LOC_INTR_MIN_INL:
            return float(cam.params[0]), float(cam.params[3])
        return None

    def sweep(sel):
        # loc_pmap preserves order, and each frame's refine is independent with its own fresh
        # Camera, so fs/ks are the same lists in the same order as the serial sweep and the
        # medians below are unchanged.  Measured 3.34x at 4 workers.
        got = [x for x in loc_pmap(fit_one, sel, tag='intr.') if x is not None]
        return [a for a, _ in got], [b for _, b in got]

    fs, ks = sweep(order[::stride])
    rec = {'seed': [seed[0], seed[1]], 'seed_prov': seed[2], 'n_pass1': len(fs),
           'stride': stride}
    if len(fs) < LOC_INTR_MIN_FRAMES:
        fs, ks = sweep(order)
        rec['retried_every_frame'] = True
        rec['n_pass1'] = len(fs)
    if len(fs) >= LOC_INTR_MIN_FRAMES:
        rec.update(focal=float(np.median(fs)), k1=float(np.median(ks)), prov='fit')
    elif len(fs) >= LOC_INTR_MIN_FALLBACK:
        rec.update(focal=float(seed[0]), k1=float(seed[1]), prov='seed')
    else:
        rec.update(focal=LOC_SEED_DEFAULT[0], k1=LOC_SEED_DEFAULT[1], prov='map_default')
    rec['focal_spread'] = (float(np.percentile(fs, 90) - np.percentile(fs, 10))
                           if len(fs) >= 3 else None)
    return rec

## `loc-bootstrap`

Place every 10th frame against the global index and interpolate an arc estimate. Offloaded to the CPU workers.

In [ ]:
# === cell: loc-bootstrap ===
def loc_row_to_point(M, rows):
    """p7 bank row -> point index into pt_xyz.  row_pt6's twin for the arc-sorted bank.

    The p7 bank is point-major WITHIN arc order, so the point owning a row is simply the CSR
    bucket the row falls in: pt_start[j] <= row < pt_start[j+1], and pt_order[j] is that
    bucket's point.  This is the single most likely thing to be wrong in the exact bootstrap,
    so it is not trusted -- b4_dryrun checks it against the shipped p6a mapping on real rows:
    row_to_point(p6a2p7[r]) must equal row_pt6[r] for every r, and it is an equality over two
    independently built arrays, not a plausibility check.
    """
    j = np.searchsorted(M.pt_start, np.asarray(rows, np.int64), 'right') - 1
    return np.asarray(M.pt_order)[j]


def loc_exhaustive_knn(B, qd, k, tile_bytes=LOC_KNN_TILE_B, tile_rows=None):
    """Top-k inner product of `qd` against EVERY row of bank tensor `B`, tiled.  (values, idx).

    The score matrix for one anchor against the whole bank is 43.9 GiB, so it is never
    materialised: each tile is reduced to its own top-k and merged into the running best, which
    costs one extra k-wide merge per tile and nothing else.  Tile width comes from a BYTE
    budget rather than a fixed row count because nq is the frame's keypoint count and varies
    2x between frames.

    Candidate generation runs at the bank's own fp16; the caller re-scores the survivors in
    fp32 (loc_exact_rerank) before any ratio test, so fp16 only ever decides which ~32 of
    12.4M rows are looked at more carefully.  That is strictly more of the bank than IVFPQ's
    coarse-quantiser pruning ever examined.
    """
    import torch
    n, nq = int(B.shape[0]), int(qd.shape[0])
    k = int(min(k, n))
    Q = torch.as_tensor(np.ascontiguousarray(qd, np.float16)).to(B.device, B.dtype)
    # tile_rows is a test seam, never used in production: the byte budget floors the width at
    # 32768 rows for GPU efficiency, which on a small bank collapses to a single tile and would
    # let the merge path below go permanently unexercised
    tile = int(tile_rows) if tile_rows else max(1 << 15, int(tile_bytes // max(2 * nq, 1)))
    bv = bi = None
    for a in range(0, n, tile):
        S = Q @ B[a:a + tile].T
        v, i = torch.topk(S, min(k, S.shape[1]), dim=1)
        del S
        i = i + a
        if bv is None:
            bv, bi = v, i
        else:
            v = torch.cat([bv, v], 1)
            i = torch.cat([bi, i], 1)
            # min(): the running best has fewer than k columns until enough tiles have been
            # merged, and a narrow tile yields fewer than k of its own.  Asking topk for k
            # columns of a narrower tensor raises, which is what the tile_rows=1 case caught.
            bv, sel = torch.topk(v, min(k, v.shape[1]), dim=1)
            bi = i.gather(1, sel)
    return bv.float().cpu().numpy(), bi.cpu().numpy().astype(np.int64)


def loc_distinct_ratio(I, Dsq, row_pt, pts=None):
    """(nq,k) ANN result -> best point + Lowe ratio against the 2nd-nearest DISTINCT point.

    A point's own other observations are its own nearest neighbours, so the second distance MUST
    come from a different point3D.  When all k neighbours belong to one point the k-th distance
    is used, which makes the ratio an UPPER bound: the fallback can only reject a match exact
    search would keep, never accept one.
    """
    nq, k = I.shape
    d = np.where(I >= 0, np.maximum(Dsq, 0.0), np.inf).astype(np.float64)
    if pts is None:
        pts = np.where(I >= 0, np.asarray(row_pt)[np.maximum(I, 0)], -1).astype(np.int64)
    o = np.argsort(d, axis=1, kind='stable')
    r = np.arange(nq)
    d = np.take_along_axis(d, o, 1)
    pts = np.take_along_axis(pts, o, 1)
    ok = np.isfinite(d)
    has_any = ok[:, 0]
    best_pt = np.where(has_any, pts[:, 0], -1)
    d1 = np.where(has_any, d[:, 0], np.inf)
    diff = ok & (pts != best_pt[:, None]) & (pts >= 0)
    has2 = diff.any(1)
    j2 = np.argmax(diff, axis=1)
    dlast = np.where(ok, d, -np.inf).max(1)
    d2 = np.where(has2, d[r, j2], dlast)
    with np.errstate(divide='ignore', invalid='ignore'):
        ratio = np.where(d2 > 1e-12, np.sqrt(d1 / d2), np.inf)
    return best_pt.astype(np.int64), np.where(has_any, ratio, np.inf)


def loc_exact_rerank(M, qd, I, p7=False):
    """Exact squared-L2 for the ANN shortlist rows -- PQ distances corrupt the ratio test.

    The shortlist is in p6a row space and the only bank shipped is p7's fp16 one, so the rows
    are permuted before they are gathered.  fp16 was measured against fp32 on 400 frames: every
    disagreement above 30 m sits on a frame with n_inl <= 9, which this gate discards anyway.
    """
    nq, k = I.shape
    # the exhaustive path already searched the p7 bank, so its rows need no permutation
    flat = (np.maximum(I, 0).ravel().astype(np.int64) if p7
            else np.asarray(M.p6a2p7)[np.maximum(I, 0).ravel()])
    R = M.bank.rows(flat).float().cpu().numpy()
    # Contract against the k axis directly instead of np.repeat'ing the queries k times: the
    # repeat materialised an (nq*k, 128) float32 copy -- 31 MB per anchor -- and was 31.5% of
    # this function.  Measured 2.23x faster and BIT-IDENTICAL (np.array_equal on every frame
    # tested); a broadcast-multiply-.sum(-1) form was also tried and rejected, being both
    # slower and not bit-identical.
    s = np.einsum('ijk,ik->ij', R.reshape(nq, k, -1), qd)
    return np.maximum(0.0, 2.0 - 2.0 * s).reshape(nq, k).astype(np.float32)


def loc_pava(y, w=None):
    """Pool-adjacent-violators: the L2-optimal non-decreasing fit to y."""
    y = np.asarray(y, float)
    w = np.ones_like(y) if w is None else np.asarray(w, float)
    val, wt, cnt = [], [], []
    for yi, wi in zip(y, w):
        val.append(yi)
        wt.append(wi)
        cnt.append(1)
        while len(val) > 1 and val[-2] > val[-1]:
            v2, w2, c2 = val.pop(), wt.pop(), cnt.pop()
            v1, w1, c1 = val.pop(), wt.pop(), cnt.pop()
            val.append((v1 * w1 + v2 * w2) / (w1 + w2))
            wt.append(w1 + w2)
            cnt.append(c1 + c2)
    return np.repeat(np.array(val), np.array(cnt))


def loc_median_filter1d(x, size):
    """scipy.ndimage.median_filter(x, size, mode='nearest') for odd `size`, in numpy."""
    x = np.asarray(x, float)
    size = int(size)
    if size <= 1 or len(x) == 0:
        return x.copy()
    assert size % 2 == 1, 'odd windows only'
    p = np.pad(x, size // 2, mode='edge')
    return np.median(np.lib.stride_tricks.sliding_window_view(p, size), axis=1)


def loc_seq_smooth(s, win=LOC_SMOOTH_WIN):
    """Median filter then isotonic (PAVA) -- NOT a cumulative max.  On a gated track this is
    bit-identically the identity, because a gated track is already monotone."""
    s = np.asarray(s, float)
    return s if len(s) == 0 else loc_pava(loc_median_filter1d(s, win))


def loc_extrapolate(tq, t, s, y, k=5, arc_max=LOC_ARC_MAX):
    """Replace np.interp's flat end-hold with a least-squares speed from the end anchors."""
    y = y.copy()
    n = len(t)
    if n < 2:
        return y
    kk = min(k, n)
    for side in (0, 1):
        m = tq < t[0] if side == 0 else tq > t[-1]
        if not m.any():
            continue
        tt, ss = (t[:kk], s[:kk]) if side == 0 else (t[-kk:], s[-kk:])
        if tt[-1] <= tt[0]:
            continue
        v = np.polyfit(tt, ss, 1)[0]
        anc_t, anc_s = (t[0], s[0]) if side == 0 else (t[-1], s[-1])
        y[m] = anc_s + v * (tq[m] - anc_t)
    return np.clip(y, 0.0, arc_max)


def loc_bootstrap(ts_all, anchor_idx, est, ok, n_inl, gate=LOC_BOOT_GATE,
                  win=LOC_SMOOTH_WIN, extrap=True):
    """Gate -> smooth -> interpolate to every frame.  Returns (arc over ALL frames, accepted).

    The GATE does the work, not the smoothing: raw ANN error is bimodal (centimetres, or
    hundreds of metres on a lookalike hill) and the failures carry low inlier counts.
    """
    acc = np.asarray(ok, bool) & (np.asarray(n_inl) >= gate)
    if acc.sum() == 0:
        return None, acc
    t = np.asarray(ts_all, np.float64)[np.asarray(anchor_idx)][acc]
    s = loc_seq_smooth(np.asarray(est, float)[acc], win=win)
    if len(s) == 1:
        return np.full(len(ts_all), s[0]), acc
    tq = np.asarray(ts_all, np.float64)
    y = np.interp(tq, t, s)
    return (loc_extrapolate(tq, t, s, y) if extrap else y), acc


def loc_blind_gap(ts_all, anchor_idx, arc_hat, acc):
    """Largest arc distance spanned between consecutive ACCEPTED anchors, head and tail
    included.  The pilot gate's second axis."""
    if arc_hat is None or acc.sum() == 0:
        return float(LOC_ARC_MAX)
    j = np.asarray(anchor_idx)[acc]
    a = np.concatenate([[arc_hat[0]], np.asarray(arc_hat)[j], [arc_hat[-1]]])
    return float(np.abs(np.diff(a)).max()) if len(a) > 1 else 0.0


def loc_boot_exact(M):
    """Does the bootstrap search the whole bank on the device, or go through faiss?

    `auto` means "wherever the bank already is".  The resident p7 fp16 bank IS the search
    structure when it is on a GPU -- no transfer, no second copy, no index to keep in step with
    it -- and on the CPU an exhaustive 6.03 TFLOP per anchor is not a search, it is a stall, so
    faiss stays the CPU answer and the dry run keeps exercising it.
    """
    if LOC_BOOT_KNN in ('gpu', 'exact'):
        return True
    if LOC_BOOT_KNN == 'faiss':
        return False
    return getattr(M.bank, 't', None) is not None


def loc_pin_omp():
    """One OpenMP thread for THIS thread.  Must run inside each worker, never in the parent.

    `omp_set_num_threads` sets a PER-OS-THREAD value, so setting it on the main thread does
    nothing for pool workers -- they inherit the runtime default of one thread per core and
    every worker then spawns a full-width parallel region.  Measured: that oversubscription
    turned an expected ~4x into 1.18x and made the results differ from the serial ones; with
    the pin applied inside each worker the same pool gives 1.94x and is BIT-IDENTICAL.
    """
    try:
        import faiss
        faiss.omp_set_num_threads(1)
    except Exception:                                           # noqa: BLE001
        pass


def loc_run_bootstrap(M, feats, seed, est, ref, stride=LOC_BOOT_STRIDE, out=None):
    """Every `stride`-th frame against the global faiss index, no position prior.

    Pooled across ANCHORS, with faiss pinned to one OpenMP thread inside each worker.  That is
    both the faster and the safer of the two ways to spend the cores on this stage, measured
    rather than assumed: coarse (pool, omp=1) is 1.94x and bit-identical to serial, while fine
    (serial, omp=8) is 1.60x and NOT bit-identical -- faiss's PQ distances vary in the last ulp
    with the OpenMP width, which is enough to flip near-ties in the returned neighbours.  Each
    anchor is wholly independent: its own query, its own Camera, its own PnP.
    """
    idx = [i for i in sorted(feats['desc32']) if i % stride == 0]
    exact = loc_boot_exact(M)
    # anchors split across the bank replicas, ONE at a time per device: the candidate search is
    # a device's whole width, so two on one device would only contend for it and double the
    # tile allocation.  The CPU half of each anchor (rerank, ratio, PnP) still overlaps freely.
    bdev = list(M.bank.devices)
    locks = {d: threading.Lock() for d in bdev}
    bbusy = {d: [0.0, 0] for d in bdev}

    def anchor(item):
        pos_i, i = item
        qd = np.ascontiguousarray(feats['desc32'][i], np.float32)
        kp = feats['kp'][i]
        if len(qd) == 0:
            return np.nan, 0, 0
        t = time.perf_counter()
        if exact:
            d = bdev[pos_i % len(bdev)]
            with locks[d]:
                t = time.perf_counter()
                _, I = loc_exhaustive_knn(M.bank.ts[d], qd, LOC_KNN)
                bbusy[d][0] += time.perf_counter() - t
                bbusy[d][1] += 1
            t = loc_prof('knn', t)
            Dsq = loc_exact_rerank(M, qd, I, p7=True)
            t = loc_prof('rerank', t)
            best, ratio = loc_distinct_ratio(I, Dsq, None, pts=loc_row_to_point(M, I))
        else:
            _, I = M.index.search(qd, LOC_KNN)
            t = loc_prof('faiss', t)
            Dsq = loc_exact_rerank(M, qd, I)
            t = loc_prof('rerank', t)
            best, ratio = loc_distinct_ratio(I, Dsq, M.row_pt6)
        loc_prof('ratio', t)
        keep = (best >= 0) & (ratio < LOC_RATIO)
        # a FRESH camera per anchor: shared mutable state has no business in a pool, and with
        # both refine flags off this is the same camera the serial path used
        r = loc_pnp(kp[keep].astype(np.float64), M.pt_xyz[best[keep]],
                    loc_camera(*seed), est, ref, M)
        return (r['arc'] if r['ok'] else np.nan), r['ok'], r['n_inl']

    # enough workers to keep every device fed AND overlap the CPU half of the other anchors
    w = LOC_BOOT_WORKERS or (max(loc_workers(), len(bdev)) if exact
                             else loc_workers())
    got = loc_pmap(anchor, list(enumerate(idx)), workers=w, tag='boot.',
                   initializer=loc_pin_omp)
    if out is not None:
        out['boot_per_device'] = {d: {'n': v[1], 'busy_s': round(v[0], 2),
                                      'ms_per_anchor': round(1000 * v[0] / max(v[1], 1), 2)}
                                  for d, v in bbusy.items()}
        out['boot_exact'] = bool(exact)
    return (np.array(idx, np.int64), np.array([g[0] for g in got], np.float64),
            np.array([g[1] for g in got], np.int8), np.array([g[2] for g in got], np.int32))

## `loc-match`

Per-frame shortlist within +/-30 m of the estimated arc, matched against the bank.

In [ ]:
# === cell: loc-match ===
def loc_shortlist(M, arc_est, half=LOC_WINDOW_M, topk=LOC_SHORT_K, cap=LOC_CAP):
    """The r0k8_cap4 candidate bank for one query, straight out of the resident bank.

    r0 = the `topk` map frames nearest in arc; their observed points, restricted to the query's
    own +/-`half` m band, capped at `cap` observations each.  The p7 bank is point-major inside
    arc order, so a point's capped rows are one contiguous slice and no per-query assembly of
    the window is needed.  Returns (rows, owner, pt_arcpos, ptr) or None.
    """
    pa, pb = np.searchsorted(M.pt_arc_sorted, (arc_est - half, arc_est + half))
    if pb <= pa:
        return None
    short = np.argsort(np.abs(M.img_arc - arc_est), kind='stable')[:topk]
    parts = [np.asarray(M.img_obs_pt[M.img_ptr[m]:M.img_ptr[m + 1]]) for m in short]
    if not parts:
        return None
    pos = np.unique(M.inv_pt_order[np.concatenate(parts)])
    pos = pos[(pos >= pa) & (pos < pb)]
    if len(pos) == 0:
        return None
    cnt = np.minimum(M.pt_start[pos + 1] - M.pt_start[pos], cap).astype(np.int64)
    pos = pos[cnt > 0]
    cnt = cnt[cnt > 0]
    ptr = np.zeros(len(cnt) + 1, np.int64)
    np.cumsum(cnt, out=ptr[1:])
    tot = int(ptr[-1])
    rows = (np.repeat(M.pt_start[pos], cnt)
            + (np.arange(tot, dtype=np.int64) - np.repeat(ptr[:-1], cnt)))
    owner = np.repeat(np.arange(len(pos), dtype=np.int32), cnt)
    assert (cnt <= cap).all(), 'a point exceeded the observation cap'
    return rows, owner, pos, ptr


def loc_shortlist_stream(M, arc_hat, n, depth=LOC_SHORTLIST_QUEUE):
    """Yield (i, shortlist) with frame i+1's bank assembled while the GPU matches frame i.

    Pure CPU numpy against a pure GPU consumer, which is the smallest honest instance of the
    3-stage overlap PLAN 4 costs the run on: measured 2.8 ms/frame to assemble a ~30,000-row
    30 m window, which was previously serialised in front of every GEMM.  Order and contents
    are exactly the serial loop's; only the wall clock changes.
    """
    q = queue.Queue(maxsize=depth)

    def run():
        try:
            for i in range(n):
                t = time.perf_counter()
                s = loc_shortlist(M, float(arc_hat[i]))
                loc_prof('shortlist', t)
                q.put((i, s, None))
        except Exception as e:                                  # noqa: BLE001
            q.put((None, None, e))
        q.put((None, None, None))

    threading.Thread(target=run, daemon=True).start()
    while True:
        i, s, err = q.get()
        if err is not None:
            raise err
        if i is None:
            return
        yield i, s


def loc_match_frame(M, qd, short, ratio_thr=LOC_RATIO, topk=LOC_TOPK, device=None):
    """fp16 GEMM + top-(CAP+1) reduction -> (query index, point arc-position, ratio, best row).

    Under cap 4 at most 4 of the top 5 columns can share a point, so the 5th is provably
    distinct and the ratio test never needs a second pass.  k is derived from CAP in code.
    """
    import torch
    rows, owner, pos, ptr = short
    assert topk == LOC_CAP + 1, 'k must be CAP+1'
    t = time.perf_counter()
    D = M.bank.rows(rows, device)
    Q = torch.from_numpy(np.ascontiguousarray(qd, np.float16)).to(D.device)
    if D.device.type == 'cpu':          # no fast fp16 GEMM on the CPU; the dry run needs this
        D, Q = D.float(), Q.float()
    own = torch.as_tensor(np.asarray(owner, np.int64), device=D.device)
    loc_prof_sync()
    t = loc_prof('gather', t)
    PTR = torch.as_tensor(np.asarray(ptr, np.int64), device=D.device)
    out_p, out_r, out_row = [], [], []
    blk = max(1, int(LOC_MATCH_BLOCK // max(1, len(rows))))
    off = torch.arange(LOC_CAP, device=D.device, dtype=torch.int64)
    for a in range(0, len(Q), blk):
        S = (Q[a:a + blk] @ D.T).float()
        loc_prof_sync()
        t = loc_prof('gemm', t)
        # BLANK-AND-ARGMAX, not topk(CAP+1).  A point's observations are a CONTIGUOUS run in
        # the shortlist, so the second-nearest DISTINCT point is found by taking the argmax,
        # blanking that point's <= CAP columns, and taking the argmax again -- two reductions
        # and CAP writes per row, against a k-wide sort of every column.  Measured on a T4 in
        # p6b: 12.33 ms vs 41.08 ms for the reduce, bit-identical over n=9411.  -2.0 is the
        # same sentinel the topk path used when no distinct second exists, and it is safe
        # because every score is a dot of unit vectors and cannot go below -1.
        c1 = S.argmax(1)
        s1 = S.gather(1, c1[:, None]).squeeze(1)
        best = own[c1]
        cols = torch.minimum(PTR[best][:, None] + off[None, :], (PTR[best + 1] - 1)[:, None])
        S.scatter_(1, cols, -2.0)
        s2 = S.amax(1)
        del S
        loc_prof_sync()
        t = loc_prof('reduce', t)
        out_p.append(best.cpu().numpy())
        out_r.append(np.stack([s1.cpu().numpy(), s2.cpu().numpy()], 1))
        out_row.append(c1.cpu().numpy())
        t = loc_prof('d2h', t)
    if not out_p:
        return (np.zeros(0, np.int64),) * 2 + (np.zeros(0, np.float64), np.zeros(0, np.int64))
    P = np.concatenate(out_p).astype(np.int64)
    V = np.concatenate(out_r).astype(np.float64)
    d1 = np.sqrt(np.maximum(0.0, 2.0 - 2.0 * V[:, 0]))
    d2 = np.sqrt(np.maximum(0.0, 2.0 - 2.0 * V[:, 1]))
    with np.errstate(divide='ignore', invalid='ignore'):
        ratio = np.where(d2 > 1e-9, d1 / d2, np.inf)
    keep = ratio < ratio_thr
    col = np.concatenate(out_row)[keep]
    loc_prof('post', t)
    return np.flatnonzero(keep), pos[P[keep]], ratio[keep], np.asarray(rows)[col]

## `loc-record`

Serialize a roll into its two output objects, including the failed-roll form.

In [ ]:
# === cell: loc-record ===
def loc_record(roll_row, feats, info, boot, intr, per, M):
    """Columnar per-frame record (PLAN 3.6) + the 2D-3D correspondences, as two npz payloads.

    Columnar, not JSONL: the JSONL comparator runs ~290 KB per 1,700 frames and parses; this
    loads straight into numpy.  There is no reason to economise on fields -- the whole record
    is negligible next to the correspondences and invisible next to descriptors.
    """
    n = len(per)
    g = lambda k, dt, d=np.nan: np.array([p.get(k, d) if p.get(k, None) is not None else d
                                          for p in per], dt)                    # noqa: E731
    centre = np.array([p['centre'] if p['centre'] is not None else [np.nan] * 3
                       for p in per], np.float64)
    quat = np.array([p['quat'] if p['quat'] is not None else [np.nan] * 4 for p in per],
                    np.float64)
    cov = np.array([p['cov'] if p['cov'] is not None else np.full((6, 6), np.nan)
                    for p in per], np.float32)
    info = dict(info)
    arc_hat = info.pop('arc_hat', None)         # stored as a column, not repeated in meta
    anchor_idx, arcs_b, ok_b, inl_b, acc = boot
    is_anchor = np.zeros(n, np.uint8)
    boot_inl = np.full(n, -1, np.int32)
    boot_ok = np.zeros(n, np.uint8)
    boot_acc = np.zeros(n, np.uint8)
    is_anchor[anchor_idx] = 1
    boot_inl[anchor_idx] = inl_b
    boot_ok[anchor_idx] = ok_b
    boot_acc[anchor_idx] = acc.astype(np.uint8)

    t_s = feats['t_ms'].astype(np.float64) / 1000.0
    dv = np.full(n, np.nan)
    dv[1:] = np.linalg.norm(np.diff(centre, axis=0), axis=1) / np.maximum(
        np.diff(t_s), 1e-6)
    chord_prev, chord_next = dv, np.roll(dv, -1)
    chord_next[-1] = np.nan
    arc = g('arc', np.float64)
    loo = np.full(n, np.nan)
    if n >= 3:
        w = (t_s[1:-1] - t_s[:-2]) / np.maximum(t_s[2:] - t_s[:-2], 1e-6)
        loo[1:-1] = arc[1:-1] - (arc[:-2] + w * (arc[2:] - arc[:-2]))

    z = dict(
        roll=np.int64(roll_row['roll']), file_id=np.int64(roll_row['file_id']),
        video=str(roll_row.get('remote') or roll_row.get('local_path') or ''),
        frame_idx=np.arange(n, dtype=np.int32), t_ms=feats['t_ms'].astype(np.int64),
        dec_idx=feats['dec_idx'],
        # `n_kp` where the caller kept only the slim columns: a roll awaiting its remote tail
        # drops ~1 GiB of keypoints and descriptors rather than hold them for a second roll
        n_kp=(np.asarray(feats['n_kp'], np.int32) if 'n_kp' in feats
              else np.array([len(k) for k in feats['kp']], np.int32)),
        centre=centre, quat=quat, arc_pnp=arc, lat_off=g('lat', np.float64),
        ok=g('ok', np.uint8, 0), n_corr=g('n_corr', np.int32, 0),
        n_inl=g('n_inl', np.int32, 0),
        inl_ratio=np.where(g('n_corr', np.float64, 0) > 0,
                           g('n_inl', np.float64, 0) / np.maximum(g('n_corr', np.float64, 1), 1),
                           np.nan),
        inl_arc_span=g('inl_arc_span', np.float32), inl_arc_med=g('inl_arc_med', np.float32),
        rep_p50=g('rep_p50', np.float32), rep_p90=g('rep_p90', np.float32), cov=cov,
        n_sup_img=g('n_sup_img', np.int32, 0), sup_run_mask=g('run_mask', np.int64, 0),
        is_anchor=is_anchor, boot_ok=boot_ok, boot_n_inl=boot_inl, boot_accepted=boot_acc,
        arc_boot=(np.asarray(arc_hat, np.float64) if arc_hat is not None
                  else np.full(n, np.nan)),
        arc_interpolated=(1 - boot_acc).astype(np.uint8),
        focal=np.float64(intr['focal']), k1=np.float64(intr['k1']),
        intr_prov=str(intr['prov']), intr_seed=np.asarray(intr['seed'], np.float64),
        intr_seed_prov=str(intr['seed_prov']),
        chord_prev=chord_prev.astype(np.float32), chord_next=chord_next.astype(np.float32),
        arc_loo_resid=loo.astype(np.float32),
        map_runs=np.array(M.runs), in_map=np.uint8(1 if roll_row.get('in_map') else 0),
        sync=str(roll_row.get('sync') or ''), meta=json.dumps(info, default=loc_jsonable))
    return z


def loc_failed_record(row, err):
    """A roll that could not be processed still gets an output object, so the cursor advances
    and the roll is not retried forever.  It carries the same columns at length 0 plus `error`,
    so a reader cannot mistake it for a solved roll."""
    z = dict(roll=np.int64(row['roll']), file_id=np.int64(row['file_id']),
             video=str(row.get('remote') or row.get('local_path') or ''), error=str(err),
             failed=np.uint8(1), in_map=np.uint8(1 if row.get('in_map') else 0),
             sync=str(row.get('sync') or ''), meta=json.dumps({'error': str(err)}),
             map_runs=np.zeros(0, '<U1'), focal=np.float64(np.nan), k1=np.float64(np.nan),
             intr_prov='failed', intr_seed=np.zeros(2), intr_seed_prov='',
             centre=np.zeros((0, 3)), quat=np.zeros((0, 4)), cov=np.zeros((0, 6, 6), np.float32))
    for k, dt in (('frame_idx', np.int32), ('t_ms', np.int64), ('dec_idx', np.int32),
                  ('n_kp', np.int32), ('ok', np.uint8), ('n_corr', np.int32),
                  ('n_inl', np.int32), ('is_anchor', np.uint8), ('boot_ok', np.uint8),
                  ('boot_n_inl', np.int32), ('boot_accepted', np.uint8),
                  ('arc_interpolated', np.uint8), ('n_sup_img', np.int32),
                  ('sup_run_mask', np.int64), ('arc_pnp', np.float64),
                  ('lat_off', np.float64), ('arc_boot', np.float64),
                  ('inl_ratio', np.float64), ('inl_arc_span', np.float32),
                  ('inl_arc_med', np.float32), ('rep_p50', np.float32),
                  ('rep_p90', np.float32), ('chord_prev', np.float32),
                  ('chord_next', np.float32), ('arc_loo_resid', np.float32)):
        z[k] = np.zeros(0, dt)
    return z, dict(off=np.zeros(1, np.int64), kp_xy=np.zeros((0, 2), np.float32),
                   point3D_id=np.zeros(0, np.int64), inlier=np.zeros(0, np.uint8),
                   ratio=np.zeros(0, np.float32))


def loc_failed_summary(row, err, wall_s=0.0):
    """The summary a failed roll carries.  Every field the loop and the pilot read, at zero."""
    return {'roll': int(row['roll']), 'in_map': bool(row.get('in_map')), 'failed': True,
            'error': f'{type(err).__name__}: {str(err)[:200]}', 'n_frames': 0,
            'n_anchor': 0, 'n_anchor_accepted': 0, 'blind_gap_m': float(LOC_ARC_MAX),
            'n_solved': 0, 'n_inl_p50': 0.0, 'rep_p50': float('nan'),
            'wall_s': round(wall_s, 1)}


def loc_record_corr(per, M):
    """The 2D-3D correspondences, so PnP is re-runnable at home without a GPU.

    ~5 KB/frame = ~10 GB over 1.9M frames against 1.11 TB for the descriptors, and it means a
    revised intrinsics choice costs CPU hours instead of 35 T4-h of re-extraction.
    """
    off = np.zeros(len(per) + 1, np.int64)
    kp, pid, inl, rat = [], [], [], []
    for i, p in enumerate(per):
        c = p.get('corr')
        if c is None:
            off[i + 1] = off[i]
            continue
        k2, pts, ratio = c
        m = p.get('inlier_mask')
        kp.append(np.asarray(k2, np.float32))
        pid.append(M.pt_id[M.pt_order[np.asarray(pts)]].astype(np.int64))
        inl.append((m if m is not None and len(m) == len(pts)
                    else np.zeros(len(pts), bool)).astype(np.uint8))
        rat.append(np.asarray(ratio, np.float32))
        off[i + 1] = off[i] + len(pts)
    cat = lambda xs, dt, w: (np.concatenate(xs) if xs else np.zeros((0, w) if w else 0, dt))
    return dict(off=off, kp_xy=cat(kp, np.float32, 2), point3D_id=cat(pid, np.int64, 0),
                inlier=cat(inl, np.uint8, 0), ratio=cat(rat, np.float32, 0))

## `loc-gate`

Pre-flight gates: interpreter, GPU inventory (which also resolves the role), proof the graph ran on a GPU provider, pilot scoring.

In [ ]:
# === cell: loc-gate ===
def loc_env_probe():
    """Is the interpreter still usable?  A GPU gate that passes and an interpreter that dies
    30 s later costs the same GPU time as a failure and looks like a different bug."""
    out = {}
    try:
        import numpy.ma
        assert float(numpy.ma.masked_array([1.0, 2.0, 3.0], mask=[0, 1, 0]).mean()) == 2.0
        import cv2
        out['cv2'] = cv2.__version__
        import faiss
        out['faiss'] = getattr(faiss, '__version__', '?')
        import pycolmap
        out['pycolmap'] = pycolmap.__version__
        import torch
        out['torch'] = torch.__version__
        out['cuda_devices'] = int(torch.cuda.device_count())
        out['numpy'] = np.__version__
        out['ok'] = True
    except Exception as e:                                      # noqa: BLE001
        out['ok'] = False
        out['error'] = f'{type(e).__name__}: {str(e)[:400]}'
    return out


def loc_gpu_inventory():
    """What accelerator this session ACTUALLY got, read two independent ways.

    `nvidia-smi -L` is the machine's own answer and the CUDA runtime's device count is what the
    execution providers will use; they can disagree (CUDA_VISIBLE_DEVICES, a wedged device), so
    both are reported verbatim and the smaller is what the budget is checked against.
    """
    inv = {'min_required': LOC_MIN_GPUS,
           'cuda_visible_devices': os.environ.get('CUDA_VISIBLE_DEVICES'),
           'devices_requested': list(LOC_DEVICES)}
    try:
        r = loc_sh('nvidia-smi -L', check=False)
        inv['nvidia_smi_L'] = [x for x in (r.stdout or '').split('\n') if x.strip()]
        inv['nvidia_smi_rc'] = r.returncode
        inv['n_smi'] = len(inv['nvidia_smi_L'])
        inv['names'] = [x.split(': ', 1)[-1].split(' (UUID')[0] for x in inv['nvidia_smi_L']]
    except Exception as e:                                      # noqa: BLE001
        inv['nvidia_smi_error'] = f'{type(e).__name__}: {str(e)[:200]}'
        inv['n_smi'] = None
    try:
        import torch
        inv['n_cuda'] = int(torch.cuda.device_count())
        inv['cuda_names'] = [torch.cuda.get_device_name(i) for i in range(inv['n_cuda'])]
        inv['total_mem_gib'] = [round(torch.cuda.get_device_properties(i).total_memory / 2 ** 30, 1)
                                for i in range(inv['n_cuda'])]
    except Exception as e:                                      # noqa: BLE001
        inv['cuda_error'] = f'{type(e).__name__}: {str(e)[:200]}'
        inv['n_cuda'] = None
    try:
        import onnxruntime as ort
        inv['ort_providers'] = ort.get_available_providers()
    except Exception as e:                                      # noqa: BLE001
        inv['ort_error'] = f'{type(e).__name__}: {str(e)[:200]}'
    counts = [c for c in (inv.get('n_smi'), inv.get('n_cuda')) if c is not None]
    inv['n_gpus'] = min(counts) if counts else None
    inv['counts_disagree'] = len(set(counts)) > 1
    inv['probe_complete'] = inv['n_gpus'] is not None
    return inv


def loc_gpu_verdict(inv, min_gpus=None):
    """None to proceed, else (exit code, reason).  Separated from the exit so the decision is
    testable without a GPU and without killing the process."""
    need = LOC_MIN_GPUS if min_gpus is None else min_gpus
    if not inv.get('probe_complete'):
        return (8, f'the GPU inventory probe did not complete '
                   f'({inv.get("nvidia_smi_error") or inv.get("cuda_error")}). This is NOT '
                   f'evidence about the accelerator - it is a broken instrument.')
    if inv['n_gpus'] < need:
        return (7, f'{inv["n_gpus"]} usable GPU(s) ({inv.get("names") or inv.get("cuda_names")}), '
                   f'but the {LOC_BUDGET_H} h budget is priced on {need} at {LOC_REF_FPS} f/s. '
                   f'Kaggle does not validate machine_shape, so this is the first point the '
                   f'accelerator can be known. On one T4 the run is ~2x the budget.')
    return None


def loc_gpu_gate(st, inv=None):
    """The kernel's FIRST real action: know the accelerator before spending on anything else."""
    inv = loc_gpu_inventory() if inv is None else inv
    st['gpu_inventory'] = inv
    LOC_HB_EXTRA.update(n_gpus=inv.get('n_gpus'),
                        gpu_names=(inv.get('names') or inv.get('cuda_names')))
    loc_log(f'GPU inventory: n_gpus={inv.get("n_gpus")} (nvidia-smi {inv.get("n_smi")}, CUDA '
            f'{inv.get("n_cuda")}{", DISAGREE" if inv.get("counts_disagree") else ""}) '
            f'{inv.get("names") or inv.get("cuda_names")} mem={inv.get("total_mem_gib")} '
            f'CUDA_VISIBLE_DEVICES={inv.get("cuda_visible_devices")}')
    for line in inv.get('nvidia_smi_L', []):
        loc_log(f'  nvidia-smi -L: {line}')
    if loc_role() == 'cpu':
        # This gate exists to refuse to spend a GPU-priced budget on the wrong accelerator.  A
        # CPU worker is not spending that budget, and it is a CPU session ON PURPOSE, so the
        # gate has nothing to say about it; running it anyway would exit 7 on every worker.
        inv['enforced'] = False
        inv['role'] = 'cpu'
        loc_log(f'CPU WORKER ROLE ({inv.get("n_gpus")} GPU(s), counts agree): the GPU inventory '
                f'gate does not apply and is SKIPPED. This session installs no onnxruntime and '
                f'no TensorRT, builds no graph, and only ever runs the bootstrap.', push_=True)
        return inv
    verdict = loc_gpu_verdict(inv)
    enforced = not LOC_DRY and LOC_EP != 'cpu' and os.environ.get('SRS_LOC_GATE', '1') != '0'
    inv['enforced'] = enforced
    if verdict and not enforced:
        loc_log(f'*** GPU inventory would FAIL (exit {verdict[0]}): {verdict[1]} - NOT '
                f'enforced because this is a dry/CPU run ***')
        return inv
    if verdict:
        st['stop_reason'] = f'gpu inventory: {verdict[1]}'
        loc_save_state(st)
        loc_heartbeat({'roll': None, 'roll_idx': 0, 'n_rolls': 0, 'frame': 0, 'n_frames': 0,
                       'rate_fps': None, 'eta_utc': None, 'stage': f'ABORT_{verdict[0]}'},
                      force=True)
        loc_log(f'GPU INVENTORY GATE FAILED (exit {verdict[0]}): {verdict[1]} Nothing further '
                f'will be spent.', push_=True)
        sys.exit(verdict[0])
    loc_log(f'GPU inventory gate PASSED: {inv["n_gpus"]} >= {LOC_MIN_GPUS} required')
    return inv


def loc_gate(model, major=None):
    """HARD precondition before anything is spent: a GPU EP must really execute this graph.

    Two probes, because they answer different questions.  The in-process LocRunner is the code
    path the run will actually use.  The subprocess probe reads ORT's own NODE-PROVIDER
    HISTOGRAM, which is the only thing that says what executed: `get_available_providers()`
    lists REGISTERED providers, and even a session's realised list can be a CUDA provider that
    assigned every node to the CPU.  Exit 2 is decided on the histogram whenever it exists.

    Distinct exit codes on purpose -- a broken instrument must not look like a negative result.
      2  no GPU execution provider actually ran the graph
      3  the environment is unusable (the GPU may be fine; a later import would abort the run)
      4  the probe itself did not complete, which is NOT evidence about the image
    """
    rec = {'ep': LOC_EP, 'env': loc_env_probe()}
    if os.environ.get('SRS_LOC_GATE', '1') == '0':
        rec['bypassed'] = True
        loc_log('*** GATE DISABLED via SRS_LOC_GATE=0 - this run\'s rates are not GPU rates ***')
        return rec
    if not rec['env']['ok']:
        loc_log(f'GATE FAILED on the ENVIRONMENT: {rec["env"].get("error")}', push_=True)
        sys.exit(3)
    try:
        import onnxruntime as ort
        rec['available_providers'] = ort.get_available_providers()   # REGISTERED, not loadable
        r = LocRunner(model, LOC_EP, LOC_DEVICES[0])
        x = np.zeros((1, 3, LOC_H + 16, LOC_W), np.float32)
        t0 = time.perf_counter()
        o = r.run(x)
        rec['realised'] = r.eps
        rec['probe_s'] = round(time.perf_counter() - t0, 3)
        rec['out_shapes'] = [list(np.asarray(a).shape) for a in o[:3]]
        rec['gpu_ep'] = any(p.startswith(('Tensorrt', 'CUDA')) for p in r.eps)
    except Exception as e:                                      # noqa: BLE001
        rec['error'] = f'{type(e).__name__}: {str(e)[:500]}'
        loc_log(f'GATE FAILED, BUT THE PROBE ITSELF DID NOT COMPLETE: {rec["error"]}. This is '
                f'NOT evidence that the image lacks a GPU provider.', push_=True)
        sys.exit(4)
    if LOC_EP != 'cpu' and not LOC_DRY:
        p = rec['node_probe'] = loc_run_probe(model, LOC_EP, LOC_DEVICES[0], major)
        rec['hist'] = p.get('hist')
        if p.get('verdict'):                  # it reached the histogram, so its answer counts
            rec['gpu_ep'] = bool(p.get('ok'))
        else:
            loc_log(f'gate node-histogram probe did not complete ({p.get("error")}); falling '
                    f'back to the in-process realised providers, which are weaker evidence')
    if LOC_EP != 'cpu' and not rec['gpu_ep']:
        loc_log(f'GATE FAILED: realised providers {rec["realised"]}, node histogram '
                f'{rec.get("hist")} - every frame would be extracted on the CPU while two T4s '
                f'are billed.', push_=True)
        sys.exit(2)
    loc_log(f'GATE PASSED: {rec["realised"]} ran the graph in {rec["probe_s"]}s, outputs '
            f'{rec["out_shapes"]}, node histogram {rec.get("hist")}; env {rec["env"]}')
    return rec


def loc_pilot_eval(rolls, per_roll, inv=None, all_rolls=None, st_done=0):
    """The pre-registered pilot gate (tmp/backlog/prereg.json).

    In-map rolls are EXCLUDED from both axes: their own frames are in the map, so their inliers
    can be partly self-matches and their numbers flatter the pipeline.

    Two things beyond the two pre-registered axes are folded in here.  The GPU count, because
    the user authorised auto-continue against a 20 h budget priced on TWO T4s and spending it
    at half rate is not what was agreed -- so a pilot that completed on fewer GPUs does not
    auto-continue whatever the two axes say.  And the MEASURED extraction rate, so the ETA is
    grounded in what this session achieved rather than the plan's 32.95 f/s projection.
    """
    back = [r for r in per_roll if not r.get('in_map')]
    extended = len(back) < 5
    n_anc = sum(r['n_anchor'] for r in back)
    n_ok = sum(r['n_anchor_accepted'] for r in back)
    frac = (n_ok / n_anc) if n_anc else 0.0
    gap = max([r['blind_gap_m'] for r in back], default=float(LOC_ARC_MAX))
    s1 = 'proceed' if frac >= 0.90 else ('marginal' if frac >= 0.60 else 'stop')
    s2 = 'proceed' if gap <= 100 else ('marginal' if gap <= 300 else 'stop')
    rank = {'proceed': 0, 'marginal': 1, 'stop': 2}
    verdict = s1 if rank[s1] >= rank[s2] else s2
    keys = ('roll', 'ctl_n', 'ctl_p50', 'ctl_p90', 'ctl_max', 'ctli_n', 'ctli_p50',
            'ctli_p90', 'ctli_max', 'ctl_offset_ms', 'ctl_offset_source',
            'ctl_offset_confidence', 'ctl_residual_dt_ms', 'ctl_residual_p50_at_dt',
            'ctl_residual_ok', 'ctl_name_scale', 'ctl_clock_overlap',
            'self_run_inlier_frac')
    ctl = [r for r in per_roll if r.get('in_map')]
    bad_ctl = [r['roll'] for r in ctl if r.get('ctl_offset_source') != 'derived'
               or r.get('ctl_residual_ok') is False]
    fps = [r['extract_fps'] for r in per_roll if r.get('extract_fps')]
    n_gpus = (inv or {}).get('n_gpus')
    gpu_ok = n_gpus is None or n_gpus >= LOC_MIN_GPUS
    fps_med = float(np.median(fps)) if fps else None
    # what a roll actually COSTS the session: the loop's own wall clock per roll, which with
    # the look-ahead engaged already excludes the extract it hid under the previous roll
    walls = [float(r['wall_s']) for r in per_roll if r.get('wall_s') and not r.get('failed')]
    wall_med = float(np.median(walls)) if walls else None
    n_plan = len(all_rolls or rolls)
    n_done = int(st_done or 0)
    budget_left = max(0.0, LOC_BUDGET_H - (LOC_BASE_S[0] + (time.time() - LOC_T0)) / 3600.0)
    return {'n_rolls': len(per_roll), 'n_backlog_rolls': len(back), 'extended': extended,
            'n_anchors': n_anc, 'n_anchors_accepted': n_ok,
            'gate40_pass_fraction': round(frac, 4), 'gate40_score': s1,
            'longest_blind_gap_m': round(gap, 2), 'blind_gap_score': s2, 'verdict': verdict,
            'n_gpus': n_gpus, 'min_gpus': LOC_MIN_GPUS, 'gpu_ok': bool(gpu_ok),
            'gpu_gate_enforced': bool((inv or {}).get('enforced', True)),
            'n_in_map_controls': len(ctl), 'controls_ok': not bad_ctl,
            'controls_failed': bad_ctl,
            'control_p50_m': (None if not ctl else round(float(np.median(
                [r['ctli_p50'] for r in ctl if r.get('ctli_p50') is not None] or [np.nan])), 4)),
            'gpu_names': (inv or {}).get('names') or (inv or {}).get('cuda_names'),
            # EXTRACTION IS A DIAGNOSTIC, NOT THE PROJECTION.  It is fully hidden under the CPU
            # stages by the look-ahead (measured: waited 0.0 s, saved 69-79 s per roll), so it
            # never appears in the critical path; projecting the plan from it under-reported
            # the cost by 2.6x -- 21.8 h against a real ~52.7 h -- and both the pilot's
            # auto-continue and the budget refusal read this number.
            'extract_fps_measured': (None if fps_med is None else round(fps_med, 2)),
            'extract_fps_reference': LOC_REF_FPS,
            'extract_fps_ratio': (None if not fps_med else round(fps_med / LOC_REF_FPS, 3)),
            'projected_basis': 'measured_wall_per_roll',
            'wall_s_per_roll_median': (None if wall_med is None else round(wall_med, 1)),
            'projected_total_h': (None if wall_med is None else
                                  round(wall_med * n_plan / 3600, 2)),
            'projected_remaining_h': (None if wall_med is None else
                                      round(wall_med * max(n_plan - n_done, 0) / 3600, 2)),
            'budget_remaining_h': round(budget_left, 2),
            'rolls_affordable': (None if not wall_med else
                                 int(max(0.0, budget_left) * 3600 / wall_med)),
            'in_map_controls': [{k: r.get(k) for k in keys}
                                for r in per_roll if r.get('in_map')],
            'rolls': [r['roll'] for r in rolls]}

## `loc-resume`

Load plan.json, rewind the cursor to the first unbanked roll, enforce the budget, prefetch the next video.

In [ ]:
# === cell: loc-resume ===
def loc_load_plan(path=None, seeds=None):
    """The work list, plus the per-roll intrinsics seed.

    The seed is a separate staged file because it is derived from data/virb_meta/videos.json
    and the srs database, neither of which exists on Kaggle; b2_stage_inputs.py resolves it
    locally and the plan rows carry only what b1_plan.py owns.
    """
    path = Path(path or os.environ.get('SRS_LOC_PLAN') or (LOC_IN / 'plan.json'))
    plan = json.loads(path.read_text())
    rolls = plan['rolls'] if isinstance(plan, dict) else plan
    meta = (plan.get('meta') or {}) if isinstance(plan, dict) else {}
    if meta.get('superseded'):
        raise RuntimeError(
            f'plan.json declares itself SUPERSEDED and says consumers must refuse it: '
            f'{meta.get("superseded_reason")!r}. Regenerate it with b1_plan.py.')
    if meta.get('time_base') not in LOC_TIME_BASE:
        raise RuntimeError(
            f'plan.json declares time_base={meta.get("time_base")!r}, this kernel consumes '
            f'{LOC_TIME_BASE!r}. start_ms/end_ms are used VERBATIM as offsets on the video '
            f'file\'s own clock and nothing here converts between conventions, so a plan '
            f'written under another time base would sample every roll from the wrong place. '
            f'Refusing to run; regenerate the plan or set SRS_LOC_TIME_BASE deliberately.')
    sp = Path(seeds or (LOC_IN / 'intrinsics_seed.json'))
    seed = json.loads(sp.read_text()) if sp.exists() else {}
    for r in rolls:
        r['roll'], r['file_id'] = int(r['roll']), int(r['file_id'])
        r['in_map'] = bool(r.get('in_map', r['roll'] in LOC_IN_MAP_ROLLS))
        s = seed.get(str(r['file_id'])) or {}
        for k in ('lens_correction', 'stabilizer', 'seed_focal', 'seed_k1'):
            r.setdefault(k, s.get(k))
    return rolls, (plan if isinstance(plan, dict) else {'rolls': rolls})


def loc_resume_cursor(rolls, st):
    """Rewind to the first roll with NO output on the remote, rather than trusting the cursor.

    The cursor is advanced only after both output objects are banked, so a cursor ahead of the
    outputs means a crash landed between the two writes; trusting it would silently skip a roll.
    """
    names = loc_lsf(LOC_OUT_REMOTE)
    have = {n[:-4] for n in names if n.endswith('.npz') and not n.endswith('_corr.npz')}
    # loc_lsf returns [] for a FAILED listing as well as an empty one; here that would rewind the
    # cursor to 0 and re-run every banked roll, so a nonzero cursor means the listing lied.
    if not names and int(st.get('cursor', 0)) > 0:
        raise RuntimeError(
            f'the output listing came back empty but state.json says cursor '
            f'{st.get("cursor")}. Trusting it would re-run every banked roll; refusing. '
            f'Re-run once {LOC_OUT_REMOTE} lists again.')
    done = [i for i, r in enumerate(rolls) if str(r['roll']) in have]
    st['done'] = sorted({int(rolls[i]['roll']) for i in done})
    cur = 0
    while cur < len(rolls) and str(rolls[cur]['roll']) in have:
        cur += 1
    if cur != int(st.get('cursor', 0)):
        loc_log(f'cursor rewound {st.get("cursor")} -> {cur} (outputs present for '
                f'{len(have)} rolls)')
    st['cursor'] = cur
    return cur, have


def loc_budget_stop(st, est_s):
    """Refuse to START a roll that would breach either limit.  Returns a reason or None."""
    if loc_session_min() + est_s / 60.0 > LOC_SESSION_MAX_MIN:
        return (f'session cap: {loc_session_min():.1f} min used + {est_s / 60.0:.1f} min '
                f'estimated > {LOC_SESSION_MAX_MIN} min (12 h Kaggle cap)')
    total_h = (LOC_BASE_S[0] + (time.time() - LOC_T0) + est_s) / 3600.0
    if total_h > LOC_BUDGET_H:
        return (f'budget cap: {total_h:.2f} h of cumulative session wall-clock would exceed '
                f'the {LOC_BUDGET_H} h budget')
    return None


def loc_fetch_video(row, dest_dir):
    """Bring one roll's video local.  In DRY mode the plan's local_path is used in place."""
    dest_dir = Path(dest_dir)
    dest_dir.mkdir(parents=True, exist_ok=True)
    if LOC_DRY:
        p = Path(row['local_path'])
        if not p.exists():
            raise FileNotFoundError(p)
        return p, 0.0
    dst = dest_dir / f'{row["roll"]}{Path(row["remote"]).suffix or ".mp4"}'
    if dst.exists():
        return dst, 0.0
    t0 = time.time()
    loc_sh(f'rclone copyto "{row["remote"]}" "{dst}"')
    return dst, time.time() - t0


class LocPrefetch:
    """Fetch roll N+1 while roll N computes.

    Kaggle<->Drive measured at 96-114 MB/s (tmp/addmatch/add.log: 6.3 GiB in 67 s), so a p50
    318 MB roll is ~3.3 s against ~62 s of GPU and the fetch disappears entirely behind it.
    Each video is deleted after its decode; the largest single roll is 1.09 GB against ~20 GB
    of working disk.
    """

    def __init__(self, dest_dir):
        self.dir = Path(dest_dir)
        self.pending = {}

    def start(self, row):
        if row is None or row['roll'] in self.pending:
            return
        box = {}
        self.pending[row['roll']] = box

        def go():
            try:
                box['path'], box['s'] = loc_fetch_video(row, self.dir)
            except Exception as e:                              # noqa: BLE001
                box['error'] = f'{type(e).__name__}: {e}'
        box['th'] = threading.Thread(target=go, daemon=True)
        box['th'].start()

    def get(self, row):
        """(path, download seconds, WAIT seconds).

        The two numbers are different things and conflating them hid the question of whether
        the prefetch was working at all.  `box['s']` is how long the download itself took, which
        is what the per-roll line has always printed as `fetch Ns`; the wait is how long the
        caller actually blocked, and on a prefetch that is a roll ahead it should be ~0.  Only
        the second belongs in a critical-path budget.
        """
        self.start(row)
        box = self.pending.pop(row['roll'])
        t0 = time.time()
        box['th'].join()
        wait = time.time() - t0
        LOC_IO['fetch_bg_s'] += box.get('s') or 0.0
        LOC_IO['fetch_block_s'] += wait
        if 'error' in box:
            raise RuntimeError(f'fetch {row["roll"]}: {box["error"]}')
        return box['path'], box['s'], wait

    def cancel(self, roll):
        """Join and delete a fetch whose roll will not be processed.  Never raises.

        The join is not optional: the thread is already writing the file, so unlinking without
        it would leave the download to recreate what was just removed.
        """
        box = self.pending.pop(roll, None)
        if box is None:
            return
        try:
            box['th'].join()
            if box.get('path'):
                self.drop(box['path'])
        except Exception:                                       # noqa: BLE001
            pass

    def drop(self, path):
        try:
            if not LOC_DRY and Path(path).exists():
                Path(path).unlink()
        except OSError:
            pass

## `loc-split`

The GPU<->worker protocol for both channels, and the loop CPU sessions run instead of the roll loop.

In [ ]:
# === cell: loc-split ===
# The bootstrap is the only stage that moves.  It is 131-153 s of pure CPU inside a session
# whose scarce resource is a pair of T4s, it needs nothing from the GPU, and its whole result is
# four arrays of length n_anchor -- under 20 kB.  Everything below is the machinery for handing
# it to a CPU-only Kaggle session and getting that back, and every piece of it exists to make
# ONE of two things true: either the answer is the one this roll asked for, or the GPU does the
# work itself.  There is no third outcome in which a roll finishes with someone else's arcs.
#
# `loc_bootstrap` (gate -> smooth -> interpolate) deliberately does NOT move: it is microseconds
# of numpy over the returned arrays and shipping it would only add a way for the two sides to
# disagree about what the arrays mean.


def loc_split_active():
    """Is this session the GPU half of a live split?"""
    return LOC_SPLIT and loc_role() == 'gpu'


def loc_file_sha(path, n=16):
    """sha256 of a file's BYTES, truncated.  The bytes, not the arrays, on purpose.

    This hash is the whole answer to the silent-wrong-answer failure: a stale or crossed result
    does not crash, it places every 30 m match window at the wrong arc and the roll comes out
    plausible and wrong.  Binding the answer to the exact bytes of the question -- and namespacing
    both under run_id -- means a mismatched pair cannot even be found, let alone consumed.
    Hashing the transferred file also makes it a transfer check for free.
    """
    h = hashlib.sha256()
    with open(path, 'rb') as f:
        for chunk in iter(lambda: f.read(1 << 22), b''):
            h.update(chunk)
    return h.hexdigest()[:n]


def loc_req_name(roll, sha, t=None):
    """`{run_id}__{roll}__{posted_epoch}__{sha}.npz` -- four facts, and a listing is enough.

    run_id namespaces the whole handoff, so a previous push's leftovers cannot be mistaken for
    live work; the epoch makes the worker's queue FIFO without a read; the sha is the binding
    between question and answer.  Everything a worker needs to choose and verify a request is in
    its name, which is what keeps the polling loop at one `rclone lsf` per remote.
    """
    return f'{LOC_RUN_ID or loc_run_id()}__{int(roll)}__{int(t or time.time())}__{sha}.npz'


def loc_req_parse(name):
    """(run_id, roll, posted_epoch, sha), or None when it is not one of ours."""
    m = re.match(r'^(.+)__(\d+)__(\d+)__([0-9a-f]{8,})\.npz$', str(name).strip('/'))
    return (m.group(1), int(m.group(2)), int(m.group(3)), m.group(4)) if m else None


def loc_claim_name(req, worker=None, t=None):
    """Claims carry their timestamp IN THE NAME so that liveness costs one listing and no reads.

    rclone reports modification times only with an extra flag and a Drive round trip per object,
    and a claim is read by the OTHER worker on every poll; putting the epoch in the name makes
    "is this claim stale?" a string operation on a list we already have.
    """
    return f'{req}__{worker or loc_session_id()}__{int(t or time.time())}.claim'


def loc_claim_parse(name):
    m = re.match(r'^(.+\.npz)__([^_]+)__(\d+)\.claim$', str(name).strip('/'))
    return (m.group(1), m.group(2), int(m.group(3))) if m else None


def loc_map_fingerprint(M):
    """Enough of the map's identity that two sessions cannot disagree about it silently.

    Both sides attach the same Kaggle dataset, so this should never fire -- which is exactly why
    it is asserted rather than assumed.  A worker searching a different index would return arcs
    that are wrong in a way no downstream stage can detect.
    """
    return {'n_obs': int(M.n_obs), 'n_img': int(M.n_img),
            'ntotal': int(getattr(M.index, 'ntotal', -1)), 'nprobe': LOC_NPROBE,
            'knn': LOC_KNN, 'gate': LOC_BOOT_GATE, 'stride': LOC_BOOT_STRIDE,
            'ratio': LOC_RATIO, 'max_error_px': LOC_PNP_MAX_ERROR_PX}


def loc_boot_payload(row, feats, M, dest):
    """Write ONE object holding every anchor this roll's bootstrap needs.  Returns its sha.

    One object, uncompressed, on purpose.  Per-anchor files land in the 1.4-1.8 MiB/s
    many-small-files regime measured on this Drive; one 110-270 MB object goes at 90-124 MB/s.
    And `savez_compressed` is refused deliberately: these are dense float descriptors, deflate
    wins almost nothing on them, and it would spend the one resource the CPU session has least
    of on both ends of the wire.
    """
    idx = [i for i in sorted(feats['desc32']) if i % LOC_BOOT_STRIDE == 0]
    seed = loc_seed_intrinsics(row)
    z = {'anchor_idx': np.asarray(idx, np.int64),
         'seed': np.asarray(seed[:2], np.float64),
         'meta': np.asarray(json.dumps({
             'roll': int(row['roll']), 'run_id': LOC_RUN_ID or loc_run_id(),
             'stride': LOC_BOOT_STRIDE, 'n_anchor': len(idx),
             'map': loc_map_fingerprint(M),
             'utc': time.strftime('%Y-%m-%dT%H:%M:%SZ', time.gmtime())}))}
    for i in idx:
        z[f'd{i}'] = np.ascontiguousarray(feats['desc32'][i], np.float32)
        z[f'k{i}'] = np.ascontiguousarray(feats['kp'][i], np.float32)
    np.savez(dest, **z)
    return loc_file_sha(dest), len(idx)


def loc_boot_unpack(path):
    """The worker's side of the payload: a `feats`-shaped dict the real bootstrap can consume."""
    z = np.load(path, allow_pickle=False)
    meta = json.loads(str(z['meta']))
    idx = [int(i) for i in z['anchor_idx']]
    feats = {'desc32': {i: np.ascontiguousarray(z[f'd{i}'], np.float32) for i in idx},
             'kp': {i: np.asarray(z[f'k{i}']) for i in idx}}
    return feats, tuple(float(x) for x in z['seed']), meta


def loc_boot_post(row, feats, M):
    """GPU side: publish this roll's anchors and return the request record, or None.

    Called from the extract phase -- the earliest instant the anchors exist -- so the worker gets
    the whole of the previous roll's CPU stages as a head start.  Every failure here is a
    non-event: the roll simply bootstraps locally, which is what an unsplit session does anyway.

    THE UPLOAD RUNS ON A BACKGROUND THREAD; the serialisation does not.  Splitting it there is
    what lets the request NAME -- which is the sha of the bytes -- be known immediately, so the
    caller has a complete request record to carry while ~200 MB goes up behind it.  Nothing on
    the GPU side needs the upload to have COMPLETED except the wait for its answer, and
    `loc_post_join` is called there and nowhere else.
    """
    if not loc_split_active():
        return None
    if LOC_BREAKER['skip_left'] > 0:
        LOC_BREAKER['skip_left'] -= 1
        return {'skipped': 'circuit breaker open', 'breaker': True,
                'breaker_left': LOC_BREAKER['skip_left']}
    if not loc_worker_alive():
        return {'skipped': 'no live CPU worker heartbeat', 'no_worker': True}
    t0 = time.time()
    p = LOC_ROOT / f'_anchors_{int(row["roll"])}.npz'
    try:
        sha, n_anchor = loc_boot_payload(row, feats, M, p)
        name = loc_req_name(row['roll'], sha)
        rec = {'name': name, 'sha': sha, 'roll': int(row['roll']), 'n_anchor': n_anchor,
               'bytes': int(p.stat().st_size), 'payload_s': round(time.time() - t0, 1),
               'posted_t': time.time(), 'async': bool(LOC_POST_ASYNC)}
        loc_boot_post_submit(p, name, rec)
        loc_log(f'  {row["roll"]}: bootstrap request {"queued" if LOC_POST_ASYNC else "posted"}, '
                f'{n_anchor} anchors, {rec["bytes"] / 2 ** 20:.0f} MB serialised in '
                f'{rec["payload_s"]}s -> {name}')
        return rec
    except Exception as e:                                      # noqa: BLE001
        p.unlink(missing_ok=True)
        loc_log(f'  {row["roll"]}: could not post the bootstrap request '
                f'({type(e).__name__}: {str(e)[:200]}); this roll bootstraps locally')
        return {'skipped': f'{type(e).__name__}: {str(e)[:120]}'}


def loc_boot_post_submit(path, name, rec):
    """Hand one already-serialised request object to the uploader.  Bounded, never unbounded.

    The bound is the point: if Drive slows to below one roll per roll, an unbounded queue would
    keep ~200 MB temp files on /tmp and hide the slowdown until the disk filled.  Waiting for the
    oldest upload instead makes the slowdown visible as blocking seconds in the per-roll I/O
    line, which is where a reader can act on it.
    """
    def _upload_anchors():
        t0 = time.time()
        try:
            loc_push(path, name, LOC_ANCHOR_REMOTE)
            rec['post_s'] = round(time.time() - t0, 1)
        except Exception as e:                                  # noqa: BLE001
            rec['post_error'] = f'{type(e).__name__}: {str(e)[:160]}'
        finally:
            Path(path).unlink(missing_ok=True)
            LOC_IO['post_bg_s'] += time.time() - t0
    if not LOC_POST_ASYNC:
        _upload_anchors()
        return
    while len(LOC_POST_FUT) >= max(1, LOC_POST_MAX_INFLIGHT):
        old = next(iter(LOC_POST_FUT))
        loc_post_join({'name': old}, why='the upload queue is full')
    LOC_POST_FUT[name] = loc_bg('post').submit(_upload_anchors)


def loc_post_join(req, why=''):
    """Block until THIS request's bytes are actually on the remote.  Idempotent, both channels.

    Called immediately before the wait for its answer, and nowhere earlier: a worker cannot
    answer a question that has not landed, so entering the poll loop with the upload still in
    flight would spend the wait clock on our own transfer.  It is also called before a request
    is abandoned, so the delete cannot race the upload and leave a ~200 MB orphan.
    """
    name = (req or {}).get('name')
    fut = LOC_POST_FUT.pop(name, None) if name else None
    if fut is None:
        return 0.0
    t0 = time.time()
    try:
        fut.result()
    except Exception as e:                                      # noqa: BLE001
        req['post_error'] = f'{type(e).__name__}: {str(e)[:160]}'
    dt = time.time() - t0
    LOC_IO['post_block_s'] += dt
    if req is not None:
        req['post_block_s'] = round(dt, 1)
    if dt > 1.0 and why:
        loc_log(f'  waited {dt:.1f}s on the anchor upload for {name} ({why})')
    return dt


def loc_boot_collect(req, gpu_has_work=None):
    """GPU side: wait for THIS request's answer.  Returns the four arrays, or None to go local.

    The wait clock is armed only once `gpu_has_work()` says this session has run out of work of
    its own.  That distinction is the difference between a split that pays and one that does
    not: while the look-ahead is extracting the next roll the session is doing exactly what it
    would have been doing anyway, so waiting is free; the moment that finishes, every further
    second is billed idle and is counted as such in `boot_stall_s`.
    """
    if not (req and req.get('name')):
        return None, 'no request'
    t_start, idle_t0 = time.time(), None
    while True:
        try:
            have = set(loc_lsf(LOC_BOOT_REMOTE))
        except Exception:                                       # noqa: BLE001
            have = set()
        if req['name'] in have:
            p = LOC_ROOT / f'_boot_{req["roll"]}.npz'
            p.unlink(missing_ok=True)
            if loc_pull(req['name'], p, LOC_BOOT_REMOTE) and p.exists():
                try:
                    z = np.load(p, allow_pickle=False)
                    assert str(z['req_sha']) == req['sha'], (
                        f'result {req["name"]} carries req_sha {str(z["req_sha"])!r}, '
                        f'not {req["sha"]!r}')
                    out = (np.asarray(z['anchor_idx'], np.int64),
                           np.asarray(z['arcs'], np.float64),
                           np.asarray(z['oks'], np.int8), np.asarray(z['inls'], np.int32))
                    LOC_WORKERS_SEEN['last_result_t'] = time.time()
                    # BILLED IDLE IS COUNTED ON THE ANSWER THAT ARRIVES TOO.  It used to be
                    # counted only on a timeout, so `boot_stall_s` -- the number the monitor
                    # calls "billed idle waiting for a worker" and the number a depth decision
                    # is made on -- silently omitted every second spent idle before an answer
                    # that did come.  The live run's 77.0 s and 44.5 s were exactly those
                    # seconds: real, billed, and invisible to the total that was supposed to
                    # report them.
                    idle_s = 0.0 if idle_t0 is None else time.time() - idle_t0
                    LOC_BOOT_STALL_S[0] += idle_s
                    return out, {'wait_s': round(time.time() - t_start, 1),
                                 'idle_s': round(idle_s, 1),
                                 'worker': str(z['worker']),
                                 'worker_s': float(z['boot_s'])}
                except Exception as e:                          # noqa: BLE001
                    loc_log(f'  {req["roll"]}: remote bootstrap result unusable '
                            f'({type(e).__name__}: {str(e)[:200]}); doing it locally')
                    return None, 'unusable result'
                finally:
                    p.unlink(missing_ok=True)
        still_working = bool(gpu_has_work and gpu_has_work())
        if still_working:
            idle_t0 = None
        else:
            if idle_t0 is None:
                idle_t0 = time.time()
                loc_log(f'  {req["roll"]}: the GPU has run out of its own work; the bootstrap '
                        f'wait clock is now armed ({LOC_BOOT_WAIT_S:.0f}s) and every second '
                        f'from here is billed idle')
            if time.time() - idle_t0 >= LOC_BOOT_WAIT_S:
                LOC_BOOT_STALL_S[0] += time.time() - idle_t0
                return None, f'timeout after {time.time() - idle_t0:.0f}s idle'
        if time.time() - t_start >= LOC_BOOT_WAIT_MAX_S:
            if idle_t0 is not None:
                LOC_BOOT_STALL_S[0] += time.time() - idle_t0
            return None, f'timeout at the hard ceiling of {LOC_BOOT_WAIT_MAX_S:.0f}s'
        time.sleep(LOC_BOOT_POLL_S)


def loc_boot_done(req, source):
    """Retire one request: count the source, move the breaker, and delete BOTH objects.

    Deleting is not housekeeping, it is a requirement.  At a 202 MB mean payload the 1,110-roll
    plan is 224 GB of anchors; keeping them would fill the Drive long before the run finished.
    Both sides delete, and `loc_rm` treats "already gone" as success, so the pair being removed
    twice is the normal case rather than an error.
    """
    loc_breaker_move(LOC_BOOT_SOURCE, LOC_BREAKER, source, 'bootstraps')
    if req and req.get('name'):
        loc_post_join(req, why='retiring the request')
        loc_rm_async([(req['name'], LOC_ANCHOR_REMOTE), (req['name'], LOC_BOOT_REMOTE)])


def loc_breaker_move(counts, breaker, source, what):
    """One channel's source counter and circuit breaker.  Shared by both channels verbatim."""
    counts[source] = counts.get(source, 0) + 1
    if source == 'remote':
        breaker['consec'] = 0
    elif source.startswith('local_') and source != 'local_breaker':
        breaker['consec'] += 1
        if breaker['consec'] >= LOC_BOOT_BREAK_AFTER:
            breaker['consec'] = 0
            breaker['skip_left'] = LOC_BOOT_BREAK_ROLLS
            breaker['opened'] += 1
            loc_log(f'*** CIRCUIT BREAKER OPEN: {LOC_BOOT_BREAK_AFTER} {what} in a row fell '
                    f'back locally, so the next {LOC_BOOT_BREAK_ROLLS} rolls do not pay the '
                    f'upload or the wait at all. One request is posted after that to re-probe, '
                    f'so a worker that comes back is picked up on its own. ***', push_=True)


def loc_boot_abandon(req):
    """Drop a request whose roll will never consume it, WITHOUT moving the breaker.

    A roll that fails for its own reasons -- a video that will not decode, no anchor past the
    gate -- says nothing about whether the split is working, so it must not be counted as a
    fallback.  The objects still have to go: at 202 MB each they are the storage problem, and
    the upload is JOINED before the delete so the two cannot cross.
    """
    if req and req.get('name'):
        loc_post_join(req, why='abandoning the request')
        loc_rm_async([(req['name'], LOC_ANCHOR_REMOTE), (req['name'], LOC_BOOT_REMOTE)])


# ---- the second channel: intrinsics + PnP -------------------------------------------------
# A SECOND CHANNEL, NOT A SECOND PROTOCOL.  Everything about how a question is named, bound to
# its answer, claimed, retired and deleted is the bootstrap's, reused verbatim; only the payload
# and the stage differ.  What is added is one assertion the bootstrap does not need: the answer
# is a POSE, and a pose is what the in-map controls are computed from, so the two sides must
# agree about the solver as exactly as they already agree about the map.
def loc_solver_fingerprint():
    """The solver's identity: pycolmap's build and every option that shapes a solve.

    Read off the option objects rather than restated, so it cannot drift away from what
    `loc_pnp_options` actually sets.  ransac.random_seed = 0 and ransac.num_threads = 1 are the
    two that make a worker's PnP bit-identical to this session's, and they are in here for that
    reason; a mismatch means the answer would differ and the request is refused rather than
    silently answered by a different solver.
    """
    import pycolmap
    est, ref = loc_pnp_options(0)
    est1, ref1 = loc_pnp_options(0, refine_focal=True, refine_extra=True)
    r = est.ransac
    return {'pycolmap': str(getattr(pycolmap, '__version__', '?')),
            'ransac': [float(r.max_error), float(r.min_inlier_ratio), float(r.confidence),
                       int(r.min_num_trials), int(r.max_num_trials),
                       int(r.random_seed), int(r.num_threads)],
            'estimate_focal_length': bool(est.estimate_focal_length),
            'refine': [bool(ref.refine_focal_length), bool(ref.refine_extra_params)],
            'refine_pass1': [bool(ref1.refine_focal_length), bool(ref1.refine_extra_params)],
            'min_corrs': LOC_MIN_CORRS, 'wh': [LOC_W, LOC_H], 'pp': list(LOC_PP),
            'intr': [LOC_INTR_STRIDE, LOC_INTR_MIN_INL, LOC_INTR_MIN_FRAMES,
                     LOC_INTR_MIN_FALLBACK], 'seed_default': list(LOC_SEED_DEFAULT)}


def loc_tail_solve(M, corr, seed, hb=None, out=None):
    """The roll's tail: the two-pass intrinsics fit, then PnP on every frame.

    THE ONE COPY.  The GPU calls it when it does the tail itself and the worker calls it when it
    does the tail instead, so `tail_source` can only ever be a note about where the time went.
    Everything that could make the two disagree is asserted here rather than assumed: a fixed
    RANSAC seed and one RANSAC thread are what make a pool of any width give the serial answer,
    and `loc_pmap` preserves order, so neither side's core count can reach the numbers.
    """
    n = len(corr)
    t0 = time.time()
    LOC_PROF_TAG[0] = 'intr.'
    intr = loc_fit_intrinsics(M, corr, seed)
    LOC_PROF_TAG[0] = ''
    intr_s = round(time.time() - t0, 2)
    t0 = time.time()
    est, ref = loc_pnp_options(0)
    assert int(est.ransac.random_seed) == 0 and int(est.ransac.num_threads) == 1, (
        f'PnP is not deterministic on this machine: random_seed='
        f'{est.ransac.random_seed} num_threads={est.ransac.num_threads}. A pose computed here '
        f'would not be the one the other half of the split computes, and the in-map controls '
        f'are read off these poses.')

    def solve(i):
        p2d, p3d, pos, ratio, rows = corr[i]
        cam = loc_camera(intr['focal'], intr['k1'])     # fresh: refinement is written in place
        arcs = M.pt_arc_sorted[pos] if len(pos) else None
        r = loc_pnp(p2d, p3d, cam, est, ref, M, arcs=arcs, rows=rows, cov=True)
        r['corr'] = (p2d, pos, ratio)
        return r

    # chunked so the heartbeat still comes from the MAIN thread: a heartbeat writes and pushes
    # a file, and it has no business being raced by every worker in the pool
    per = []
    for a in range(0, n, LOC_PNP_CHUNK):
        if hb is not None:
            hb(a, n)
        per += loc_pmap(solve, range(a, min(n, a + LOC_PNP_CHUNK)), tag='pnp.')
    rec = {'intrinsics_s': intr_s, 'pnp_s': round(time.time() - t0, 2)}
    if out is not None:
        out.update(rec)
    return intr, per, rec


def loc_tail_payload(row, corr, seed, M, dest):
    """One object holding every correspondence the tail needs.  Returns (sha, n_frames, bytes).

    Three arrays and an offset table, and each column is the SMALLEST type that is still exact:

      kp_xy  float32  the keypoints are float32 out of the extractor and only widened to float64
                      at the match's boundary, so float32 round-trips them exactly -- which is
                      checked here per roll rather than trusted, because a silent narrowing
                      would move a pose and the controls read poses.
      pos    int32    an index into a 12.4M-row bank
      rows   int32    the same

    `p3d` is NOT shipped: it is M.pt_xyz[M.pt_order[pos]] and the worker has the same map, so
    sending it would be 24 B/correspondence of a quantity the receiver can derive exactly.  At a
    measured 618-963 correspondences/frame that is the difference between ~24 MB and ~50 MB a
    roll, on the link that is the whole reason this is hard.
    """
    order = sorted(corr)
    assert order == list(range(len(order))), 'the correspondence keys are not 0..n-1'
    off = np.zeros(len(order) + 1, np.int64)
    kp, pos, rows = [], [], []
    for j, i in enumerate(order):
        p2d, _p3d, ps, _ratio, rw = corr[i]
        ps, rw = np.asarray(ps), np.asarray(rw)
        p2d = np.asarray(p2d, np.float64)
        k32 = p2d.astype(np.float32)
        assert np.array_equal(k32.astype(np.float64), p2d), (
            f'frame {i}: the keypoints do not survive float32, so shipping them narrowed would '
            f'change the pose. Refusing to post this roll.')
        assert len(ps) == len(rw) == len(p2d)
        assert not len(ps) or (int(ps.max()) < 2 ** 31 and int(rw.max()) < 2 ** 31)
        kp.append(k32)
        pos.append(ps.astype(np.int32))
        rows.append(rw.astype(np.int32))
        off[j + 1] = off[j] + len(ps)
    cat = lambda xs, dt, w: (np.concatenate(xs) if xs else                        # noqa: E731
                             np.zeros((0, w) if w else 0, dt))
    z = {'off': off, 'kp_xy': cat(kp, np.float32, 2), 'pos': cat(pos, np.int32, 0),
         'rows': cat(rows, np.int32, 0),
         'meta': np.asarray(json.dumps({
             'roll': int(row['roll']), 'run_id': LOC_RUN_ID or loc_run_id(),
             'n_frames': len(order), 'seed': [float(seed[0]), float(seed[1]), str(seed[2])],
             'map': loc_map_fingerprint(M), 'solver': loc_solver_fingerprint(),
             'utc': time.strftime('%Y-%m-%dT%H:%M:%SZ', time.gmtime())}))}
    np.savez(dest, **z)
    return loc_file_sha(dest), len(order), int(Path(dest).stat().st_size)


def loc_tail_unpack(path, M):
    """The worker's side: the `corr` dict loc_tail_solve consumes, rebuilt exactly.

    `p3d` is re-derived from `pos` through the same two lookups the GPU used, and `ratio` is not
    reconstructed at all -- the solve never reads it; it exists only for the record the GPU
    writes, and the GPU still has its own copy.
    """
    z = np.load(path, allow_pickle=False)
    meta = json.loads(str(z['meta']))
    off = np.asarray(z['off'], np.int64)
    kp, pos, rows = z['kp_xy'], z['pos'], z['rows']
    corr = {}
    for i in range(len(off) - 1):
        a, b = int(off[i]), int(off[i + 1])
        p = np.asarray(pos[a:b], np.int64)
        corr[i] = (np.ascontiguousarray(kp[a:b], np.float64),
                   M.pt_xyz[M.pt_order[p]], p, np.zeros(b - a),
                   np.asarray(rows[a:b], np.int64))
    return corr, tuple(meta['seed'][:2]), meta


def loc_tail_pack_result(per, intr, sha, rec):
    """`per` as columns.  Every float stays float64: `loc_record` is what narrows them."""
    n = len(per)
    nan3, nan4 = [np.nan] * 3, [np.nan] * 4
    z = {'req_sha': np.asarray(sha), 'worker': np.asarray(loc_session_id()),
         'intr': np.asarray(json.dumps(intr, default=loc_jsonable)),
         'tail_s': np.asarray(rec.get('tail_s', 0.0), np.float64),
         'intrinsics_s': np.asarray(rec.get('intrinsics_s', 0.0), np.float64),
         'pnp_s': np.asarray(rec.get('pnp_s', 0.0), np.float64),
         'centre': np.array([p['centre'] if p['centre'] is not None else nan3 for p in per],
                            np.float64).reshape(n, 3),
         'quat': np.array([p['quat'] if p['quat'] is not None else nan4 for p in per],
                          np.float64).reshape(n, 4),
         'cov': np.array([p['cov'] if p['cov'] is not None else np.full((6, 6), np.nan)
                          for p in per], np.float64).reshape(n, 6, 6),
         'cov_ok': np.array([p['cov'] is not None for p in per], np.uint8)}
    for k, dt in LOC_TAIL_COLS:
        d = 0 if dt.__name__.startswith(('int', 'uint')) else np.nan
        z[k] = np.array([d if p.get(k) is None else p[k] for p in per], dt)
    im_off = np.zeros(n + 1, np.int64)
    ims = []
    for i, p in enumerate(per):
        m = p.get('inlier_mask')
        ims.append(np.zeros(0, np.uint8) if m is None else np.asarray(m).astype(np.uint8))
        im_off[i + 1] = im_off[i] + len(ims[-1])
    z['im_off'] = im_off
    z['im_ok'] = np.array([p.get('inlier_mask') is not None for p in per], np.uint8)
    z['im'] = np.concatenate(ims) if ims else np.zeros(0, np.uint8)
    return z


def loc_tail_unpack_result(z, corr):
    """Columns back to the `per` list, with `corr` re-attached from OUR OWN copy.

    The correspondences never come back over the wire.  They are the biggest thing in the roll,
    this session already holds them, and `loc_record_corr` reads them off `per` -- so
    reconstructing them locally is both cheaper and one fewer thing that could differ.
    """
    # every column is materialised ONCE.  `z` is an NpzFile and each subscript decompresses the
    # whole member, so reading z['centre'][i] inside the loop would decode a 1,700-row array
    # per frame per column -- ~19,000 decompressions for one roll.
    cols = {k: np.asarray(z[k]) for k, _ in LOC_TAIL_COLS}
    centre, quat, cov = np.asarray(z['centre']), np.asarray(z['quat']), np.asarray(z['cov'])
    cov_ok, im_ok = np.asarray(z['cov_ok']), np.asarray(z['im_ok'])
    im_off, im = np.asarray(z['im_off']), np.asarray(z['im'])
    n = len(cols['ok'])
    per = []
    for i in range(n):
        p = {k: (int(cols[k][i]) if cols[k].dtype.kind in 'iu' else float(cols[k][i]))
             for k, _ in LOC_TAIL_COLS}
        c = np.asarray(centre[i], np.float64)
        q = np.asarray(quat[i], np.float64)
        p['centre'] = None if np.isnan(c).all() else c
        p['quat'] = None if np.isnan(q).all() else q
        p['cov'] = np.asarray(cov[i], np.float64) if cov_ok[i] else None
        a, b = int(im_off[i]), int(im_off[i + 1])
        p['inlier_mask'] = (im[a:b].astype(bool) if im_ok[i] else None)
        p2d, _p3d, pos, ratio, _rows = corr[i]
        assert p['n_corr'] == len(p2d), (
            f'frame {i}: the answer reports {p["n_corr"]} correspondences, this roll has '
            f'{len(p2d)}. The result does not belong to this question.')
        p['corr'] = (p2d, pos, ratio)
        per.append(p)
    return per, json.loads(str(z['intr']))


def loc_tail_post(row, corr, seed, M):
    """GPU side: publish this roll's correspondences the instant they exist.

    Posted here and COLLECTED A ROLL LATER, which is the only arrangement in which it can pay:
    the worker box is the same 2 physical cores as this one, so it computes the tail in the same
    ~49 s and every second of transfer on top of that is loss unless it is hidden behind a whole
    roll.  A failure here is a non-event -- the roll does its own tail, exactly as an unsplit
    session does.
    """
    if not (loc_split_active() and LOC_TAIL_SPLIT):
        return None
    if LOC_TAIL_BREAKER['skip_left'] > 0:
        LOC_TAIL_BREAKER['skip_left'] -= 1
        return {'skipped': 'circuit breaker open', 'breaker': True,
                'breaker_left': LOC_TAIL_BREAKER['skip_left']}
    if not loc_worker_alive():
        return {'skipped': 'no live CPU worker heartbeat', 'no_worker': True}
    t0 = time.time()
    p = LOC_ROOT / f'_tail_{int(row["roll"])}.npz'
    try:
        sha, n_f, nb = loc_tail_payload(row, corr, seed, M, p)
        name = loc_req_name(row['roll'], sha)
        rec = {'name': name, 'sha': sha, 'roll': int(row['roll']), 'n_frames': n_f,
               'bytes': nb, 'payload_s': round(time.time() - t0, 1), 'posted_t': time.time()}
        loc_tail_post_submit(p, name, rec)
        loc_log(f'  {row["roll"]}: tail request queued, {n_f} frames, '
                f'{nb / 2 ** 20:.1f} MB serialised in {rec["payload_s"]}s -> {name}')
        return rec
    except Exception as e:                                      # noqa: BLE001
        p.unlink(missing_ok=True)
        loc_log(f'  {row["roll"]}: could not post the tail request ({type(e).__name__}: '
                f'{str(e)[:200]}); this roll does its own intrinsics and PnP')
        return {'skipped': f'{type(e).__name__}: {str(e)[:120]}'}


def loc_tail_post_submit(path, name, rec):
    def _upload_tail():
        t0 = time.time()
        try:
            loc_push(path, name, LOC_TAIL_REMOTE)
            rec['post_s'] = round(time.time() - t0, 1)
        except Exception as e:                                  # noqa: BLE001
            rec['post_error'] = f'{type(e).__name__}: {str(e)[:160]}'
        finally:
            Path(path).unlink(missing_ok=True)
            LOC_IO['post_bg_s'] += time.time() - t0
    if not LOC_POST_ASYNC:
        _upload_tail()
        return
    while len(LOC_POST_FUT) >= max(1, LOC_POST_MAX_INFLIGHT) + 1:
        loc_post_join({'name': next(iter(LOC_POST_FUT))}, why='the upload queue is full')
    LOC_POST_FUT[name] = loc_bg('post').submit(_upload_tail)


def loc_tail_collect(req, corr, gpu_has_work=None):
    """GPU side: wait for THIS roll's poses.  Returns (per, intr, why), or (None, None, why).

    Same wait clock as the bootstrap and for the same reason: waiting while this session still
    has GPU work of its own is free, and every second after that is billed and counted.  The
    idle timeout is shorter than the bootstrap's because the stage it stands in for is shorter.
    """
    if not (req and req.get('name')):
        return None, None, 'no request'
    t_start, idle_t0 = time.time(), None
    while True:
        try:
            have = set(loc_lsf(LOC_TAILRES_REMOTE))
        except Exception:                                       # noqa: BLE001
            have = set()
        if req['name'] in have:
            p = LOC_ROOT / f'_tailres_{req["roll"]}.npz'
            p.unlink(missing_ok=True)
            if loc_pull(req['name'], p, LOC_TAILRES_REMOTE) and p.exists():
                try:
                    z = np.load(p, allow_pickle=False)
                    assert str(z['req_sha']) == req['sha'], (
                        f'result {req["name"]} carries req_sha {str(z["req_sha"])!r}, '
                        f'not {req["sha"]!r}')
                    per, intr = loc_tail_unpack_result(z, corr)
                    LOC_WORKERS_SEEN['last_result_t'] = time.time()
                    # counted on the answer that arrives, exactly as the bootstrap's is, and for
                    # the same reason: the monitor adds the two channels together and calls the
                    # sum billed idle, so one channel omitting its successful waits would make
                    # the total mean neither one thing nor the other
                    idle_s = 0.0 if idle_t0 is None else time.time() - idle_t0
                    LOC_TAIL_STALL_S[0] += idle_s
                    return per, intr, {'wait_s': round(time.time() - t_start, 1),
                                       'idle_s': round(idle_s, 1),
                                       'worker': str(z['worker']),
                                       'worker_s': float(z['tail_s']),
                                       'intrinsics_s': float(z['intrinsics_s']),
                                       'pnp_s': float(z['pnp_s'])}
                except Exception as e:                          # noqa: BLE001
                    loc_log(f'  {req["roll"]}: remote tail result unusable '
                            f'({type(e).__name__}: {str(e)[:200]}); doing it locally')
                    return None, None, 'unusable result'
                finally:
                    p.unlink(missing_ok=True)
        if bool(gpu_has_work and gpu_has_work()):
            idle_t0 = None
        else:
            if idle_t0 is None:
                idle_t0 = time.time()
            if time.time() - idle_t0 >= LOC_TAIL_WAIT_S:
                LOC_TAIL_STALL_S[0] += time.time() - idle_t0
                return None, None, f'timeout after {time.time() - idle_t0:.0f}s idle'
        if time.time() - t_start >= LOC_TAIL_WAIT_MAX_S:
            if idle_t0 is not None:
                LOC_TAIL_STALL_S[0] += time.time() - idle_t0
            return None, None, f'timeout at the hard ceiling of {LOC_TAIL_WAIT_MAX_S:.0f}s'
        time.sleep(LOC_TAIL_POLL_S)


def loc_tail_done(req, source):
    loc_breaker_move(LOC_TAIL_SOURCE, LOC_TAIL_BREAKER, source, 'tails')
    if req and req.get('name'):
        loc_post_join(req, why='retiring the tail request')
        loc_rm_async([(req['name'], LOC_TAIL_REMOTE), (req['name'], LOC_TAILRES_REMOTE)])


def loc_tail_abandon(req):
    if req and req.get('name'):
        loc_post_join(req, why='abandoning the tail request')
        loc_rm_async([(req['name'], LOC_TAIL_REMOTE), (req['name'], LOC_TAILRES_REMOTE)])


def loc_worker_alive(force=False):
    """Is at least one CPU worker live?  Cached, and free whenever a result arrived recently.

    A result consumed inside LOC_WORKER_TRUST_S is proof of a live worker that cost nothing to
    observe, so the heartbeat files are only read when that evidence has gone stale -- and even
    then only the live ones, since the listing already carries every age.
    """
    if not LOC_SPLIT:
        return False
    now = time.time()
    if now - LOC_WORKERS_SEEN['last_result_t'] < LOC_WORKER_TRUST_S:
        return True
    if not force and (now - LOC_WORKERS_SEEN['t']) < LOC_WORKER_POLL_S * 4:
        return bool(LOC_WORKERS_SEEN['alive'])
    LOC_WORKERS_SEEN['t'] = now
    detail = []
    try:
        rows = loc_lsf_t(LOC_REMOTE)
        if not rows:
            raise RuntimeError('the remote listing came back empty')
        for t, n in rows:
            m = re.match(r'^heartbeat_cpu_(.+)\.json$', n)
            if not m:
                continue
            age, d = now - t, {}
            if age <= LOC_WORKER_STALE_S:
                p = LOC_ROOT / '_whb.json'
                p.unlink(missing_ok=True)
                if loc_pull(n, p, LOC_REMOTE) and p.exists():
                    d = json.loads(p.read_text())
            detail.append({'name': n, 'session_id': d.get('session_id') or m.group(1),
                           'age_s': round(age, 1), 'stage': d.get('stage'),
                           'min_to_cap': d.get('min_to_cap'), 'boot_done': d.get('boot_done'),
                           'tail_done': d.get('tail_done'), 'busy_s': d.get('busy_s'),
                           'busy_frac': d.get('busy_frac')})
    except Exception as e:                                      # noqa: BLE001
        # an unreadable heartbeat is not evidence of a dead worker; keep the previous answer
        loc_log(f'worker liveness check failed ({type(e).__name__}: {str(e)[:120]}); keeping '
                f'the previous answer {LOC_WORKERS_SEEN["alive"]}')
        return bool(LOC_WORKERS_SEEN['alive'])
    LOC_WORKERS_SEEN['detail'] = detail
    LOC_WORKERS_SEEN['beats'] = {d['session_id']: d['age_s'] for d in detail if d['session_id']}
    LOC_WORKERS_SEEN['beats_t'] = now
    LOC_WORKERS_SEEN['alive'] = any(d['age_s'] <= LOC_WORKER_STALE_S for d in detail)
    if detail and not LOC_WORKERS_SEEN['alive']:
        loc_log(f'no CPU worker heartbeat is fresher than {LOC_WORKER_STALE_S:.0f}s: {detail}. '
                f'The bootstrap runs locally until one comes back.')
    return bool(LOC_WORKERS_SEEN['alive'])


def loc_split_report():
    """Everything the operator needs about the split, in one dict, for state and heartbeat."""
    tot = sum(LOC_BOOT_SOURCE.values()) or 0
    return {'role': loc_role(), 'split': LOC_SPLIT, 'session_id': loc_session_id(),
            'boot_source': dict(sorted(LOC_BOOT_SOURCE.items())),
            'boot_remote_frac': (round(LOC_BOOT_SOURCE.get('remote', 0) / tot, 3)
                                 if tot else None),
            'boot_stall_s': round(LOC_BOOT_STALL_S[0], 1),
            'boot_stall_h': round(LOC_BOOT_STALL_S[0] / 3600.0, 3),
            'boot_stall_pct_budget': (round(100.0 * LOC_BOOT_STALL_S[0] / 3600.0
                                            / max(LOC_BUDGET_H, 1e-9), 2)),
            'breaker': dict(LOC_BREAKER), 'workers': LOC_WORKERS_SEEN['detail'],
            'workers_alive': LOC_WORKERS_SEEN['alive'],
            'ahead': dict(LOC_AHEAD_STATS),
            'tail_split': LOC_TAIL_SPLIT,
            'tail_source': dict(sorted(LOC_TAIL_SOURCE.items())),
            'tail_remote_frac': (round(LOC_TAIL_SOURCE.get('remote', 0)
                                       / sum(LOC_TAIL_SOURCE.values()), 3)
                                 if sum(LOC_TAIL_SOURCE.values()) else None),
            'tail_stall_s': round(LOC_TAIL_STALL_S[0], 1),
            'tail_breaker': dict(LOC_TAIL_BREAKER),
            'io': loc_io_report(),
            'match_devices': LOC_MATCH_DEVICES_EFF[0],
            'match_ab': LOC_MATCH_AB_STATE[0]}


def loc_io_report():
    """Overlapped vs blocking network seconds, cumulative and per roll.

    The whole point of moving the transfers off the main thread is invisible from the wall
    clock alone -- a session that got slower for another reason and one whose overlap silently
    stopped working look identical.  So both halves are counted: `_bg` is what the background
    thread spent moving bytes and `_block` is what the main thread waited for it, and the
    difference is what the change actually bought.
    """
    n = max(LOC_IO['n_rolls'], 1)
    out = {k: round(v, 1) for k, v in LOC_IO.items() if k != 'n_rolls'}
    out['n_rolls'] = LOC_IO['n_rolls']
    for k in ('post', 'bank', 'fetch'):
        bg, bl = LOC_IO[f'{k}_bg_s'], LOC_IO[f'{k}_block_s']
        out[f'{k}_overlapped_s'] = round(bg - bl, 1)
        out[f'{k}_overlapped_frac'] = round((bg - bl) / bg, 3) if bg > 0 else None
        out[f'{k}_blocking_s_per_roll'] = round(bl / n, 1)
    out['io_hidden_s_per_roll'] = round(
        sum(LOC_IO[f'{k}_bg_s'] - LOC_IO[f'{k}_block_s'] for k in ('post', 'bank', 'fetch'))
        / n, 1)
    return out


LOC_HB_HOOK[0] = loc_split_report


# ---- the CPU worker -----------------------------------------------------------------------
def loc_worker_stop():
    """The graceful stop, as a WORKER has to see it.  Returns the flag, or None.

    `stop.json` is renamed to `stop_honoured_<utc>.json` the moment the GPU session honours it,
    so that the next session does not stop before roll 1.  That rename is a race against a
    worker that happens to be 80 s into a bootstrap and has not polled: it would come back to an
    empty remote, see no flag, and sit out its full idle cap on a run the operator has stopped.

    So the worker accepts EITHER: the live flag, or an honoured stamp newer than its own session
    start.  The stamp's time is in its name, so this costs one listing and no reads -- and a
    stamp older than this session is somebody else's stop, correctly ignored.
    """
    d = loc_stop_requested(force=True, where='worker poll')
    if d:
        return d
    try:
        cut = time.strftime('%Y%m%dT%H%M%SZ', time.gmtime(LOC_T0))
        for n in loc_lsf(LOC_REMOTE):
            m = re.match(r'^stop_honoured_(\d{8}T\d{6}Z)\.json$', n)
            if m and m.group(1) >= cut:
                LOC_STOP['seen'] = {'stop': True, 'reason': f'honoured by the GPU session as {n}',
                                    'requested_utc': m.group(1)}
                LOC_STOP['where'] = 'worker poll (honoured stamp)'
                loc_log(f'*** the GPU session has already HONOURED a stop ({n}); this worker '
                        f'finishes here too rather than idling out its cap ***', push_=True)
                return LOC_STOP['seen']
    except Exception:                                           # noqa: BLE001
        pass
    return None


def loc_worker_beats(force=False):
    """{session_id: heartbeat age in seconds} for every worker, cached.  ONE listing, no reads.

    The id is in the name and the age is the modification time, so reading each file fetched
    what the listing already carried -- a round trip per heartbeat, and they accumulate.
    """
    now = time.time()
    if force or (now - LOC_WORKERS_SEEN['beats_t']) > LOC_WORKER_POLL_S * 2:
        beats, seen = {}, LOC_WORKERS_SEEN
        try:
            rows = loc_lsf_t(LOC_REMOTE)
            if not rows:
                return dict(seen['beats'])                      # unreadable != dead
            for t, n in rows:
                m = re.match(r'^heartbeat_cpu_(.+)\.json$', n)
                if m:
                    beats[m.group(1)] = now - t
        except Exception:                                       # noqa: BLE001
            return dict(seen['beats'])                          # unreadable != dead
        seen['beats'], seen['beats_t'] = beats, now
    return dict(LOC_WORKERS_SEEN['beats'])


def loc_claim_is_live(claim, beats, now=None):
    """Is a claim held by someone who is still alive?

    HEARTBEAT FIRST, age only as a backstop.  A claim whose owner heartbeat is fresh is live
    however old the claim is -- a 400 s roll is a slow roll, not a dead worker -- and a claim
    whose owner has been silent for LOC_CLAIM_BEAT_STALE_S is void however new the claim is.
    The age rule survives only for an owner that left no heartbeat at all to read.
    """
    now = time.time() if now is None else now
    t, worker, _ = claim
    age = beats.get(worker)
    if age is not None:
        return age <= LOC_CLAIM_BEAT_STALE_S
    return (now - t) <= LOC_CLAIM_TTL_S


def loc_worker_pick(seen=None, remote=None, done_remote=None, beats=None):
    """One unclaimed (or dead-owner-claimed) request, or None.  Two listings, no reads.

    Claims rather than a static shard because the shard's failure mode is the one that matters:
    a shard that loses its worker leaves half the rolls unanswered and the GPU pays a timeout on
    every one of them, while a claim whose owner dies is picked up by another worker.  What this
    function returns is a CANDIDATE, never a decision: `loc_worker_claim_confirmed` writes the
    claim and then re-reads, because listing-then-writing over a Drive with seconds of write
    visibility is a check-then-act race and the live run duplicated 81 of 135 requests through it.
    """
    remote = remote or LOC_ANCHOR_REMOTE
    done_remote = done_remote or LOC_BOOT_REMOTE
    run = LOC_RUN_ID or loc_run_id()
    reqs = [n for n in loc_lsf(remote) if n.endswith('.npz')]
    if not reqs:
        return None, {'n_req': 0}
    done = set(loc_lsf(done_remote))
    claims = loc_claims_by_req()
    now = time.time()
    beats = loc_worker_beats() if beats is None else beats
    mine, cand = loc_session_id(), []
    for n in sorted(reqs):
        p = loc_req_parse(n)
        if not p or p[0] != run or n in done or n in (seen or ()):
            continue
        held = claims.get(n) or []
        live = [c for c in held if c[1] != mine and loc_claim_is_live(c, beats, now)]
        # unclaimed first, then oldest posted: the GPU consumes in the order it asked
        cand.append((len(live) > 0, p[2], n, held))
    cand.sort()
    if not cand or cand[0][0]:
        return None, {'n_req': len(reqs), 'n_free': sum(1 for c in cand if not c[0]),
                      'n_claimed': sum(1 for c in cand if c[0])}
    _, _, name, held = cand[0]
    stolen = [c[2] for c in held if not loc_claim_is_live(c, beats, now)]
    return name, {'n_req': len(reqs), 'stolen_from': stolen,
                  'n_free': sum(1 for c in cand if not c[0])}


def loc_claims_by_req():
    """{request name: [(epoch, worker, claim object name), ...]} -- one listing, no reads."""
    claims = {}
    for c in loc_lsf(LOC_CLAIM_REMOTE):
        pc = loc_claim_parse(c)
        if pc:
            claims.setdefault(pc[0], []).append((pc[2], pc[1], c))
    return claims


def loc_worker_claim(name, drop=()):
    p = LOC_ROOT / '_claim.json'
    p.write_text(json.dumps({'req': name, 'worker': loc_session_id(),
                             'utc': time.strftime('%Y-%m-%dT%H:%M:%SZ', time.gmtime())}))
    cn = loc_claim_name(name)
    loc_push(p, cn, LOC_CLAIM_REMOTE)
    for old in drop:
        loc_rm(old, LOC_CLAIM_REMOTE)
    return cn


def loc_worker_claim_confirmed(name, drop=(), beats=None):
    """Claim `name`, then RE-READ and yield if someone else got there first.

    This is the fix for the duplication the live run actually showed.  `loc_worker_pick` lists,
    finds nothing, and the claim it then writes is two rclone round trips away from being
    visible; with the GPU keeping exactly one request outstanding and four workers polling it,
    every worker's list falls inside every other worker's write window.  Measured: 81 of 135
    requests worked by 2-4 workers, 22 of them losing a `moveto` race at the very end and being
    counted as `boot_failed` after 130-230 s of finished work.  No steal was involved in any of
    them -- `taking over a STALE claim` appears zero times in nine logs.

    So: write, wait out the visibility lag, list again, and the EARLIEST live claim wins (ties
    broken on worker id, which is stable and total).  A loser deletes its own claim and goes
    back to the queue, which costs it one poll and costs the run nothing.  Returns the claim
    object's name, or None when this worker yielded.
    """
    cn = loc_worker_claim(name, drop=drop)
    if LOC_CLAIM_CONFIRM_S <= 0:
        return cn
    mine, t_mine = loc_session_id(), loc_claim_parse(cn)[2]
    time.sleep(LOC_CLAIM_CONFIRM_S)
    beats = loc_worker_beats() if beats is None else beats
    try:
        held = loc_claims_by_req().get(name) or []
    except Exception:                                           # noqa: BLE001
        return cn                                # an unreadable listing is not a lost race
    rivals = [c for c in held
              if c[1] != mine and loc_claim_is_live(c, beats) and (c[0], c[1]) < (t_mine, mine)]
    if not rivals:
        return cn
    loc_rm(cn, LOC_CLAIM_REMOTE)
    loc_log(f'  yielding {name}: {rivals[0][1]} claimed it first ({rivals[0][0]} vs {t_mine}); '
            f'two workers on one roll is bit-identical but wasted, and half the pool\'s work '
            f'went that way before this check existed')
    return None


def loc_worker_once(M, st, seen):
    """Claim one item on either channel, answer it, publish.  A per-item record, or None.

    THE TAIL IS TRIED FIRST and that ordering is deliberate: a bootstrap request is posted a
    whole roll before its answer is needed, while a tail request is posted one roll before its
    and is the last thing between a finished roll and the next one, so latency on the tail
    channel is worth more than latency on the bootstrap channel.
    """
    beats = loc_worker_beats()
    for key, req_remote, res_remote, stage in LOC_CHANNELS:
        remote, res = globals()[req_remote], globals()[res_remote]
        name, info = loc_worker_pick(seen, remote=remote, done_remote=res, beats=beats)
        if not name:
            continue
        info['channel'] = key
        roll = loc_req_parse(name)[1]
        claim = loc_worker_claim_confirmed(name, drop=info.get('stolen_from') or (),
                                           beats=beats)
        if claim is None:
            return {'roll': roll, 'req': name, 'channel': key, 'yielded': True}, info
        if info.get('stolen_from'):
            loc_log(f'  roll {roll} ({key}): taking over the claim of a worker whose heartbeat '
                    f'has gone silent ({info["stolen_from"]})')
        return loc_worker_answer(M, st, key, name, claim, roll, remote, res, stage), info
    return None, {'n_req': 0, 'channel': None}


def loc_worker_answer(M, st, key, name, claim, roll, remote, res_remote, stage):
    """Download one request, run its stage, publish the answer, tear the question down.

    Structurally the same for both channels, which is the point: one claim protocol, one sha
    binding, one answer-before-the-question-is-deleted ordering, one refresher thread.  Only
    `loc_worker_run_<key>` differs.

    The `gone` outcome is NOT a failure and is counted apart from one.  It was the whole of the
    live run's `boot_failed`: 22 records across eight sessions, every one of them a lost race to
    `rclone moveto` after the work was finished, because two workers had claimed the same roll
    and written the same `.tmp`. The claim confirmation above is what stops that happening; this
    is what stops the residue being read as a defect in the bootstrap.
    """
    t0 = time.time()
    p = LOC_ROOT / f'_req_{key}_{roll}.npz'
    p.unlink(missing_ok=True)
    rec = {'roll': roll, 'req': name, 'claim': claim, 'channel': key}
    # every claim this worker has written for this request, so a failure releases the CURRENT
    # one and not just the first: a refreshed claim left behind would block the other worker
    held = [claim]
    try:
        if not (loc_pull(name, p, remote) and p.exists()):
            rec['gone'] = 'the request object was gone before it could be downloaded - another '\
                          'worker answered it, or the GPU retired it'
            loc_rm(claim, LOC_CLAIM_REMOTE)
            loc_log(f'  roll {roll} ({key}): {rec["gone"]}; releasing the claim and polling on. '
                    f'This is a race, not a failure.')
            return rec
        rec['download_s'] = round(time.time() - t0, 1)
        rec['bytes'] = int(p.stat().st_size)
        sha = loc_file_sha(p)
        assert loc_req_parse(name)[3] == sha, (
            f'{name} hashes to {sha}: the object on the remote is NOT the one this name '
            f'promises. Refusing to answer it -- a wrong answer here is silent.')
        t1 = time.time()
        # A refresher thread, so a roll that takes longer than the TTL is never stolen from a
        # worker that is still working on it.  It carries the HEARTBEAT too: the poll loop is
        # blocked inside the stage for ~50-140 s, and a worker that looks dead for that long on
        # every roll would trip the GPU's breaker on a session that is working perfectly.
        stop = threading.Event()

        def refresh():
            while not stop.wait(LOC_CLAIM_REFRESH_S):
                try:
                    held.append(loc_worker_claim(name, drop=[held[-1]]))
                    loc_heartbeat(dict(loc_worker_hb(st, stage), roll=roll,
                                       elapsed_s=round(time.time() - t1, 1)), force=True)
                except Exception:                               # noqa: BLE001
                    pass
        th = threading.Thread(target=refresh, daemon=True)
        th.start()
        try:
            out, note = (loc_worker_run_boot(M, p, sha, rec) if key == 'boot'
                         else loc_worker_run_tail(M, p, sha, rec))
        finally:
            stop.set()
            th.join(timeout=5)
        st['worker']['busy_s'] = round(st['worker'].get('busy_s', 0.0) + time.time() - t1, 1)
        # the ANSWER lands before the question is torn up: a worker that dies between these two
        # leaves a durable result and an orphan request, which the GPU deletes when it consumes
        # the result.  The other order would leave the GPU with neither.
        loc_push(out, name, res_remote)
        out.unlink(missing_ok=True)
        loc_rm(name, remote)
        for c in held:
            loc_rm(c, LOC_CLAIM_REMOTE)
        rec['ok'] = True
        loc_log(f'  roll {roll} ({key}): {note}, {rec["bytes"] / 2 ** 20:.1f} MB down in '
                f'{rec["download_s"]}s, answered as {name}')
    except Exception as e:                                      # noqa: BLE001
        rec['error'] = f'{type(e).__name__}: {str(e)[:300]}'
        rec['ok'] = False
        for c in held:
            loc_rm(c, LOC_CLAIM_REMOTE)
        loc_log(f'  roll {roll} ({key}) FAILED on this worker: {rec["error"]}. The claim is '
                f'released; the GPU falls back locally if nobody else answers.', push_=True)
    finally:
        p.unlink(missing_ok=True)
    return rec


def loc_worker_run_boot(M, p, sha, rec):
    feats, seed, meta = loc_boot_unpack(p)
    fp = loc_map_fingerprint(M)
    assert meta['map'] == fp, f'map/parameter mismatch: request {meta["map"]} vs mine {fp}'
    t1 = time.time()
    est0, ref0 = loc_pnp_options(0)
    anchor_idx, arcs, oks, inls = loc_run_bootstrap(M, feats, seed, est0, ref0)
    rec['boot_s'] = round(time.time() - t1, 1)
    rec['n_anchor'] = int(len(anchor_idx))
    rec['s_per_anchor'] = round(rec['boot_s'] / max(len(anchor_idx), 1), 3)
    out = LOC_ROOT / f'_res_{rec["roll"]}.npz'
    np.savez(out, anchor_idx=anchor_idx, arcs=arcs, oks=oks, inls=inls,
             req_sha=np.asarray(sha), worker=np.asarray(loc_session_id()),
             boot_s=np.asarray(rec['boot_s'], np.float64))
    return out, (f'{rec["n_anchor"]} anchors in {rec["boot_s"]}s '
                 f'({rec["s_per_anchor"]}s/anchor)')


def loc_worker_run_tail(M, p, sha, rec):
    corr, seed, meta = loc_tail_unpack(p, M)
    fp = loc_map_fingerprint(M)
    assert meta['map'] == fp, f'map/parameter mismatch: request {meta["map"]} vs mine {fp}'
    sf = loc_solver_fingerprint()
    assert meta['solver'] == sf, (
        f'solver mismatch: request {meta["solver"]} vs mine {sf}. The pose this worker computed '
        f'would not be the one the GPU session computes, and the in-map controls are read off '
        f'these poses -- refusing rather than answering with a different solver.')
    t1 = time.time()
    intr, per, prof = loc_tail_solve(M, corr, meta['seed'])
    rec.update(prof, tail_s=round(time.time() - t1, 1), n_frames=len(per),
               n_solved=int(sum(p_['ok'] for p_ in per)))
    out = LOC_ROOT / f'_tailres_{rec["roll"]}.npz'
    np.savez(out, **loc_tail_pack_result(per, intr, sha, dict(prof, tail_s=rec['tail_s'])))
    return out, (f'{rec["n_solved"]}/{rec["n_frames"]} frames solved in {rec["tail_s"]}s '
                 f'(intrinsics {prof["intrinsics_s"]}s, pnp {prof["pnp_s"]}s), '
                 f'f={intr["focal"]:.1f} k1={intr["k1"]:.4f}')


def loc_worker_hb(st, stage):
    """The worker's heartbeat, with its OWN busy accounting.

    Cumulative busy seconds and the fraction of the session they are, measured by the worker
    itself, because every outside estimate of worker utilisation so far has been wrong: one was
    computed against the wrong denominator, and the per-item numbers visible from the GPU side
    include poll delay, download and upload rather than compute.  A decision about how many
    workers to run needs the worker's own clock.
    """
    w = st.get('worker') or {}
    elapsed = max(time.time() - LOC_T0, 1e-9)
    return {'roll': None, 'roll_idx': None, 'n_rolls': None, 'frame': 0, 'n_frames': 0,
            'rate_fps': None, 'eta_utc': None, 'stage': stage,
            'min_to_cap': round(LOC_SESSION_MAX_MIN - loc_session_min(), 1),
            'boot_done': w.get('n_ok_boot', 0), 'tail_done': w.get('n_ok_tail', 0),
            'boot_failed': w.get('n_fail', 0), 'n_gone': w.get('n_gone', 0),
            'n_yielded': w.get('n_yielded', 0),
            'busy_s': round(w.get('busy_s', 0.0), 1),
            'busy_frac': round(w.get('busy_s', 0.0) / elapsed, 3),
            'session_s': round(elapsed, 1)}


def loc_worker_loop(M, st):
    """The CPU session's whole life: claim, bootstrap, answer, repeat.

    It owns none of the run's state -- its own state/heartbeat/log are suffixed with its session
    id -- and it can be killed at any instant without costing the run more than one roll's wait.
    The GPU session is the one that decides anything.
    """
    loc_log(f'CPU WORKER {loc_session_id()} ready: run_id={LOC_RUN_ID}, '
            f'{loc_workers()} bootstrap workers on {loc_physical_cores()} physical cores, '
            f'faiss ntotal={M.index.ntotal}, session cap {LOC_SESSION_MAX_MIN:.0f} min, '
            f'idle cap {LOC_WORKER_IDLE_MAX_S / 60:.0f} min', push_=True)
    st['worker'] = {'session_id': loc_session_id(), 'items': [], 'n_ok': 0, 'n_fail': 0,
                    'n_ok_boot': 0, 'n_ok_tail': 0, 'n_gone': 0, 'n_yielded': 0,
                    'busy_s': 0.0}
    seen, idle_t0, n_poll = set(), time.time(), 0
    reason = None
    while True:
        min_to_cap = LOC_SESSION_MAX_MIN - loc_session_min()
        hb = dict(loc_worker_hb(st, 'worker_idle'),
                  idle_s=round(time.time() - idle_t0, 1), n_poll=n_poll)
        # not forced: the poll is 15 s and a forced heartbeat is two rclone calls, so forcing
        # would spend ~13% of an idle worker's session telling Drive it is idle
        loc_heartbeat(hb)
        if min_to_cap <= 2.0:
            reason = f'session cap ({LOC_SESSION_MAX_MIN:.0f} min) reached'
            break
        if loc_worker_stop():
            reason = f'stop requested ({(LOC_STOP["seen"] or {}).get("reason")})'
            break
        # LOC_BUDGET_H IS NOT APPLIED HERE.  It is the GPU-QUOTA cap -- Kaggle bills the GPU
        # session's wall-clock against a weekly allowance -- and a CPU session consumes none of
        # it.  Retiring a worker on the GPU's budget would take the pool away exactly when the
        # last GPU session of the week needs it most; the 11h30 session cap is what retires a
        # worker, and the idle cap is the backstop for one whose GPU session has gone.
        n_poll += 1
        try:
            rec, info = loc_worker_once(M, st, seen)
        except Exception as e:                                  # noqa: BLE001
            loc_log(f'worker poll failed (non-fatal): {type(e).__name__}: {str(e)[:200]}')
            rec, info = None, {'error': str(e)[:120]}
        if rec is None:
            if time.time() - idle_t0 > LOC_WORKER_IDLE_MAX_S:
                reason = (f'idle for {LOC_WORKER_IDLE_MAX_S / 3600:.1f} h - the GPU session this '
                          f'was helping is gone')
                break
            time.sleep(LOC_WORKER_POLL_S)
            continue
        w = st['worker']
        w['items'].append(rec)
        w['items'] = w['items'][-50:]
        if rec.get('ok'):
            idle_t0 = time.time()
            w['n_ok'] += 1
            w[f'n_ok_{rec["channel"]}'] += 1
        elif rec.get('yielded'):
            # lost the claim race, deliberately: not idle time, not a failure, and the request
            # must NOT go in `seen` -- the winner may die and this worker should be able to
            # take it back
            idle_t0 = time.time()
            w['n_yielded'] += 1
        elif rec.get('gone'):
            idle_t0 = time.time()
            w['n_gone'] += 1
            seen.add(rec['req'])           # it is not there; do not poll it again
        else:
            idle_t0 = time.time()
            w['n_fail'] += 1
            seen.add(rec['req'])           # do not spin on a request this worker cannot answer
        w['n_seen_unanswerable'] = len(seen)
        loc_save_state(st)
    st['stop_reason'] = reason
    st['finished_utc'] = time.strftime('%Y-%m-%dT%H:%M:%SZ', time.gmtime())
    if LOC_STOP['seen']:
        st['stop_requested'] = dict(LOC_STOP['seen'], seen_at=LOC_STOP['where'])
    st['io_pools'] = loc_pools_flush()
    loc_save_state(st)
    loc_heartbeat(dict(loc_worker_hb(st, 'worker_done'), stop_reason=reason), force=True)
    w = st['worker']
    loc_log(f'CPU WORKER DONE: {w["n_ok_boot"]} bootstraps + {w["n_ok_tail"]} tails answered in '
            f'{w["busy_s"]:.0f}s of compute ({100 * w["busy_s"] / max(time.time() - LOC_T0, 1):.0f}'
            f'% of the session), {w["n_fail"]} failed, {w["n_gone"]} already gone, '
            f'{w["n_yielded"]} yielded to a faster claim, reason: {reason}', push_=True)
    return st


def loc_split_gc(st):
    """Delete handoff objects that belong to no live question.  224 GB is what skipping costs.

    ROLE-DEPENDENT, because the two roles know different things.  The GPU session is the only
    thing that ever asks a question, so at ITS startup no outstanding request can still be
    wanted: whatever is there was asked by a session that is gone, and this one will re-post
    what it needs.  (The payload is re-serialised each time and a zip carries timestamps, so a
    re-post is a new name anyway -- an orphan would never be reused, only stored.)

    A worker must not apply that rule: it does not know whether the GPU is alive, and clearing
    the queue would delete the very work it exists to do.  It removes only what is provably
    dead -- a different run_id, or a roll whose output is already banked -- and claims whose
    request has gone.
    """
    run = LOC_RUN_ID or loc_run_id()
    done = {int(r) for r in (st.get('done') or [])}
    sweep = loc_role() == 'gpu'
    rec = {'role': loc_role(), 'sweep_all': sweep, 'anchors': 0, 'boot': 0, 'claims': 0,
           'tail': 0, 'tailres': 0, 'foreign': 0, 'tmp': 0}
    try:
        live = set()
        for remote, key in ((LOC_ANCHOR_REMOTE, 'anchors'), (LOC_BOOT_REMOTE, 'boot'),
                            (LOC_TAIL_REMOTE, 'tail'), (LOC_TAILRES_REMOTE, 'tailres')):
            for n in loc_lsf(remote):
                p = loc_req_parse(n)
                if p is None:
                    # a half-finished push, left by a session that died between copyto and
                    # moveto.  Nothing will ever claim it -- the name cannot parse -- and the
                    # live run had eight of them sitting at 5-6 kB each.  Only the GPU sweeps:
                    # a worker cannot tell one of its peers' in-flight pushes from an orphan.
                    if sweep and n.endswith('.tmp'):
                        rec['tmp'] += 1
                        loc_rm(n, remote)
                    continue
                if p[0] != run:
                    rec['foreign'] += 1
                    loc_rm(n, remote)
                elif sweep or p[1] in done:
                    rec[key] += 1
                    loc_rm(n, remote)
                else:
                    live.add(n)
        for n in loc_lsf(LOC_CLAIM_REMOTE):
            pc = loc_claim_parse(n)
            if pc is None or pc[0] not in live:
                rec['claims'] += 1
                loc_rm(n, LOC_CLAIM_REMOTE)
        # Otherwise every session's heartbeat and state object stay for ever, and the liveness
        # checks slow down with the pile.  The wide margin keeps a live worker's file safe.
        if sweep:
            for t, n in loc_lsf_t(LOC_REMOTE):
                if not re.match(r'^(heartbeat|state)_cpu_.+\.json$', n):
                    continue
                if time.time() - t > max(LOC_WORKER_STALE_S * 20, 3600.0):
                    rec['stale_cpu'] = rec.get('stale_cpu', 0) + 1
                    loc_rm(n, LOC_REMOTE)
    except Exception as e:                                      # noqa: BLE001
        rec['error'] = f'{type(e).__name__}: {str(e)[:200]}'
    if any(v for k, v in rec.items() if k not in ('error', 'role', 'sweep_all')):
        loc_log(f'split gc: {rec}')
    return rec

## `loc-roll`

One roll end to end, plus the in-map controls.

In [ ]:
# === cell: loc-roll ===
def loc_feats_bytes(feats):
    """Resident bytes of one roll's features, counted rather than estimated.

    This is what the cross-roll overlap has to fit twice over, so it is measured from the roll
    just extracted and used to decide whether the next one may be held alongside it.
    """
    n = int(feats['t_ms'].nbytes) + int(feats['dec_idx'].nbytes)
    for k in ('kp', 'desc', 'score'):
        n += int(sum(a.nbytes for a in feats[k]))
    n += int(sum(a.nbytes for a in feats['desc32'].values() if a is not None))
    return n


def loc_mem_available_b():
    """MemAvailable, or None when it cannot be read.

    MemAvailable and not MemFree: on a machine whose spare RAM is all page cache, MemFree
    would refuse an overlap the kernel would have granted without hesitation.  None means the
    guard could not measure, and an unmeasured guard is not a guard -- the caller declines the
    overlap rather than proceeding on an assumption.
    """
    try:
        for line in Path('/proc/meminfo').read_text().split('\n'):
            if line.startswith('MemAvailable:'):
                return int(line.split()[1]) * 1024
    except Exception:                                           # noqa: BLE001
        pass
    return None


def loc_extract_phase(runners, row, video, hb_base=None, M=None):
    """Decode + ALIKED for one roll: the GPU half, separable so it can run a roll ahead.

    Returns (feats, info).  `hb_base` is None when this runs on the look-ahead thread -- the
    heartbeat writes and pushes a single file, and letting a background extract race the main
    thread's own heartbeats would corrupt the one artefact the laptop watches the run through.
    The main thread is heartbeating its own stage throughout the overlap, so liveness is not
    lost by keeping this quiet.
    """
    info = {'roll': int(row['roll']), 'file_id': int(row['file_id']),
            'start_ms': row['start_ms'], 'end_ms': row['end_ms'],
            'clamped': row.get('clamped'), 'src_wh': row.get('src_wh'),
            'mask_frac': row.get('mask_frac'), 'hz': LOC_HZ, 't0_roll': time.time()}
    vinfo = loc_video_info(video)
    info['video'] = vinfo
    targets, clamp = loc_targets(row['start_ms'], row['end_ms'], vinfo['video_ms'])
    info['clamp'] = clamp
    info['n_targets'] = len(targets)
    if clamp['n_targets_past_eof'] or clamp['start_clamped_ms']:
        loc_log(f'  {row["roll"]}: clamped to the footage that exists - '
                f'{clamp["n_targets_past_eof"]}/{clamp["n_targets_full"]} targets past EOF '
                f'({clamp["end_clamped_ms"]:.0f} ms over), head clamped by '
                f'{clamp["start_clamped_ms"]:.0f} ms. The event data is not edited, the grid is.')
    if len(targets) == 0:
        raise RuntimeError(f'roll {row["roll"]}: no 10 Hz target inside the video')
    mask = loc_mask(LOC_IN / 'masks' / f'{row["file_id"]}.png') if row.get('mask') else None
    hb = None if hb_base is None else (
        lambda k: loc_heartbeat(dict(hb_base, frame=k, n_frames=len(targets),
                                     stage='extract')))
    t0 = time.time()
    feats = loc_extract_roll(runners, video, targets, mask, out=info,
                             max_frames=LOC_MAX_FRAMES, hb=hb)
    n = len(feats['t_ms'])
    info['extract_s'] = round(time.time() - t0, 2)
    info['n_frames'] = n
    info['feats_bytes'] = loc_feats_bytes(feats)
    loc_stop_requested(where='after extract')
    if info.get('n_intro_dropped'):
        # b1_plan.py floors the start at local_start_ms=3750 for all 485 title-card rolls and
        # 0 of them now start inside the card, so a firing here is not routine housekeeping --
        # it means the plan's bounds or the source events regressed upstream.
        loc_log(f'  *** {row["roll"]}: {info["n_intro_dropped"]} VIRB Edit TITLE-CARD frames '
                f'dropped. On this plan that should be ZERO - the start floor is supposed to '
                f'clear the card. Check plan.json bounds for this roll. ***', push_=True)
    info['kp_mean'] = float(np.mean([len(k) for k in feats['kp']])) if n else 0.0
    # The request goes out HERE, at the first instant the anchors exist, which on the look-ahead
    # path is roughly a full roll before the answer is needed.  That head start is the whole
    # reason the split can hide a 131-153 s stage behind a 120-145 s one; posting it at the top
    # of loc_process_head instead would leave the worker's ~88 s entirely exposed.
    if M is not None:
        info['boot_req'] = loc_boot_post(row, feats, M)
    # Memory, four ways, because MemAvailable alone cannot say WHERE the ~25.8 MiB/roll goes:
    # `feats` is an accounting sum over this roll's arrays, `rss` what the process really holds,
    # `hwm` its peak, `free` MemAvailable.  rss rising with free falling is native allocation in
    # here; rss flat with free falling is a loss outside this process entirely.  `trim@roll` is
    # the last malloc_trim, which the look-ahead leaves a roll or three behind.
    free_b = loc_mem_available_b()
    free_s = '?' if free_b is None else f'{free_b / 2 ** 30:.2f}'
    rss_b, hwm_b = loc_rss_b(), loc_vm_hwm_b()
    rss_s = '?' if rss_b is None else f'{rss_b / 2 ** 30:.2f}'
    hwm_s = '?' if hwm_b is None else f'{hwm_b / 2 ** 30:.2f}'
    info['rss_b'], info['vm_hwm_b'], info['mem_available_b'] = rss_b, hwm_b, free_b
    loc_log(f'  {row["roll"]}: {n} frames, {info["kp_mean"]:.0f} kp/frame, extract '
            f'{info["extract_s"]}s ({n / max(info["extract_s"], 1e-6):.1f} f/s), '
            f'{info["feats_bytes"] / 2 ** 30:.2f} GiB feats, {rss_s} GiB rss, {hwm_s} GiB hwm, '
            f'{free_s} GiB free, {loc_trim_str()}; per-device '
            + ' '.join(f'gpu{k}: {d["n"]} frames ({info["device_share"][str(k)]:.0%}) '
                       f'{d["busy_s"]}s {d["ms_per_frame"]}ms/frame {d["eps"][0]}'
                       for k, d in sorted(info['per_device'].items())))
    return feats, info


class LocAhead:
    """Extract the NEXT FEW rolls on the GPU while this roll's CPU stages run.  PLAN 4's
    cross-roll overlap, as a bounded QUEUE rather than the single slot it started as.

    The GPU is idle for most of a roll: a roll is roughly 77 s of extract against ~94 s of
    bootstrap + intrinsics, which hide each other almost exactly.  Nothing about WHAT is
    computed changes -- the same loc_extract_phase runs on the same frames and its output is
    handed to the same loc_process_head -- only when.

    ⚠ DEPTH IS NOT CONCURRENCY.  One extract thread, always: two extracts racing for the same
    two T4s finish no sooner and hold a second roll's features the whole time they do it.  What
    the queue buys is that a roll's BOOTSTRAP REQUEST goes out further ahead of the moment its
    answer is needed -- loc_extract_phase posts it at the instant the anchors exist, so a queue
    d deep posts it roughly d roll-times early.  At depth 1 the live run had 2 bootstrap + 1
    tail questions outstanding against 4 CPU workers: one worker idle by construction, ~15 s of
    billed idle per roll, and 5 of 21 bootstraps timing out and paying a full ~140 s local stage
    on top of the wait.  Depth 3 keeps 3-4 bootstrap questions out at once.

    THREE RULES, because this is the one piece of the pipeline that can be wrong quietly:
      * it never holds another roll's features without first measuring that they fit, and it
        measures against how full the queue ALREADY is rather than against a constant;
      * `take` matches by ROLL ID across the queue and then re-checks the identity of what the
        extract returned.  Roll order is not something this class gets to assume, and handing
        back the wrong roll's features is the one failure here that would not look like one:
        every stage downstream would work perfectly, on the wrong frames; and
      * any failure at all degrades to serial extraction of that roll rather than propagating.
        A look-ahead that fails must not be able to mark a roll failed: the cause may be
        pressure or contention that has nothing to do with the roll, so the roll is simply
        re-extracted inline and any real defect surfaces through the normal path.
    """

    def __init__(self, runners, pre, M=None, depth=None):
        import concurrent.futures as cf
        self.runners, self.pre, self.M = runners, pre, M
        self.depth = max(1, int(LOC_AHEAD_DEPTH if depth is None else depth))
        self.pool = cf.ThreadPoolExecutor(max_workers=1)
        self.q = []                     # [{'roll', 'idx', 'need_b', 'fut'}], in plan order
        LOC_AHEAD_STATS['depth_target'] = self.depth

    def _work(self, row, after):
        video, fetch_s, fetch_wait = self.pre.get(row)
        self.pre.start(after)                  # keep the DOWNLOAD a roll ahead of that
        feats, info = loc_extract_phase(self.runners, row, video, M=self.M)
        self.pre.drop(video)                   # the features are what is needed; free the disk
        info['fetch_s'] = round(fetch_s, 1)
        info['fetch_wait_s'] = round(fetch_wait, 1)
        return feats, info

    def busy(self):
        """Is the GPU still doing work of its own?  What arms the bootstrap's wait clock.

        `not busy` does not mean "idle in general" -- it means this session has nothing left to
        put on the accelerator until the roll in hand finishes, so from this instant on, waiting
        for the worker is billed rather than free.  With a queue that reads "no entry is still
        running": an entry whose extract has finished is a result in hand, not work.
        """
        return any(not e['fut'].done() for e in self.q)

    def _sized_for(self):
        """How many rolls' worth of pages the guard must find room for: the UNALLOCATED ones.

        The candidate plus any unfinished queued extract.  Everything already resident has been
        subtracted from MemAvailable, so counting it here asks for the same memory twice.
        Residency is bounded by LOC_AHEAD_DEPTH, not by this number.
        """
        return LOC_AHEAD_ROLLS or (1 + sum(1 for e in self.q if not e['fut'].done()))

    def _admit(self, row, need_b):
        """(ok, record) -- the memory guard, on measured MemAvailable, per slot."""
        free = loc_mem_available_b()
        n_res = self._sized_for()
        want = int(need_b * n_res * LOC_AHEAD_SAFETY) + LOC_AHEAD_HEADROOM_B
        rec = {'roll': int(row['roll']), 'slot': len(self.q) + 1,
               'resident_rolls_sized_for': n_res + 1,
               'free_gib': None if free is None else round(free / 2 ** 30, 2),
               'need_gib': round(want / 2 ** 30, 2)}
        if free is None or free < want:
            rec['reason'] = ('MemAvailable unreadable' if free is None
                             else 'not enough free RAM')
            return False, rec
        return True, rec

    def fill(self, rows, i0, i1, bytes_per_frame):
        """Top the queue up to `depth`, over plan indices [i0, i1).  Returns what it achieved.

        CONTIGUOUS BY CONSTRUCTION: the queue is the next rolls in plan order and a fill stops
        at the first roll it cannot admit, so `take` is never handed a hole to step over.
        `bytes_per_frame` is measured on the roll just extracted and scaled by each candidate's
        own frame estimate, so a long roll is sized as a long roll rather than as an average.
        """
        rec = {'depth_target': self.depth, 'depth_before': len(self.q), 'started': [],
               'declined': None, 'reason': None}
        if not LOC_AHEAD:
            rec['reason'] = 'SRS_LOC_AHEAD=0'
        while LOC_AHEAD and len(self.q) < self.depth:
            j = max(i0, self.q[-1]['idx'] + 1 if self.q else i0)
            if j >= i1:
                rec['reason'] = 'no further roll to extract'
                break
            row = rows[j]
            need_b = int(bytes_per_frame * float(row.get('n_frames_est') or 1717))
            ok, dec = self._admit(row, need_b)
            if not ok:
                rec['declined'] = dec
                self._decline(row, dec)
                break
            self.q.append({'roll': int(row['roll']), 'idx': j, 'need_b': need_b,
                           'fut': self.pool.submit(self._work, row,
                                                   rows[j + 1] if j + 1 < i1 else None)})
            LOC_AHEAD_STATS['started_total'] += 1
            rec['started'].append(int(row['roll']))
        rec['depth'] = len(self.q)
        rec['rolls'] = [e['roll'] for e in self.q]
        # `declined_consec` means, and has always meant, "the next roll will be extracted
        # serially".  With N slots a PARTIAL fill is ordinary -- the plan runs out, or the third
        # slot is refused while the first two were granted -- and only a fill that leaves the
        # queue EMPTY actually serialises anything, so that is what the counter and its
        # escalation stay on.  Which slot was refused is counted separately, per slot.
        if rec['depth'] == 0 and rec['declined']:
            LOC_AHEAD_STATS['declined_consec'] += 1
            # A permanent decline is the failure that actually happens: it is silent, it costs
            # ~1.3x on every remaining roll, and from outside it is indistinguishable from a
            # slow session.  An OOM would at least be loud.  So it is escalated on a count.
            if LOC_AHEAD_STATS['declined_consec'] == LOC_AHEAD_DECLINE_WARN:
                d = rec['declined']
                loc_log(f'*** THE LOOK-AHEAD HAS DECLINED {LOC_AHEAD_DECLINE_WARN} ROLLS IN A '
                        f'ROW ({d["reason"]}, {d["free_gib"]} GiB free vs {d["need_gib"]} GiB '
                        f'needed for {d["resident_rolls_sized_for"]} resident rolls). The run '
                        f'is now effectively SERIAL and ~1.3x slower than the projection, and '
                        f'nothing else will say so. If free RAM is nowhere near the requirement '
                        f'this is a leak, not pressure. ***', push_=True)
        elif rec['depth']:
            LOC_AHEAD_STATS['declined_consec'] = 0
        return rec

    def _decline(self, row, rec):
        """One refused slot: counted per slot, logged, and the DOWNLOAD still runs ahead.

        Per slot because the two declines mean different things.  Slot 1 refused is the serious
        one -- the next roll gets no overlap at all and the run goes serial.  Slot 3 refused
        just means a shallower queue and a shorter head start on the bootstrap request, which
        costs latency, not a stage.  One counter for both would read as the first and usually
        be the second.
        """
        LOC_AHEAD_STATS['declined_total'] += 1
        LOC_AHEAD_STATS['last_reason'] = rec['reason']
        LOC_AHEAD_STATS['last_free_gib'] = rec['free_gib']
        by = LOC_AHEAD_STATS['declined_by_slot']
        by[str(rec['slot'])] = by.get(str(rec['slot']), 0) + 1
        self.pre.start(row)                # the fetch is disk, not RAM; it can still run ahead
        loc_log(f'  look-ahead DECLINED slot {rec["slot"]}/{self.depth} for roll {rec["roll"]}: '
                f'{rec["reason"]} (free {rec["free_gib"]} GiB < {rec["need_gib"]} GiB needed '
                f'for {rec["resident_rolls_sized_for"]} resident rolls). '
                + ('This roll is extracted serially, which costs time and nothing else.'
                   if rec['slot'] == 1 else
                   'The next roll is still extracted ahead; only the queue is shallower.'))

    def take(self, row):
        """(feats, info, wait_s) when `row` was extracted ahead, else None.  Never raises.

        MATCHED BY ROLL ID, NEVER BY POSITION, and then checked again against the roll id the
        extract itself recorded.  Plan order is normally fixed, but a retry or a roll the loop
        skipped would shift it, and a queue that answered positionally would hand back another
        roll's features -- silently, with every stage downstream working perfectly on the wrong
        frames and an output that names the right roll.  Two checks, and a re-extract if either
        fires, is the price of never having to reason about that afterwards.
        """
        want = int(row['roll'])
        depth, ready = len(self.q), sum(1 for e in self.q if e['fut'].done())
        LOC_AHEAD_STATS['depth_last'], LOC_AHEAD_STATS['ready_last'] = depth, ready
        hist = LOC_AHEAD_STATS['depth_hist']
        hist[str(depth)] = hist.get(str(depth), 0) + 1
        at = next((i for i, e in enumerate(self.q) if e['roll'] == want), None)
        if at is None:
            LOC_AHEAD_STATS['miss_total'] += 1
            LOC_AHEAD_STATS['miss_consec'] += 1
            if depth:
                loc_log(f'  roll {want} is not in the look-ahead queue '
                        f'{[e["roll"] for e in self.q]}, so it is extracted here. Nothing '
                        f'queued is discarded: those rolls are still ahead of this one.')
            return None
        LOC_AHEAD_STATS['miss_consec'] = 0
        if at:
            self._discard(self.q[:at], f'roll {want} was queued at position {at}, not the head')
            del self.q[:at]
        e = self.q.pop(0)
        t = time.time()
        try:
            feats, info = e['fut'].result()
        except Exception as exc:                                # noqa: BLE001
            LOC_AHEAD_STATS['failed'] += 1
            loc_log(f'  look-ahead extract of roll {want} FAILED '
                    f'({type(exc).__name__}: {str(exc)[:200]}); re-extracting it serially. A '
                    f'look-ahead failure is not a roll failure and is not counted as one.')
            return None
        if int(info.get('roll', -1)) != want:
            LOC_AHEAD_STATS['mismatched'] += 1
            loc_log(f'*** THE LOOK-AHEAD RETURNED ROLL {info.get("roll")} WHERE ROLL {want} WAS '
                    f'ASKED FOR. Nothing from it is used and roll {want} is extracted here. '
                    f'The queue is matched by roll id, so a reordering alone cannot produce '
                    f'this -- it means the extract mislabelled its own output, and every roll '
                    f'banked before this one should be treated as suspect. ***', push_=True)
            loc_boot_abandon(info.get('boot_req'))
            return None
        return feats, info, time.time() - t

    def _discard(self, entries, why):
        """Drop queue entries whose rolls the loop has already passed.  Off the main thread.

        Their bootstrap requests are ABANDONED rather than left behind: at ~202 MB each they are
        the storage problem, and nothing will ever consume them.  Abandoning does not move the
        circuit breaker -- a reordering says nothing about whether the split is working.
        """
        if not entries:
            return
        LOC_AHEAD_STATS['reordered'] += 1
        loc_log(f'*** THE LOOK-AHEAD QUEUE IS OUT OF PLAN ORDER: {why}. Dropping '
                f'{[e["roll"] for e in entries]} and their requests. Those rolls are simply '
                f'extracted again if the plan reaches them; no roll is failed by this. ***',
                push_=True)

        def _drain():
            for e in entries:
                e['fut'].cancel()
                self.pre.cancel(e['roll'])
                try:
                    _, info = e['fut'].result()
                except Exception:                               # noqa: BLE001
                    continue
                loc_boot_abandon(info.get('boot_req'))
        threading.Thread(target=_drain, daemon=True).start()

    def shutdown(self):
        """Stop extracting rolls this session will not reach.

        What is already on the remote for a queued roll is left there.  The objects are named
        under the run id and the next GPU session's `loc_split_gc` sweeps every outstanding
        request at startup, which is cheaper and far more certain than blocking a session that
        may be seconds from being reclaimed on ~200 MB of upload it no longer needs.
        """
        self.pool.shutdown(wait=False, cancel_futures=True)
        self.q = []


def loc_boot_phase(M, row, feats, seed, hb_base, n, info, ahead=None):
    """This roll's anchor solves: the CPU worker's answer if it arrives, otherwise our own.

    The two paths are interchangeable by construction, not by hope.  Both run the same
    `loc_run_bootstrap` against the same map bundle with faiss pinned to one OpenMP thread per
    pool worker, pycolmap RANSAC seeded at 0 with one thread, and the fp16->fp32 widening of the
    bank rows is exact on either device -- so the remote answer is bit-identical to the one this
    session would have computed, and `boot_source` is a note about where time went, never about
    what the numbers mean.
    """
    est0, ref0 = loc_pnp_options(0)
    req = info.get('boot_req')
    loc_heartbeat(dict(hb_base, frame=0, n_frames=n, stage='bootstrap'), force=True)
    if req and req.get('name'):
        # THE ONE PLACE the upload has to have finished: a worker cannot answer a question whose
        # bytes are not on the remote yet, so entering the poll loop before this would spend the
        # wait clock on our own transfer.  Everywhere else the upload runs behind the GPU.
        loc_post_join(req, why='before waiting for its answer')
        got, why = loc_boot_collect(req,
                                    gpu_has_work=(ahead.busy if ahead is not None else None))
        if got is not None:
            info['boot_wait'] = why
            info['boot_source'] = 'remote'
            loc_boot_done(req, 'remote')
            loc_log(f'  {row["roll"]}: bootstrap answered by worker {why["worker"]} in '
                    f'{why["worker_s"]}s; this session waited {why["wait_s"]}s of which '
                    f'{why["idle_s"]}s was billed idle')
            return got
        info['boot_fallback_reason'] = why
        src = ('local_timeout' if isinstance(why, str) and 'timeout' in why else 'local_failed')
        loc_boot_done(req, src)
        info['boot_source'] = src
        loc_log(f'  {row["roll"]}: no remote bootstrap ({why}); doing it here. A fallback costs '
                f'the wait plus the stage, which is why the wait is {LOC_BOOT_WAIT_S:.0f}s and '
                f'not the 240 s first proposed.')
    else:
        info['boot_source'] = ('local_breaker' if (req or {}).get('breaker') else
                               'local_no_worker' if (req or {}).get('no_worker') else
                               'local_post_failed' if (req or {}).get('skipped') else
                               'local_disabled')
        LOC_BOOT_SOURCE[info['boot_source']] = LOC_BOOT_SOURCE.get(info['boot_source'], 0) + 1
    LOC_PROF_TAG[0] = 'boot.'
    out = loc_run_bootstrap(M, feats, seed[:2], est0, ref0, out=info)
    LOC_PROF_TAG[0] = ''
    return out


def loc_match_phase(M, feats, arc_hat, hb_base, n, n_dev):
    """Match every frame against its own +/-30 m window on `n_dev` device(s).

    Pulled out of the roll's head so the A/B can run it twice on ONE roll's identical frames
    and identical shortlist stream.  Returns (corr, wall_s, per_device).
    """
    t0 = time.time()
    corr = {}
    LOC_PROF_TAG[0] = 'match.'
    # one worker per bank replica, frame i pinned to device i % n: both T4s hold the same bank,
    # so any device can answer any frame and the split needs no coordination
    mdev = list(M.bank.devices)[:max(1, n_dev)]
    mbusy = {d: [0.0, 0] for d in mdev}

    def match_one(item):
        i, short = item
        if short is None or len(feats['kp'][i]) == 0:
            return i, (np.zeros((0, 2), np.float64), np.zeros((0, 3)),
                       np.zeros(0, np.int64), np.zeros(0), np.zeros(0, np.int64))
        d = mdev[i % len(mdev)]
        t = time.perf_counter()
        qi, pos, ratio, rows = loc_match_frame(M, feats['desc'][i], short, device=d)
        mbusy[d][0] += time.perf_counter() - t
        mbusy[d][1] += 1
        return i, (feats['kp'][i][qi].astype(np.float64), M.pt_xyz[M.pt_order[pos]], pos,
                   ratio, rows)

    buf = []
    for item in loc_shortlist_stream(M, arc_hat, n):
        buf.append(item)
        if len(buf) >= LOC_MATCH_CHUNK:
            loc_heartbeat(dict(hb_base, frame=item[0], n_frames=n, stage='match'))
            corr.update(dict(loc_pmap(match_one, buf, workers=len(mdev))))
            buf = []
    corr.update(dict(loc_pmap(match_one, buf, workers=len(mdev))))
    LOC_PROF_TAG[0] = ''
    per_dev = {d: {'n': v[1], 'busy_s': round(v[0], 2),
                   'ms_per_frame': round(1000 * v[0] / max(v[1], 1), 2)}
               for d, v in mbusy.items()}
    return corr, round(time.time() - t0, 2), per_dev


def loc_corr_same(a, b):
    """Do two match runs agree element for element?  The A/B's correctness half."""
    if set(a) != set(b):
        return False
    for i in a:
        for x, y in zip(a[i], b[i]):
            x, y = np.asarray(x), np.asarray(y)
            if x.shape != y.shape or not np.array_equal(x, y, equal_nan=x.dtype.kind == 'f'):
                return False
    return True


def loc_match_ab(M, feats, arc_hat, hb_base, n, info):
    """One roll of the two-device A/B, or the plain match once the decision is made.

    ONE ROLL, BOTH ARMS.  kp/frame spans 1,244-3,097 across this plan -- a 2.5x swing in the
    thing match is linear in -- so an A/B that put arm A on roll X and arm B on roll Y would be
    measuring the rolls.  Running both arms back to back on the same frames, the same arc_hat and
    the same shortlist stream leaves the device count as the only difference.

    The roll's OUTPUT always comes from arm 1, the incumbent, so a sampled roll is not computed
    differently from an unsampled one; arm 2's correspondences are compared to arm 1's and the
    verdict recorded, which is also a free check that the second T4 answers identically.
    """
    ab = LOC_MATCH_AB_STATE[0] or {'samples': [], 'chosen': None}
    LOC_MATCH_AB_STATE[0] = ab
    ready = len(list(M.bank.devices)) >= 2
    if not (LOC_MATCH_AB and ab.get('chosen') is None and ready
            and len(ab['samples']) < LOC_MATCH_AB_ROLLS):
        if LOC_MATCH_AB and ab.get('chosen') is None and not ready:
            ab['skipped'] = (f'the bank is resident on {len(list(M.bank.devices))} device(s); '
                             f'a two-device arm is not runnable')
        corr, s, pd = loc_match_phase(M, feats, arc_hat, hb_base, n, LOC_MATCH_DEVICES_EFF[0])
        info['match_per_device'] = pd
        return corr, s

    corr1, s1, d1 = loc_match_phase(M, feats, arc_hat, hb_base, n, 1)
    corr2, s2, d2 = loc_match_phase(M, feats, arc_hat, hb_base, n, 2)
    same = loc_corr_same(corr1, corr2)
    smp = {'roll': int(info['roll']), 'n_frames': n, 'kp_mean': round(info['kp_mean'], 1),
           'devices_1': {'wall_s': s1, 'per_device': d1},
           'devices_2': {'wall_s': s2, 'per_device': d2},
           'speedup_2_over_1': round(s1 / max(s2, 1e-9), 3),
           'identical_correspondences': bool(same)}
    ab['samples'].append(smp)
    loc_log(f'  {info["roll"]}: MATCH A/B on identical input - 1 device {s1}s {d1}; '
            f'2 devices {s2}s {d2}; speedup {smp["speedup_2_over_1"]}x; '
            f'correspondences identical={same}', push_=True)
    if not same:
        loc_log(f'*** the two-device match returned DIFFERENT correspondences from the '
                f'one-device match on roll {info["roll"]}. Arm 1 is what this roll banks, and '
                f'two devices will not be adopted on this evidence. ***', push_=True)
    if len(ab['samples']) >= LOC_MATCH_AB_ROLLS:
        w1 = float(np.median([x['devices_1']['wall_s'] for x in ab['samples']]))
        w2 = float(np.median([x['devices_2']['wall_s'] for x in ab['samples']]))
        allsame = all(x['identical_correspondences'] for x in ab['samples'])
        win2 = allsame and w2 < w1 * (1.0 - LOC_MATCH_AB_MARGIN)
        ab.update(chosen=2 if win2 else 1, median_wall_s={'1': round(w1, 2), '2': round(w2, 2)},
                  margin=LOC_MATCH_AB_MARGIN, all_identical=allsame,
                  decided_utc=time.strftime('%Y-%m-%dT%H:%M:%SZ', time.gmtime()),
                  why=('two devices win by more than the margin' if win2 else
                       'correspondences differed between devices' if not allsame else
                       f'two devices are not {LOC_MATCH_AB_MARGIN:.0%} faster; the incumbent '
                       f'stays'))
        LOC_MATCH_DEVICES_EFF[0] = ab['chosen']
        loc_log(f'*** MATCH A/B DECIDED after {len(ab["samples"])} rolls: '
                f'LOC_MATCH_DEVICES={ab["chosen"]} ({ab["why"]}). Median wall 1 device '
                f'{w1:.1f}s vs 2 devices {w2:.1f}s. This is settled for run_id {LOC_RUN_ID} and '
                f'is not re-measured on resume. ***', push_=True)
    info['match_ab'] = smp
    info['match_per_device'] = d1
    return corr1, round(s1 + s2, 2)


def loc_process_head(M, runners, row, video, hb_base, pre_extracted=None, ahead=None):
    """One roll up to and including the match, then its tail question.  Returns a `pend` dict.

    The roll is split HERE and nowhere else, at the one boundary where a stage's whole input is
    already materialised and small enough to send: after the match the tail needs the
    correspondences and the map and nothing that lives on this box.  `loc_process_tail` finishes
    it, one roll later, and the two together are exactly what one `loc_process_roll` was --
    same calls, same order, same numbers.

    `pre_extracted` is the (feats, info) the look-ahead already produced for this roll; the
    extract phase is skipped when it is supplied and run inline when it is not, so the serial
    path and the pipelined path compute exactly the same thing in the same order.
    """
    LOC_PROF.clear()
    LOC_PROF_TAG[0] = ''
    LOC_PROF_SYNC[0] = LOC_ROLLS_DONE[0] < LOC_PROF_SYNC_ROLLS
    feats, info = (pre_extracted if pre_extracted is not None
                   else loc_extract_phase(runners, row, video, hb_base, M=M))
    n = info['n_frames']
    # the clock the loop is charged on starts where the loop's own work starts: with the
    # look-ahead engaged the extract was paid for during the PREVIOUS roll
    t_roll = time.time() if pre_extracted is not None else info['t0_roll']

    seed = loc_seed_intrinsics(row)
    t0 = time.time()
    anchor_idx, arcs_b, ok_b, inl_b = loc_boot_phase(M, row, feats, seed, hb_base, n, info,
                                                     ahead=ahead)
    ts = feats['t_ms'].astype(np.float64)
    arc_hat, acc = loc_bootstrap(ts, anchor_idx, arcs_b, ok_b, inl_b)
    info['bootstrap_s'] = round(time.time() - t0, 2)
    loc_stop_requested(where='after bootstrap')
    info['n_anchor'] = int(len(anchor_idx))
    info['n_anchor_accepted'] = int(acc.sum())
    info['blind_gap_m'] = loc_blind_gap(ts, anchor_idx, arc_hat, acc)
    info['arc_hat'] = None if arc_hat is None else np.round(arc_hat, 3).tolist()
    loc_log(f'  {row["roll"]}: bootstrap {info["n_anchor_accepted"]}/{info["n_anchor"]} '
            f'anchors pass gate {LOC_BOOT_GATE}, blind gap {info["blind_gap_m"]:.1f} m, '
            f'{info["bootstrap_s"]}s')
    if arc_hat is None:
        raise RuntimeError(f'roll {row["roll"]}: no bootstrap anchor survived the gate')

    # match every frame ONCE; the correspondences do not depend on intrinsics, so pass 1 and
    # pass 2 of the intrinsics fit both re-use them and only one roll is ever buffered
    corr, info['match_s'] = loc_match_ab(M, feats, arc_hat, hb_base, n, info)
    loc_stop_requested(where='after match')

    # The tail's question goes out at the first instant its inputs exist, and its answer is
    # consumed at the end of the NEXT roll.  Everything below this line is the tail.
    info['tail_req'] = loc_tail_post(row, corr, seed, M)
    info['head_s'] = round(time.time() - t_roll, 2)
    # A roll waiting for its tail holds only what the tail and the record need: the
    # correspondences (~24 MB) and three per-frame columns.  The features are ~1 GiB and none of
    # them is read again, so they are dropped here rather than kept resident for a second roll,
    # which is what would otherwise break the look-ahead's memory guard.
    slim = {'t_ms': feats['t_ms'], 'dec_idx': feats['dec_idx'],
            'n_kp': np.array([len(k) for k in feats['kp']], np.int32)}
    return {'row': row, 'info': info, 'seed': seed, 'corr': corr, 'n': n, 'feats': slim,
            'boot': (anchor_idx, arcs_b, ok_b, inl_b, acc), 'hb': hb_base,
            't_roll': t_roll, 't_head_end': time.time(), 'prof': loc_prof_report(n, reset=True)}


def loc_tail_phase(M, pend, ahead=None):
    """This roll's intrinsics and PnP: the CPU worker's answer if it arrived, otherwise our own.

    Interchangeable by construction, exactly as the bootstrap's two paths are.  Both sides call
    the same `loc_tail_solve` over the same correspondences with the same map, the same
    `loc_pnp_options(0)` (seed 0, one RANSAC thread) and the same order-preserving pool, and the
    solver's own identity is asserted into the request -- so `tail_source` is a note about where
    the time went and never about what the numbers are.
    """
    row, info, corr = pend['row'], pend['info'], pend['corr']
    req = info.get('tail_req')
    hb_base, n = pend['hb'], pend['n']
    loc_heartbeat(dict(hb_base, frame=0, n_frames=n, stage='tail'), force=True)
    if req and req.get('name'):
        loc_post_join(req, why='before waiting for its answer')
        per, intr, why = loc_tail_collect(
            req, corr, gpu_has_work=(ahead.busy if ahead is not None else None))
        if per is not None:
            info['tail_wait'] = why
            info['tail_source'] = 'remote'
            # the WORKER's spans, so `stage_s` still says what the stage cost wherever it ran
            info['intrinsics_s'] = why['intrinsics_s']
            info['pnp_s'] = why['pnp_s']
            loc_tail_done(req, 'remote')
            loc_log(f'  {row["roll"]}: tail answered by worker {why["worker"]} in '
                    f'{why["worker_s"]}s; this session waited {why["wait_s"]}s of which '
                    f'{why["idle_s"]}s was billed idle')
            return intr, per
        info['tail_fallback_reason'] = why
        src = ('local_timeout' if isinstance(why, str) and 'timeout' in why else 'local_failed')
        loc_tail_done(req, src)
        info['tail_source'] = src
        loc_log(f'  {row["roll"]}: no remote tail ({why}); doing it here. A fallback costs the '
                f'wait plus the stage, which is why the wait is {LOC_TAIL_WAIT_S:.0f}s against '
                f'a stage measured at ~49 s.')
    else:
        info['tail_source'] = ('local_breaker' if (req or {}).get('breaker') else
                               'local_no_worker' if (req or {}).get('no_worker') else
                               'local_post_failed' if (req or {}).get('skipped') else
                               'local_disabled')
        LOC_TAIL_SOURCE[info['tail_source']] = LOC_TAIL_SOURCE.get(info['tail_source'], 0) + 1
    intr, per, _ = loc_tail_solve(
        M, corr, pend['seed'], out=info,
        hb=lambda a, nn: loc_heartbeat(dict(hb_base, frame=a, n_frames=nn, stage='pnp')))
    return intr, per


def loc_process_tail(M, pend, ahead=None):
    """One roll's tail, then its record.  Returns (record npz dict, corr npz dict, summary)."""
    row, info, n, feats = pend['row'], pend['info'], pend['n'], pend['feats']
    anchor_idx, arcs_b, ok_b, inl_b, acc = pend['boot']
    LOC_PROF.clear()
    LOC_PROF_TAG[0] = ''
    t_tail = time.time()
    info['defer_s'] = round(t_tail - pend['t_head_end'], 2)
    intr, per = loc_tail_phase(M, pend, ahead=ahead)
    info['intrinsics'] = intr
    info['tail_s'] = round(time.time() - t_tail, 2)
    loc_log(f'  {row["roll"]}: intrinsics f={intr["focal"]:.1f} k1={intr["k1"]:.4f} '
            f'({intr["prov"]}, {intr["n_pass1"]} good frames, seed {intr["seed_prov"]} '
            f'{intr["seed"][0]:.0f}/{intr["seed"][1]:.3f}), tail {info["tail_s"]}s '
            f'from {info.get("tail_source")}')
    corr = pend['corr']

    summ = {'roll': int(row['roll']), 'in_map': bool(row.get('in_map')),
            'n_frames': n, 'n_anchor': info['n_anchor'],
            'n_anchor_accepted': info['n_anchor_accepted'],
            'blind_gap_m': info['blind_gap_m'],
            'extract_fps': round(n / max(info['extract_s'], 1e-6), 3),
            'n_solved': int(sum(p['ok'] for p in per)),
            'n_inl_p50': float(np.median([p['n_inl'] for p in per])) if n else 0.0,
            'rep_p50': float(np.nanmedian([p['rep_p50'] for p in per])) if n else np.nan,
            'focal': intr['focal'], 'k1': intr['k1'], 'intr_prov': intr['prov'],
            # the roll's own work, head plus tail, with the wait between them left out: the
            # deferral is a roll of the NEXT roll's time, not this one's, and charging it here
            # would double-count every roll in the pilot's projection
            'wall_s': round(info['head_s'] + info['tail_s'], 1),
            'head_s': info['head_s'], 'tail_s': info['tail_s'], 'defer_s': info['defer_s'],
            'ctl_n': 0, 'ctl_p50': None, 'ctl_p90': None, 'ctl_max': None,
            'ctli_n': 0, 'ctli_p50': None, 'ctli_p90': None, 'ctli_max': None,
            'ctl_offset_ms': None, 'ctl_offset_source': None,
            'ctl_offset_confidence': None, 'ctl_residual_dt_ms': None,
            'ctl_residual_p50_at_dt': None, 'ctl_residual_ok': None,
            'ctl_residual_max_ms': LOC_CTL_MAX_RESIDUAL_MS,
            'ctl_name_scale': None, 'ctl_clock_overlap': None,
            'self_run_inlier_frac': None}
    if row.get('in_map'):
        summ.update(loc_in_map_control(M, row, feats, per))
    LOC_PROF_SYNC[0] = False
    LOC_ROLLS_DONE[0] += 1
    info['stage_s'] = {k: info.get(k) for k in ('extract_s', 'bootstrap_s', 'match_s',
                                                'intrinsics_s', 'pnp_s')}
    # the head's spans were reported when the head ended; merge them back so one roll's profile
    # is still one dict
    info['prof'] = dict(pend.get('prof') or {}, **loc_prof_report(n))
    # solve_s, not roll_s: with the look-ahead engaged the extract was paid for during the
    # PREVIOUS roll, and the tail's answer was waited for during the NEXT one, so this span is
    # the roll's own work and deliberately excludes both.  The loop's own dt is the end-to-end
    # per-roll cost and that is what the projection is built on.
    summ['stage_s'] = dict(info['stage_s'], solve_s=round(info['head_s'] + info['tail_s'], 2),
                           tail_s=info['tail_s'], defer_s=info['defer_s'])
    # every GPU stage's per-device busy time in one place: the look-ahead extracts roll N+1 on
    # both devices while this roll's match and bootstrap also want both, so if the scheduler
    # serialises badly the wall will exceed the sum and these are the numbers that show it
    summ['gpu_per_device'] = {'extract': info.get('per_device'),
                              'match': info.get('match_per_device'),
                              'bootstrap': info.get('boot_per_device')}
    summ['device_share'] = info.get('device_share')
    summ['boot_exact'] = info.get('boot_exact')
    summ['boot_source'] = info.get('boot_source')
    summ['boot_wait'] = info.get('boot_wait')
    summ['boot_fallback_reason'] = info.get('boot_fallback_reason')
    summ['boot_req'] = info.get('boot_req')
    summ['tail_source'] = info.get('tail_source')
    summ['tail_wait'] = info.get('tail_wait')
    summ['tail_fallback_reason'] = info.get('tail_fallback_reason')
    summ['tail_req'] = info.get('tail_req')
    summ['match_devices'] = LOC_MATCH_DEVICES_EFF[0]
    summ['match_ab'] = info.get('match_ab')
    loc_log(f'  {row["roll"]}: gpu per-device extract={info.get("device_share")} '
            f'match={info.get("match_per_device")} bootstrap={info.get("boot_per_device")} '
            f'(exact={info.get("boot_exact")})')
    summ['prof'] = {k: v['s'] for k, v in info['prof'].items()}
    loc_log(f'  {row["roll"]}: stages {summ["stage_s"]}; split '
            + f' [gpu spans synced={bool(info["prof"]["_gpu_spans_synced"]["n"])}] '
            + ' '.join(f'{k}={v["s"]:.1f}s/{v["ms_per_frame"]:.1f}ms'
                       for k, v in info['prof'].items()
                       if not k.startswith('_') and v['ms_per_frame'] is not None))
    info['summary'] = summ
    return loc_record(row, feats, info, (anchor_idx, arcs_b, ok_b, inl_b, acc), intr, per, M), \
        loc_record_corr(per, M), summ


def loc_bank_write(roll, z, zc):
    """Compress and push one roll's two output objects.  Raises if either does not land."""
    for suf, d in (('', z), ('_corr', zc)):
        p = LOC_OUT / f'{roll}{suf}.npz'
        np.savez_compressed(p, **d)
        loc_push(p, f'{roll}{suf}.npz')


def loc_bank_submit(roll, z, zc):
    """Bank one roll on a background thread.  Returns the record `loc_bank_join` consumes.

    Measured at 43.5 s p50 -- 22% of a 200 s roll -- with everything else stopped, dominated by
    compressing and uploading the ~14 MB correspondence object.  Nothing in the next roll reads
    it, so it comes off the thread the accelerator waits behind.  What does NOT move is the rule
    it enforces: the cursor may not pass a roll whose bytes are not on Drive.  That is what
    `loc_bank_join` is for, and it is called at the cursor and nowhere else.
    """
    rec = {'roll': int(roll), 'queued_t': time.time()}

    def _bank():
        t0 = time.time()
        try:
            loc_bank_write(roll, z, zc)
            rec['ok'] = True
        except Exception as e:                                  # noqa: BLE001
            rec['error'] = f'{type(e).__name__}: {str(e)[:200]}'
            rec['ok'] = False
        finally:
            rec['bank_s'] = round(time.time() - t0, 1)
            LOC_IO['bank_bg_s'] += time.time() - t0
    rec['z'], rec['zc'] = z, zc                 # kept only for the retry inside the join
    if not LOC_BANK_ASYNC:
        _bank()
        return rec
    rec['fut'] = loc_bg('bank').submit(_bank)
    return rec


def loc_bank_join(rec):
    """Block until this roll is banked.  ONE synchronous retry, then it is fatal.

    Fatal on purpose, and the same as it always was: a roll whose output never reached Drive
    must not have the cursor moved past it.  The session dies, and the next one rewinds by
    LISTING out/ rather than trusting the cursor, so nothing is lost but the roll in flight.
    """
    if not rec:
        return None
    t0 = time.time()
    fut = rec.pop('fut', None)
    if fut is not None:
        fut.result()
    dt = time.time() - t0
    LOC_IO['bank_block_s'] += dt
    rec['bank_block_s'] = round(dt, 1)
    if not rec.get('ok'):
        loc_log(f'  {rec["roll"]}: the background bank FAILED ({rec.get("error")}); retrying '
                f'it here before the cursor is allowed past this roll', push_=True)
        loc_bank_write(rec['roll'], rec.pop('z', None), rec.pop('zc', None))
        rec['ok'], rec['retried'] = True, True
    rec.pop('z', None)
    rec.pop('zc', None)
    return rec


def loc_map_offsets(path=None):
    """run -> the DERIVED map naming-clock record: video_ms = name/unit - offset_ms.

    Derived from the device's own metadata (FIT camera_event/video_start), not fitted from the
    poses, and that distinction is the whole point: a fitted shift can absorb a genuine
    geometric error into the offset and still report sub-metre, so the control would be
    decoration.  With the offset derived independently, whatever error the control still shows
    is real.  Four mutually independent derivations agree to the integer millisecond on all ten
    runs; the fit this kernel ran before agreed to 28 ms on roll 1401 (3600 fitted on a 100 ms
    grid vs 3572 derived), sub-frame at 29.97 fps.
    """
    if LOC_MAP_OFFSETS[0] is None:
        p = Path(path or (LOC_IN / 'mapclock_offsets.json'))
        try:
            d = json.loads(p.read_text())
            d = d.get('offsets', d)
            LOC_MAP_OFFSETS[0] = {
                str(k): (v if isinstance(v, dict) else {'offset_ms': float(v)})
                for k, v in d.items()}
        except Exception:
            LOC_MAP_OFFSETS[0] = {}
    return LOC_MAP_OFFSETS[0]


def loc_check_offsets(rolls, require=True):
    """Every in-map roll in the work list must have a derived offset, checked BEFORE spending.

    Without this a run whose offsets file was staged incompletely would silently fall back to
    an unjoined control and the strongest correctness signal in the run would quietly vanish.
    """
    need = sorted({str(r['roll']) for r in rolls if r.get('in_map')})
    have = loc_map_offsets()
    missing = [r for r in need if r not in have]
    rec = {'in_map_rolls': need, 'n_offsets': len(have), 'missing': missing,
           'offsets': {k: have[k].get('offset_ms') for k in need if k in have},
           'name_units': {k: have[k].get('name_unit') for k in need if k in have}}
    if missing and require:
        raise RuntimeError(
            f'no derived map-clock offset for in-map run(s) {missing}. The in-map control is '
            f'the strongest early correctness signal in this run and it cannot be joined '
            f'without them; a fitted shift is NOT an acceptable substitute because it can '
            f'absorb a real geometric error. Stage tmp/backlog/mapclock/offsets.json, or set '
            f'SRS_LOC_REQUIRE_OFFSETS=0 deliberately.')
    return rec


def loc_map_run_times(M, run):
    """(t_ms on the ROLL's video clock, map row) of one map run, sorted.

    The map does not name every run the same way: 391/764/916 were added by a1_extract and are
    named '<video_ms>.jpg' (offset 0); 37/38/45/1388/1401 carry NANOSECOND names offset by the
    camera's video_start (3.18-3.57 s); 982 and 1005 carry nanosecond names on a different
    origin entirely (544 s and 5,296 s).  `name_unit` and `offset_ms` both come from the derived
    record -- nothing here is inferred from the numbers themselves any more.
    """
    rows = [(int(n.split('/')[1].split('.')[0]), i) for i, n in enumerate(M.names)
            if n.startswith(run + '/')]
    if not rows:
        return np.zeros(0), np.zeros(0, np.int64), None
    t = np.array([r[0] for r in rows], np.float64)
    j = np.array([r[1] for r in rows], np.int64)
    rec = loc_map_offsets().get(run)
    if rec is None:
        return np.zeros(0), np.zeros(0, np.int64), 'NO_DERIVED_OFFSET'
    unit = rec.get('name_unit') or ('ms' if np.median(t) < 1e10 else 'ns')
    if unit == 'ns':
        t = t / 1e6
    t = t - float(rec.get('offset_ms', 0.0))            # -> the roll's own video clock
    o = np.argsort(t)
    return t[o], j[o], f'{unit}-{rec.get("offset_ms")}ms'


def loc_in_map_control(M, row, feats, per, max_gap_ms=1000.0):
    """The free correctness check on the 10 in-map rolls, and the strongest early signal there is.

    Two readings, both against p4/img_centre.npy:
      ctl_*   frames whose name '<roll>/<t_ms>.jpg' IS a map image -- exact, but only possible
              where the map names that run in milliseconds off the same grid;
      ctli_*  the map run's own centres interpolated to this frame's time, used only when the
              bracketing map frames are within `max_gap_ms` and the two clocks overlap.
    Sub-metre is expected.  Metres means something is wrong -- most likely the observation-bank
    permutation, the preprocessing or the camera -- and it should be diagnosed before any gate
    verdict is acted on.
    """
    run = str(row['roll'])
    t_map, j_map, scale = loc_map_run_times(M, run)
    ok = np.array([bool(p['ok']) for p in per])
    tq = feats['t_ms'].astype(np.float64)
    errs, ierrs, keep = [], [], []
    overlap = 0.0
    if len(t_map):
        lo, hi = max(t_map[0], tq.min()), min(t_map[-1], tq.max())
        overlap = max(0.0, hi - lo) / max(1.0, tq.max() - tq.min())
    for i, p in enumerate(per):
        if not p['ok']:
            continue
        j = M.name_row.get(f'{run}/{int(tq[i])}.jpg')
        if j is not None:
            errs.append(float(np.linalg.norm(p['centre'] - M.img_centre[j])))
        if len(t_map) >= 2 and overlap >= 0.5:
            k = int(np.searchsorted(t_map, tq[i]))
            if 0 < k < len(t_map) and (t_map[k] - t_map[k - 1]) <= max_gap_ms:
                w = (tq[i] - t_map[k - 1]) / max(t_map[k] - t_map[k - 1], 1e-9)
                c = (M.img_centre[j_map[k - 1]] * (1 - w) + M.img_centre[j_map[k]] * w)
                ierrs.append(float(np.linalg.norm(p['centre'] - c)))
                keep.append(i)
    try:
        rc = M.runs.index(run)
        tot = sum(p['n_inl'] for p in per if p['ok'])
        self_ = sum(p['n_inl'] for p in per if p['ok'] and (p['run_mask'] >> rc) & 1)
        frac = (self_ / tot) if tot else None
    except ValueError:
        frac = None
    q = lambda e, f: (float(f(e)) if len(e) else None)                           # noqa: E731
    e, ie = np.array(errs), np.array(ierrs)
    # The join is already on the derived offset, so this is NOT a fit any more -- it is the
    # RESIDUAL the join still leaves.  A residual inside one frame at 29.97 fps says the
    # derivation and the poses agree; a large one now means something regressed upstream, and
    # crucially it can no longer be absorbed into the offset and hidden.
    dt_best, dt_p50 = None, None
    if len(ie) >= 5:
        C = np.stack([p['centre'] for p in per])[np.asarray(keep)]
        tt = tq[np.asarray(keep)]
        best = (np.median(ie), 0.0)
        for dt in np.arange(-500.0, 501.0, 5.0):       # sub-frame resolution, not 100 ms
            c = np.stack([np.interp(tt + dt, t_map, M.img_centre[j_map][:, k])
                          for k in range(3)], 1)
            v = float(np.median(np.linalg.norm(C - c, axis=1)))
            if v < best[0]:
                best = (v, float(dt))
        dt_p50, dt_best = best
    orec = loc_map_offsets().get(run) or {}
    resid_ok = (dt_best is None) or (abs(dt_best) <= LOC_CTL_MAX_RESIDUAL_MS)
    return {'ctl_n': len(e), 'ctl_p50': q(e, np.median),
            'ctl_offset_ms': orec.get('offset_ms'),
            'ctl_offset_source': ('derived' if orec else 'MISSING'),
            'ctl_offset_confidence': orec.get('confidence'),
            'ctl_residual_dt_ms': dt_best, 'ctl_residual_p50_at_dt': dt_p50,
            'ctl_residual_max_ms': LOC_CTL_MAX_RESIDUAL_MS, 'ctl_residual_ok': bool(resid_ok),
            'ctl_p90': q(e, lambda x: np.percentile(x, 90)), 'ctl_max': q(e, np.max),
            'ctli_n': len(ie), 'ctli_p50': q(ie, np.median),
            'ctli_p90': q(ie, lambda x: np.percentile(x, 90)), 'ctli_max': q(ie, np.max),
            'ctl_name_scale': scale, 'ctl_clock_overlap': round(overlap, 3),
            'n_solved_frames': int(ok.sum()), 'self_run_inlier_frac': frac}

## `loc-main`

Driver: gate, load the map, run the roll loop. CPU sessions branch to the worker loop here.

In [ ]:
# === cell: loc-main ===
def loc_rclone_setup():
    """Both remotes on disk.  Deliberately separate from fetching anything: it has to run
    before the GPU inventory gate (so the gate can push its verdict) while the inputs must not."""
    LOC_ROOT.mkdir(parents=True, exist_ok=True)
    LOC_IN.mkdir(parents=True, exist_ok=True)
    if LOC_DRY:
        return {'dry': True}
    p = Path('~/.config/rclone').expanduser()
    p.mkdir(parents=True, exist_ok=True)
    conf = RCLONE_CONF_SRS.rstrip('\n') + '\n\n' + RCLONE_CONF_BACKUP.rstrip('\n') + '\n'
    (p / 'rclone.conf').write_text(conf)
    (p / 'rclone.conf').chmod(0o600)
    loc_sh('which rclone || (curl -s https://rclone.org/install.sh | sudo bash)')
    have = {s.strip('[]') for s in re.findall(r'(?m)^\[\w+\]$', conf)}
    assert {'srs', 'yusufishabazz'} <= have, f'rclone.conf is missing a remote: {have}'
    return {'remotes': sorted(have)}


def loc_fetch_inputs():
    """The ~12 MB of staged inputs and the plan.  Runs AFTER the GPU inventory gate."""
    if LOC_DRY:
        return {'dry': True}
    loc_sh(f'rclone copy "{LOC_IN_REMOTE}" "{LOC_IN}"')
    loc_pull('plan.json', LOC_IN / 'plan.json', LOC_REMOTE)
    return {'inputs': sorted(x.name for x in LOC_IN.iterdir()),
            'free_gb': round(loc_freespace(str(LOC_ROOT)), 1)}


def loc_bank_devices(st):
    """Which devices the observation bank is replicated onto this session.

    Both, while the match A/B is still undecided -- a two-device arm is not runnable without a
    second replica.  Once decided it follows the verdict, so a run that settles on one device
    stops paying 3.17 GB of the other T4's VRAM on every later session.
    """
    if LOC_BANK_DEVICES:                                   # an explicit request always wins
        return list(LOC_BANK_DEVICES)
    if not LOC_BANK_DEVICE.startswith('cuda'):             # SRS_LOC_BANK_DEVICE=cpu, dry runs
        return [LOC_BANK_DEVICE]
    ab = (st.get('match_ab') or {})
    if LOC_MATCH_AB and ab.get('chosen') is None and len(LOC_DEVICES) > 1:
        return [f'cuda:{d}' for d in LOC_DEVICES]
    n = int(ab.get('chosen') or LOC_MATCH_DEVICES)
    return [f'cuda:{d}' for d in LOC_DEVICES[:max(1, n)]] or [LOC_BANK_DEVICE]


def loc_main():
    LOC_ROOT.mkdir(parents=True, exist_ok=True)
    LOC_VID.mkdir(parents=True, exist_ok=True)
    if 'setup' in LOC_STAGES:
        setup = loc_rclone_setup()
    else:
        setup = {'skipped': 'setup not in SRS_LOC_STAGES'}
    # FIRST real action: no video pulled, no model built, no input fetched.  One minute of quota
    # to find out whether this session got the accelerator the budget is priced on -- and, since
    # Kaggle metadata cannot carry an environment variable, it is also the ONLY way this process
    # can learn which half of the split it is.  So it runs before the state is even loaded: the
    # role decides which state, heartbeat and log file this session owns.
    inv = loc_gpu_inventory()
    role = loc_resolve_role(inv)
    st = loc_load_state()
    st['setup'] = setup
    gpu_role = role == 'gpu'
    loc_log(f'run_id={LOC_RUN_ID} role={role} session={loc_session_id()} '
            f'chunk={st.get("chunk")} stages={LOC_STAGES} dry={LOC_DRY} '
            f'budget={LOC_BUDGET_H}h used={float(st.get("gpu_seconds", 0)) / 3600:.2f}h '
            f'split={LOC_SPLIT}')
    if st.get('previous_run_id'):
        # F3: editing the source is what a rebuild IS, and it resets this counter.  The banked
        # work is safe (the cursor rewinds by listing out/), but the BUDGET is not carried, so
        # the number that was spent has to be visible or the 20 h cap silently becomes 20 h more.
        loc_log(f'*** BUDGET ACCOUNTING RESET: the state on the remote belongs to run '
                f'{st["previous_run_id"]}, which had banked {st.get("previous_done")} rolls and '
                f'{float(st.get("previous_gpu_seconds") or 0) / 3600:.2f} h of session '
                f'wall-clock. This session starts its budget at 0 against a {LOC_BUDGET_H} h '
                f'cap; that cap is inlined at push time and is meant to ALREADY have the spent '
                f'hours subtracted. ***', push_=True)
    st['gpu_inventory'] = loc_gpu_gate(st, inv)
    if 'setup' in LOC_STAGES:
        st['setup'] = dict(st['setup'], **loc_fetch_inputs())
        loc_log(f'setup: {st["setup"]}')
        # onnxruntime is not on the image; nothing below this line can import it until now.
        # The CPU worker needs faiss and pycolmap and NOTHING else: no onnxruntime, no
        # TensorRT, no engine cache, no re-exec -- it never touches a graph.  Installing them
        # anyway would spend minutes of a session whose entire job is a faiss scan.
        st['install'] = loc_install(LOC_IN / 'aliked-n16rot-static-const.onnx',
                                    prev=st.get('install'), cpu_only=not gpu_role)
        loc_save_state(st)

    model = None
    if gpu_role:
        # A CPU-only session, so it needs onnxruntime and nothing else: the graph TensorRT will
        # be given does not exist until this has run, and preoptimising it is what needs ORT.
        model, st['model'] = loc_build_model()
        loc_log(f'model {model} nodes={st["model"]["pre"]["n_nodes"]} '
                f'trt_ready={st["model"]["pre"]["trt_ready"]} '
                f'topk={st["model"]["topk"]["after"]}')
        if 'setup' in LOC_STAGES:
            if LOC_EP == 'trt':
                st['install']['trt'] = loc_install_trt(
                    model, st['install'].get('cuda_major'), prev=st['install'].get('trt'))
                loc_log(f'tensorrt install: chosen={st["install"]["trt"].get("chosen")} '
                        f'ok={st["install"]["trt"].get("ok")} '
                        f'{st["install"]["trt"].get("skipped", "")}')
            # AFTER the last install, never before one: v2 preloaded at 3.7 min and installed
            # TensorRT at 7.1 min, so libnvinfer.so.10 was not on disk when the preload looked.
            st['install']['preload'] = loc_preload_gpu_libs(st['install'].get('cuda_major'))
            loc_log(f'gpu lib preload: {len(st["install"]["preload"]["loaded"])} loaded, '
                    f'{len(st["install"]["preload"]["failed"])} failed '
                    f'{st["install"]["preload"]["failed"]}')
            loc_save_state(st)
            # the ONE point at which every install is finished; does not return if it re-execs
            st['install']['reexec'] = loc_maybe_reexec(st)
        if 'gate' in LOC_STAGES:
            st['gate'] = loc_gate(model, (st.get('install') or {}).get('cuda_major'))
            loc_save_state(st)

    rolls, plan = (loc_load_plan() if gpu_role else ([], {}))
    if gpu_role:
        loc_log(f'plan: {len(rolls)} rolls, {sum(r.get("n_frames_est", 0) for r in rolls)} '
                f'frames estimated')
        st['mapclock'] = loc_check_offsets(rolls, require=LOC_REQUIRE_OFFSETS)
        loc_log(f'map-clock offsets: {st["mapclock"]["n_offsets"]} derived, in-map rolls '
                f'{st["mapclock"]["in_map_rolls"]}, offsets {st["mapclock"]["offsets"]}, units '
                f'{st["mapclock"]["name_units"]}, missing {st["mapclock"]["missing"]}')
    t0 = time.time()
    LOC_MATCH_AB_STATE[0] = st.get('match_ab')
    LOC_MATCH_DEVICES_EFF[0] = int((st.get('match_ab') or {}).get('chosen')
                                   or LOC_MATCH_DEVICES)
    bank_dev = loc_bank_devices(st) if gpu_role else 'cpu'
    M = LocMap(bank_device=bank_dev)
    M.load_index()
    st['cores'] = {'os_cpu_count': os.cpu_count(),
                   'affinity': len(os.sched_getaffinity(0)),
                   'physical': loc_physical_cores(), 'workers': loc_workers(),
                   'faiss_main_thread_omp': M.threads,
                   'mem_available_gib': (None if loc_mem_available_b() is None
                                         else round(loc_mem_available_b() / 2 ** 30, 1))}
    # cpu_count is LOGGED, never assumed: the plan budgets 4 cores for the CPU stages and
    # Kaggle's allocation is not promised, so the number the parallelism was sized against has
    # to appear in the log of the run that used it.
    loc_log(f'map {M.root}: {M.n_img} images, {M.n_obs} observations, bank on '
            f'{bank_dev} ({M.bank.devices}), faiss ntotal={M.index.ntotal}, loaded in '
            f'{time.time() - t0:.0f}s; cores {st["cores"]}')
    st['split'] = loc_split_report()
    st['split_gc'] = loc_split_gc(st)
    loc_save_state(st)
    if not gpu_role:
        return loc_worker_loop(M, st)

    runners = [LocRunner(model, LOC_EP, d) for d in LOC_DEVICES]
    loc_log(f'runners: {[(r.device_id, r.eps) for r in runners]}')

    cur, _ = loc_resume_cursor(rolls, st)
    end = len(rolls) if not LOC_MAX_ROLLS else min(len(rolls), cur + LOC_MAX_ROLLS)
    pre = LocPrefetch(LOC_VID)
    ahead = LocAhead(runners, pre, M=M)
    per_roll = list(st.get('pilot_rolls') or [])
    sec_per_frame = float(st.get('sec_per_frame') or LOC_SEC_PER_FRAME0)
    stop_reason = None
    # The pilot gates the FIRST ten rolls of a run.  A rebuild changes loc_run_id and throws the
    # collected summaries away, so without this the eleventh session of a campaign would re-gate
    # on rolls 900-910 as if they were the pilot -- against a projection built from ten rolls
    # that tell it nothing it did not already know from the 900 banked.
    if LOC_PILOT and not st.get('pilot') and len(st.get('done') or []) >= LOC_PILOT_N:
        st['pilot'] = {'skipped': True, 'n_done': len(st.get('done') or []),
                       'reason': (f'{len(st.get("done") or [])} rolls are already banked, which '
                                  f'is past the {LOC_PILOT_N}-roll pilot; the gate has nothing '
                                  f'left to decide')}
        loc_log(f'pilot SKIPPED: {st["pilot"]["reason"]}')

    # The roll in flight, and the roll being banked.  Both exist because the two things a roll
    # ends with -- waiting for its tail and pushing its output -- are network round trips that
    # nothing in the NEXT roll depends on, and both used to be paid for with the accelerator
    # stopped.  `pending` is the roll whose tail question is out; `bank` is the roll whose bytes
    # are going up.  The cursor never passes either until the thing it is waiting for has landed.
    pending, bank, t_last = None, None, time.time()

    def flush_bank():
        """Join the bank in flight and move the persisted cursor past exactly that roll."""
        nonlocal bank
        if bank is None:
            return None
        rec, idx_ = loc_bank_join(bank), bank['idx']
        bank = None
        st['cursor'] = idx_ + 1
        st['done'] = sorted(set(st.get('done', [])) | {int(rolls[idx_]['roll'])})
        return rec

    def finish(pend):
        """Finish ONE roll: its tail, its record, its bank, its bookkeeping.  Returns a signal.

        Called a roll after `loc_process_head` produced `pend`, which is what gives the tail's
        worker a full roll of head start -- the same head start the bootstrap already gets, and
        the only arrangement in which shipping a ~49 s stage to a box with the same two cores
        can win rather than lose.
        """
        nonlocal bank, sec_per_frame, t_last
        row_, idx = pend['row'], pend['idx']
        if pend.get('failed'):
            z, zc, err = pend['failed']
            summ = loc_failed_summary(row_, err, 0.0)
            n_fail = int(st.get('n_consec_fail', 0))
        else:
            try:
                z, zc, summ = loc_process_tail(M, pend, ahead=ahead)
                summ['ahead'] = pend.get('ahead_rec')
                n_fail = 0
            except Exception as e:                              # noqa: BLE001
                loc_tail_abandon(pend['info'].get('tail_req'))
                n_fail = int(st.get('n_consec_fail', 0)) + 1
                loc_log(f'  {row_["roll"]} FAILED in the tail ({n_fail} in a row): '
                        f'{type(e).__name__}: {str(e)[:300]}', push_=True)
                z, zc = loc_failed_record(row_, e)
                summ = loc_failed_summary(row_, e, pend['info'].get('head_s') or 0.0)
            st['n_consec_fail'] = n_fail
        # ---- the bank goes to a background thread, and the PREVIOUS one is joined here.
        # `bank BEFORE the cursor advances` is unchanged and is exactly what this enforces: the
        # cursor is moved past roll `bank['idx']` and no further, and it is moved AFTER the join
        # that proves that roll's two objects are on Drive.  The roll being submitted here is
        # deliberately NOT counted -- a crash with its bytes in flight costs it and nothing else.
        flush_bank()
        bank = loc_bank_submit(row_['roll'], z, zc)
        bank['idx'] = idx
        # AFTER the bank of the previous roll, BEFORE the cursor advances.  Both halves matter:
        # polling earlier could exit with an unbanked output, and the bookkeeping below still
        # runs so state.json leaves with the cursor past the roll that is safely on Drive.
        stop_req_ = loc_stop_requested(force=True, where='between rolls')
        dt = time.time() - t_last
        t_last = time.time()
        LOC_IO['n_rolls'] += 1
        # a FAILED roll must not move the rate estimate: its n_frames is 0, so dividing by
        # max(n,1) charges the whole roll to a single frame and the ETA explodes -- enough for
        # the budget guard to stop a healthy session on the strength of one bad video
        if summ.get('n_frames'):
            sec_per_frame = 0.7 * sec_per_frame + 0.3 * (dt / summ['n_frames'])
        st['sec_per_frame'] = round(sec_per_frame, 5)
        # the pilot's summaries are carried IN state.json, so the gate still fires when the
        # first 10 rolls straddle a session boundary
        if len(per_roll) < LOC_PILOT_N:
            per_roll.append(summ)
            st['pilot_rolls'] = per_roll
        st['last'] = summ
        st['split'] = loc_split_report()
        st['match_ab'] = LOC_MATCH_AB_STATE[0]
        loc_save_state(st)
        # The ACHIEVED queue depth is printed per roll, not left to be inferred from the rate.
        # A queue that never fills costs the head start the bootstrap request needs and shows up
        # nowhere else: the rolls still come out right, just later, which is exactly the shape
        # of failure this component is prone to.
        ah = summ.get('ahead') or {}
        loc_log(f'  {row_["roll"]}: boot_source={summ.get("boot_source")} '
                f'tail_source={summ.get("tail_source")} '
                f'match_devices={summ.get("match_devices")} '
                f'ahead={ah.get("depth")}/{ah.get("depth_target")} queued '
                f'({ah.get("ready")} ready, {"used" if ah.get("used") else "SERIAL"}) '
                f'boot_stall={st["split"]["boot_stall_s"]}s tail_stall='
                f'{st["split"]["tail_stall_s"]}s '
                f'({st["split"]["boot_stall_pct_budget"]}% of the {LOC_BUDGET_H} h budget), '
                f'sources {st["split"]["boot_source"]} / {st["split"]["tail_source"]}')
        io = st['split']['io']
        loc_log(f'  {row_["roll"]}: I/O overlapped {io["io_hidden_s_per_roll"]}s/roll hidden; '
                f'anchor post {io["post_overlapped_frac"]} overlapped '
                f'({io["post_blocking_s_per_roll"]}s/roll still blocking), bank '
                f'{io["bank_overlapped_frac"]} ({io["bank_blocking_s_per_roll"]}s/roll), fetch '
                f'{io["fetch_overlapped_frac"]} ({io["fetch_blocking_s_per_roll"]}s/roll), '
                f'deletes {io["rm_bg_s"]}s all off-thread')
        # `roll_s`, NOT a stage: this is the whole roll.  v3's line said only "356s" here and
        # it was read as the solve stage, which put the blame on the wrong stage entirely.
        loc_log(f'  {row_["roll"]}: solved {summ["n_solved"]}/{summ["n_frames"]}, '
                f'n_inl p50 {summ["n_inl_p50"]:.0f}, reproj p50 {summ["rep_p50"]:.2f} px, '
                f'roll_s={dt:.0f}s end-to-end '
                f'({summ["n_frames"] / max(dt, 1e-6):.1f} f/s over the whole roll)'
                + (f', IN-MAP CONTROL (derived offset {summ["ctl_offset_ms"]} ms) name '
                   f'n={summ["ctl_n"]} p50={summ["ctl_p50"]}, interp n={summ["ctli_n"]} '
                   f'p50={summ["ctli_p50"]}, residual shift {summ["ctl_residual_dt_ms"]} ms '
                   f'(limit +/-{summ["ctl_residual_max_ms"]}, ok={summ["ctl_residual_ok"]})'
                   if summ.get('ctl_n') or summ.get('ctli_n') else ''), push_=True)
        if n_fail >= LOC_MAX_CONSEC_FAIL:
            st['stop_reason'] = f'{n_fail} consecutive roll failures'
            flush_bank()
            loc_save_state(st)
            loc_log(f'*** {n_fail} rolls in a row failed - this is a defect, not bad luck. '
                    f'HALTING. ***', push_=True)
            sys.exit(6)
        # EVERY in-map roll is a control, not only the ten the pilot happens to see.  The pilot
        # is skipped on a resume that is already past LOC_PILOT_N -- which the next session is,
        # with 193 rolls banked -- so without this the exit-10 regression check would simply
        # not exist for the rest of the campaign, at exactly the moment the pose stage moved to
        # another machine.  Same rule, same threshold, same exit code as the pilot's.
        if summ.get('in_map') and not summ.get('failed'):
            if (summ.get('ctl_offset_source') != 'derived'
                    or summ.get('ctl_residual_ok') is False):
                loc_log(f'*** IN-MAP CONTROL FAILED on roll {row_["roll"]}: offset source '
                        f'{summ.get("ctl_offset_source")}, residual shift '
                        f'{summ.get("ctl_residual_dt_ms")} ms against a limit of '
                        f'+/-{LOC_CTL_MAX_RESIDUAL_MS}, name p50 {summ.get("ctl_p50")} m, '
                        f'interp p50 {summ.get("ctli_p50")} m, tail_source '
                        f'{summ.get("tail_source")}. The map-clock offsets are DERIVED from '
                        f'device metadata and four independent derivations agree to the '
                        f'millisecond, so this is a regression upstream and not a bad offset. '
                        f'HALTING. ***', push_=True)
                st['stop_reason'] = f'in-map control failed on roll {row_["roll"]}'
                flush_bank()
                loc_save_state(st)
                sys.exit(10)
        # the gate lives HERE, with the roll it gates, and not in the loop body: the last roll
        # of a session is finished after the loop has ended, and a gate that only ran inside
        # the loop would not see it
        if not stop_req_:
            pilot_gate(idx)
        return stop_req_

    def pilot_gate(idx):
        if not (LOC_PILOT and len(per_roll) >= LOC_PILOT_N and not st.get('pilot')):
            return
        st['pilot'] = loc_pilot_eval(rolls[:idx + 1], per_roll, st.get('gpu_inventory'),
                                     all_rolls=rolls, st_done=len(st.get('done') or []))
        loc_save_state(st)
        p = st['pilot']
        loc_log(f'PILOT {p["verdict"].upper()}: '
                f'gate40={p["gate40_pass_fraction"]:.3f} ({p["gate40_score"]}), blind gap '
                f'{p["longest_blind_gap_m"]:.1f} m ({p["blind_gap_score"]}); '
                f'{p["n_gpus"]} GPU(s) {p["gpu_names"]}; extraction '
                f'{p["extract_fps_measured"]} f/s measured vs {p["extract_fps_reference"]} '
                f'reference ({p["extract_fps_ratio"]}x, a DIAGNOSTIC - the look-ahead hides '
                f'extraction under the CPU stages); wall {p["wall_s_per_roll_median"]} '
                f's/roll measured => {p["projected_total_h"]} h for the whole plan, '
                f'{p["projected_remaining_h"]} h for what is left against '
                f'{p["budget_remaining_h"]} h of budget = {p["rolls_affordable"]} rolls '
                f'affordable; in-map controls {p["in_map_controls"]}', push_=True)
        if not p['controls_ok']:
            loc_log(f'*** IN-MAP CONTROL FAILED on roll(s) {p["controls_failed"]}. The '
                    f'map-clock offsets are DERIVED from device metadata and four '
                    f'independent derivations agree to the millisecond, so a residual '
                    f'shift beyond {LOC_CTL_MAX_RESIDUAL_MS} ms is a regression upstream, '
                    f'not a bad offset - and it can no longer be absorbed into the offset '
                    f'and hidden. HALTING. ***', push_=True)
            flush_bank()
            loc_save_state(st)
            sys.exit(10)
        if not p['gpu_ok']:
            loc_log(f'*** PILOT COMPLETED ON {p["n_gpus"]} GPU(s), NOT {LOC_MIN_GPUS}. The '
                    f'{LOC_BUDGET_H} h budget was authorised at the 2-GPU rate; spending '
                    f'it at {p["extract_fps_ratio"]}x is not what was agreed. HALTING for '
                    f'the user - the plan is in descending roll order, so stopping here is '
                    f'cheap. ***', push_=True)
            flush_bank()
            loc_save_state(st)
            sys.exit(9)
        if p['verdict'] == 'stop':
            loc_log('*** PILOT GATE FAILED against tmp/backlog/prereg.json - HALTING '
                    'before roll 11. Nothing further will be spent. ***', push_=True)
            flush_bank()
            loc_save_state(st)
            sys.exit(5)
        if p['verdict'] == 'marginal':
            loc_log('*** PILOT MARGINAL - continuing per the pre-registration, but this '
                    'run is NOT clean; see state.json["pilot"] ***', push_=True)

    while cur < end:
        row = rolls[cur]
        est_s = max(30.0, sec_per_frame * float(row.get('n_frames_est') or 1717))
        stop_reason = loc_budget_stop(st, est_s)
        if stop_reason:
            loc_log(f'STOPPING CLEANLY before roll {row["roll"]}: {stop_reason}', push_=True)
            break
        hb = {'roll': int(row['roll']), 'roll_idx': cur, 'n_rolls': len(rolls),
              'rate_fps': round(1.0 / sec_per_frame, 2),
              'eta_utc': time.strftime('%Y-%m-%dT%H:%M:%SZ', time.gmtime(
                  time.time() + sec_per_frame * sum(
                      float(r.get('n_frames_est') or 1717) for r in rolls[cur:end])))}
        loc_heartbeat(dict(hb, frame=0, n_frames=0, stage='fetch'), force=True)
        got = ahead.take(row)
        depth_at_take = LOC_AHEAD_STATS['depth_last']
        video = None
        pex = None
        head = None
        head_err = None
        try:
            if got is None:
                video, fetch_s, fetch_wait = pre.get(row)
                loc_log(f'[{cur + 1}/{len(rolls)}] roll {row["roll"]} '
                        f'({row.get("n_frames_est")} frames est, fetch {fetch_s:.1f}s '
                        f'of which {fetch_wait:.1f}s blocked)')
                pex = loc_extract_phase(runners, row, video, hb, M=M)
                pex[1]['fetch_s'] = round(fetch_s, 1)
                pex[1]['fetch_wait_s'] = round(fetch_wait, 1)
                pre.drop(video)
                video = None
                saved = None
            else:
                pex, saved = (got[0], got[1]), max(0.0, got[1]['extract_s'] - got[2])
                loc_log(f'[{cur + 1}/{len(rolls)}] roll {row["roll"]} '
                        f'({row.get("n_frames_est")} frames est, fetch '
                        f'{got[1].get("fetch_s", 0.0):.1f}s of which '
                        f'{got[1].get("fetch_wait_s", 0.0):.1f}s blocked the look-ahead) '
                        f'EXTRACTED AHEAD under the previous roll: waited {got[2]:.1f}s instead '
                        f'of running {got[1]["extract_s"]:.1f}s of extract, saved {saved:.1f}s')
            # Top the queue back up against a MEASURED feature size PER FRAME, so each queued
            # roll is sized by its own frame estimate.  A stop seen mid-roll does not abandon
            # this roll, but it does mean not paying ~77 s of GPU per slot to extract rolls that
            # will never be processed -- an empty index range is how that is said.
            st['ahead'] = ahead.fill(rolls, cur + 1, cur + 1 if LOC_STOP['seen'] else end,
                                     pex[1]['feats_bytes'] / max(pex[1]['n_frames'], 1))
            head = loc_process_head(M, runners, row, None, hb, pre_extracted=pex, ahead=ahead)
            pex = (None, pex[1])   # the drop in loc_process_head is defeated by this reference
            got = (None, got[1], got[2]) if got is not None else None   # and by this one
            head['idx'] = cur
            head['ahead_rec'] = {'used': got is not None, 'saved_s': saved,
                                 'wait_s': None if got is None else round(got[2], 2),
                                 'depth': depth_at_take, 'depth_target': ahead.depth,
                                 'ready': LOC_AHEAD_STATS['ready_last'], 'next': st['ahead']}
        except Exception as e:                                  # noqa: BLE001
            # a roll that failed for any reason still owns a request on the remote; it will
            # never be consumed now, and 202 MB per abandoned roll is not something to leave.
            # NOT counted against the breaker: the failure is the roll's, not the split's.
            loc_boot_abandon((pex[1] if pex else {}).get('boot_req'))
            loc_tail_abandon((pex[1] if pex else {}).get('tail_req'))
            head_err = e
        if video:                        # only the serial path still holds one; None otherwise
            pre.drop(video)
        # ---- retire the PREVIOUS roll first, so rolls are always banked in plan order
        stop_req = finish(pending) if pending is not None else None
        pending = head
        if head_err is not None:
            n_fail = int(st.get('n_consec_fail', 0)) + 1
            loc_log(f'  {row["roll"]} FAILED ({n_fail} in a row): {type(head_err).__name__}: '
                    f'{str(head_err)[:300]}', push_=True)
            st['n_consec_fail'] = n_fail
            zf, zcf = loc_failed_record(row, head_err)
            # a head that failed has no tail to wait for, so it is finished immediately -- but
            # still THROUGH `finish`, so the bank ordering and the cursor rule are one code path
            stop_req = finish({'row': row, 'idx': cur, 'failed': (zf, zcf, head_err),
                               'info': {}, 'ahead_rec': None}) or stop_req
            pending = None
        cur += 1
        # Whatever this roll freed is still on the heap until it is handed back; the next roll's
        # extract line reports what came of it.
        loc_malloc_trim(int(row['roll']))

        if stop_req:
            stop_reason = (f'requested ({stop_req.get("reason")!r}, requested_utc '
                           f'{stop_req.get("requested_utc")}, seen at {LOC_STOP["where"]})')
            st['stop_requested'] = dict(stop_req, seen_at=LOC_STOP['where'],
                                        withdrawn_as=loc_stop_clear())
            loc_log(f'STOPPING CLEANLY ON REQUEST after roll {row["roll"]}: {stop_reason}. '
                    f'{len(st["done"])} rolls banked, cursor {st["cursor"]}/{len(rolls)}. '
                    f'Exit 0 - a requested stop is not a failure.', push_=True)
            break
    ahead.shutdown()      # a look-ahead for a roll this session will not reach is dead weight
    # The roll whose tail question is still out is FINISHED, never abandoned: its extract, its
    # bootstrap and its match are already paid for, and the only thing left is a stage this
    # session can always do itself.  Then the last bank is joined, which is what finally moves
    # the cursor past it.
    if pending is not None:
        last_stop = finish(pending)
        pending = None
        if last_stop and not st.get('stop_requested'):
            stop_reason = (f'requested ({last_stop.get("reason")!r}, requested_utc '
                           f'{last_stop.get("requested_utc")}, seen at {LOC_STOP["where"]})')
            st['stop_requested'] = dict(last_stop, seen_at=LOC_STOP['where'],
                                        withdrawn_as=loc_stop_clear())
    flush_bank()
    if LOC_PILOT and not st.get('pilot'):
        st['pilot'] = {'skipped': False, 'incomplete': True, 'n_rolls': len(per_roll)}
    if not LOC_PILOT:
        st['pilot'] = {'skipped': True}

    st['stop_reason'] = stop_reason or ('plan exhausted' if cur >= end else 'range exhausted')
    st['stop_requested_utc'] = (LOC_STOP['seen'] or {}).get('requested_utc')
    st['finished_utc'] = time.strftime('%Y-%m-%dT%H:%M:%SZ', time.gmtime())
    # every queued delete and upload lands BEFORE the final state is written, so the remote and
    # state.json cannot disagree in the seconds after this session ends
    st['io'] = loc_io_report()
    st['io_pools'] = loc_pools_flush()
    loc_save_state(st)
    loc_heartbeat({'roll': None, 'roll_idx': cur, 'n_rolls': len(rolls), 'frame': 0,
                   'n_frames': 0, 'rate_fps': round(1.0 / sec_per_frame, 2), 'eta_utc': None,
                   'stage': 'stopped' if LOC_STOP['seen'] else 'done',
                   'stop_reason': st['stop_reason'],
                   'stop_requested_utc': st['stop_requested_utc']}, force=True)
    loc_log(f'DONE: cursor {cur}/{len(rolls)}, {len(st["done"])} rolls banked, '
            f'{st["gpu_seconds"] / 3600:.2f} h of the {LOC_BUDGET_H} h budget used, '
            f'reason: {st["stop_reason"]}', push_=True)
    return st


if __name__ == '__main__':
    loc_main()